# Reg2Chain: pipeline and evaluation

This notebook runs the full Reg2Chain pipeline on Commission Delegated Regulation (EU) 2022/2104 (olive oil), one agent per section, each followed by its evaluation. Sections are meant to be run in order; all outputs are written to `MyDrive/Reg2Chain_revision/`, so a Colab disconnection does not lose anything.

Credentials (Neo4j) are read from the Colab Secrets panel with `userdata.get` and are never stored in the notebook.

| Section | Content |
|---|---|
| 0 | Setup |
| 1 | Corpus, gold standard, requirements checklist |
| 2 | Preprocessing Agent |
| 3 | Entity Extraction Agent and Evaluation 1 (Table 1) |
| 4 | Ontology Agent and expert refinement |
| 5 | Graph Construction Agent and Evaluation 2 (Table 2, coverage) |
| 6 | Regulatory Query Agent |
| 7 | Prompt Engineering Agent and contract generation |
| 8 | Evaluation 3: generated contracts (Table 3, expert review) |
| 9 | Export for the repository |

## 0. Setup

### 0.1 Mount Google Drive and create the project folders

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
ROOT = '/content/drive/MyDrive/Reg2Chain_revision'
for sub in ['data', 'scripts', 'step1_gold', 'step1_gold/annotated', 'step2_extraction',
            'step3_kg', 'step4_generation', 'step5_contract_eval']:
    os.makedirs(f'{ROOT}/{sub}', exist_ok=True)

# the regulation PDF (in the Drive root)
src = '/content/drive/MyDrive/CELEX_32022R2104_EN_TXT.pdf'
dst = f'{ROOT}/data/CELEX_32022R2104_EN_TXT.pdf'
if not os.path.exists(dst):
    shutil.copy(src, dst)
os.chdir(ROOT)
print('Working in', os.getcwd()); print(os.listdir('data'))

### 0.2 Install libraries

In [ ]:
!pip install -q pymupdf openpyxl

## 1. Corpus, gold standard and requirements checklist
Evaluation references built independently of the system: derived from the regulation text and annotated manually.

### 1.1 Extract the regulation text (PDF)

In [ ]:
import pymupdf
doc = pymupdf.open('data/CELEX_32022R2104_EN_TXT.pdf')
text = ''.join(page.get_text() for page in doc)
open('data/regulation_text.txt', 'w', encoding='utf-8').write(text)
print(doc.page_count, 'pages,', len(text), 'characters')

### 1.2 Step 1 scripts
The scripts are saved in `scripts/` so they can be deposited as they are in the repository.

In [ ]:
%%writefile scripts/segment_regulation.py
"""
Step 1 - Segment Regulation (EU) 2022/2104 into provision units for annotation.

Scope: enacting terms (Articles 1-14) + Annex I, Table A (quality characteristics,
linearised one unit per category). Recitals are excluded (not normative).

Unit granularity: one unit per numbered paragraph. A paragraph longer than
MAX_LEN characters with lettered points (a), (b)... is split into its lead-in and
one unit per point (no text is duplicated across units).

Output: units.csv  (unit_id, article, title, split, double_annotation, text)
Usage:  python segment_regulation.py regulation_text.txt units.csv
"""
import csv
import random
import re
import sys

MAX_LEN = 600
DEV_RATIO = 0.25          # dev units: used to design rules / prompts only
DOUBLE_RATIO = 0.30       # share of TEST units annotated by both annotators
SEED = 2104

HEADER_PATTERNS = [
    r"^EN$", r"^Official Journal of the European Union\s*$", r"^L 284/\d+\s*$",
    r"^\d{1,2}\.\d{1,2}\.\d{4}\s*$",
]


def clean_lines(text):
    lines = []
    for ln in text.splitlines():
        s = ln.strip()
        if any(re.match(p, s) for p in HEADER_PATTERNS):
            continue
        lines.append(s)
    return lines


def join(lines):
    t = " ".join(l for l in lines if l)
    t = re.sub(r"\s+", " ", t)
    t = re.sub(r"\(\s+", "(", t)
    return t.strip()


def enacting_terms(lines):
    start = lines.index("HAS ADOPTED THIS REGULATION:") + 1
    end = next(i for i, l in enumerate(lines) if l.startswith("This Regulation shall be binding in its entirety"))
    body = lines[start:end]
    # drop footnotes "(12) Council Regulation ..." at the end of Article 14
    body = [l for l in body if not re.match(r"^\(\d{2}\) Council Regulation", l)
            and not re.match(r"^22\.7\.1989|^\(OJ L", l)]
    articles, cur = [], None
    i = 0
    while i < len(body):
        m = re.match(r"^Article (\d+)$", body[i])
        if m:
            cur = {"num": int(m.group(1)), "title": body[i + 1], "lines": []}
            articles.append(cur)
            i += 2
            continue
        if cur is not None:
            cur["lines"].append(body[i])
        i += 1
    return articles


def split_paragraphs(lines):
    paras, cur = [], {"num": None, "lines": []}
    for l in lines:
        m = re.match(r"^(\d+)\.$", l)
        if m:
            if cur["lines"]:
                paras.append(cur)
            cur = {"num": m.group(1), "lines": []}
        else:
            cur["lines"].append(l)
    if cur["lines"]:
        paras.append(cur)
    return paras


def split_points(text):
    """Split 'lead-in (a) ... (b) ...' at top-level lettered points."""
    parts = re.split(r"(?<=[:;])\s*(?=\([a-h]\)\s)|(?<=[:;] or)\s*(?=\([a-h]\)\s)", text)
    if len(parts) == 1:
        parts = re.split(r"\s(?=\([a-h]\)\s)", text)
    out = []
    for p in parts:
        m = re.match(r"^\(([a-h])\)\s*(.*)$", p.strip())
        out.append((m.group(1), m.group(2)) if m else (None, p.strip()))
    return out


def article_units(articles):
    units = []
    for a in articles:
        for p in split_paragraphs(a["lines"]):
            text = join(p["lines"])
            base = f"ART{a['num']}" + (f".{p['num']}" if p["num"] else "")
            if len(text) > MAX_LEN and re.search(r"\([a-h]\)\s", text):
                for letter, t in split_points(text):
                    if t:
                        units.append((base + (f".{letter}" if letter else ".lead"), a, t))
            else:
                units.append((base, a, text))
    return units


TABLE_A_PARAMS = [
    "Acidity (%)", "Peroxide value (mEq O2/kg)", "K232", "K268 or K270", "ΔK",
    "Median of defect (Md)", "Fruity median (Mf)", "Fatty acid ethyl esters (mg/kg)",
]


def annex_table_a(lines):
    """Linearise Annex I Table A: one unit per category."""
    s = lines.index("A. Quality characteristics")
    e = next(i for i in range(s, len(lines)) if lines[i].startswith("(1) The median of the defects"))
    rows = lines[s:e]
    units, i = [], 0
    while i < len(rows):
        m = re.match(r"^(\d)\. (.+)$", rows[i])
        if m:
            name = m.group(2)
            j = i + 1
            if not rows[j].startswith(("≤", ">", "—", "Md", "Mf")):
                name += " " + rows[j]
                j += 1
            vals = []
            while j < len(rows) and not re.match(r"^\d\. ", rows[j]):
                vals.append(rows[j])
                j += 1
            vals = [re.sub(r"^(Md|Mf)\s+", "", re.sub(r"\s*\(\d\)$", "", v)) for v in vals]
            pairs = [f"{p} {v}" for p, v in zip(TABLE_A_PARAMS, vals) if v != "—"]
            text = (f"Annex I, Table A, quality characteristics of {name.lower()}: "
                    + ("; ".join(pairs) if pairs else "no limits set") + ".")
            units.append((f"ANX1.A.{m.group(1)}", {"num": "Annex I", "title": "Quality characteristics"}, text))
            i = j
        else:
            i += 1
    return units


def main(src, dst):
    lines = clean_lines(open(src, encoding="utf-8").read())
    units = article_units(enacting_terms(lines)) + annex_table_a(lines)

    rnd = random.Random(SEED)
    ids = [u[0] for u in units]
    dev = set(rnd.sample(ids, round(DEV_RATIO * len(ids))))
    test = [i for i in ids if i not in dev]
    double = set(rnd.sample(test, round(DOUBLE_RATIO * len(test))))

    with open(dst, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["unit_id", "article", "title", "split", "double_annotation", "text"])
        for uid, a, text in units:
            w.writerow([uid, a["num"], a["title"], "dev" if uid in dev else "test",
                        "Y" if uid in double else "N", text])
    n_dev = len(dev)
    print(f"{len(units)} units -> {dst}  (dev {n_dev}, test {len(units) - n_dev}, double-annotated {len(double)})")


if __name__ == "__main__":
    main(*sys.argv[1:3])


In [ ]:
%%writefile scripts/build_annotation_workbooks.py
"""
Step 1 - Build the annotation workbooks and the requirements checklist.

Outputs (in the output folder):
  Reg2Chain_annotation_A1.xlsx      annotator 1, all 63 units (dev + test)
  Reg2Chain_annotation_A2.xlsx      annotator 2, only the double-annotated test units
  Reg2Chain_requirements_checklist.xlsx   legal expert: validate / edit / delete / add

Usage: python build_annotation_workbooks.py units.csv OUT_DIR
"""
import csv
import os
import sys

from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation

ENTITY_TYPES = ["PRODUCT", "PARAMETER", "VALUE", "LABEL_ITEM", "ACTOR", "LEGAL_REF", "PROCESS"]
RELATION_TYPES = ["OBLIGATION", "PERMISSION", "PROHIBITION", "LIMIT", "CONDITION", "REFERS_TO"]

HEAD = Font(bold=True, color="FFFFFF")
HEAD_FILL = PatternFill("solid", fgColor="2F5496")
WRAP = Alignment(wrap_text=True, vertical="top")

GUIDELINES = [
    ("Reg2Chain annotation guidelines (v1.0)", None),
    ("Purpose", "Build an independent gold standard for entity and relation extraction from "
                "Regulation (EU) 2022/2104. Annotate from the text only. Do not look at any system output."),
    ("Unit", "One provision unit = one numbered paragraph (or one lettered point of a long paragraph) "
             "of Articles 1-14, or one linearised row of Annex I, Table A. Annotate each unit on its own."),
    ("How to annotate an entity", "In sheet 'Entities', add one row per entity: unit_id, the entity text COPIED EXACTLY "
                                  "from the unit (same spelling, spaces, quotes), and its type. Leave 'occurrence' at 1. "
                                  "Use 2, 3... only if the same entity text appears several times as separate entities "
                                  "(text inside a longer entity, e.g. 'virgin olive oil' inside 'extra virgin olive oil', "
                                  "is not counted)."),
    ("No overlap", "Entities never overlap or nest. Take the longest span that still has a single type. "
                   "Example: 'oils referred to in Article 1, point (b)' is ONE PRODUCT (do not also mark "
                   "'Article 1, point (b)' as LEGAL_REF)."),
    ("Spans", "Exclude leading articles (the, a, an) and final punctuation. Keep quotes for quoted label texts."),
    ("Entity types", None),
    ("PRODUCT", "Olive-oil categories and other products: extra virgin olive oil; virgin olive oil; lampante olive oil; "
                "refined olive oil; olive oil composed of refined olive oils and virgin olive oils; crude olive-pomace oil; "
                "refined olive-pomace oil; olive-pomace oil; blends of olive oil and other vegetable oils; foodstuffs; "
                "groups such as 'oils referred to in Article 1, point (b)'."),
    ("PARAMETER", "A measurable characteristic: acidity, peroxide value, K232, K268 or K270, ΔK, median of defect, "
                  "fruity median, fatty acid ethyl esters, wax content, ultraviolet absorption, temperature, "
                  "maximum capacity, percentage of oil."),
    ("VALUE", "A numeric limit or quantity, with its operator and unit: '≤ 0,80', 'five litres', '27 °C' "
              "(write 'below 27 °C' if 'below' is part of the condition), 'more than 50 %', '100 %'."),
    ("LABEL_ITEM", "Something that must / may / must not appear on the label or container: legal name, category "
                   "information, place of origin, harvesting year, storage conditions information, alphanumeric "
                   "identification of the packaging plant, 'first cold pressing', 'cold extraction', organoleptic "
                   "characteristics, trade description, a quoted mandatory wording."),
    ("ACTOR", "An organisation or role: Member States, Commission, final consumer, packaging plant, mill, "
              "restaurants/hospitals/canteens (collective establishments), panel leader."),
    ("LEGAL_REF", "A reference to a legal act or provision when it is not part of a PRODUCT span: "
                  "'Regulation (EU) No 1308/2013', 'Commission Implementing Regulation (EU) 2022/2105', 'Article 6(1)', "
                  "'Annex I'. Take the maximal reference ('Part VIII, point 1(a), of Annex VII to Regulation (EU) No 1308/2013')."),
    ("PROCESS", "A production, extraction or packaging method or feature: mechanical means, refining, first mechanical "
                "pressing, percolation, centrifugation, hydraulic presses, opening system that can no longer be sealed."),
    ("Relation types (between two entities already annotated in the same unit)", None),
    ("OBLIGATION", "subject must have / do / show object ('shall', 'must', 'mandatory'). "
                   "Ex.: (extra virgin olive oil, OBLIGATION, place of origin) in Art. 8(1)."),
    ("PERMISSION", "subject may have / do / show object, including restricted permissions ('may only', 'may appear "
                   "only for'). Ex.: (extra virgin olive oil, PERMISSION, harvesting year) in Art. 11(1)."),
    ("PROHIBITION", "subject must not have / show object ('shall not', 'no ... shall'). "
                    "Ex.: (olive-pomace oil ..., PROHIBITION, place of origin) in Art. 8(2)."),
    ("LIMIT", "a PRODUCT has a limit VALUE for a PARAMETER. Subject = PRODUCT, object = VALUE, and write the "
              "PARAMETER text in the 'qualifier' column. Ex.: (extra virgin olive oil, LIMIT, ≤ 0,80, qualifier: Acidity (%))."),
    ("CONDITION", "a permission/obligation on the subject applies only if the object holds. Subject = LABEL_ITEM or "
                  "PRODUCT, object = VALUE or PROCESS; PARAMETER in 'qualifier' if any. "
                  "Ex.: ('first cold pressing', CONDITION, below 27 °C, qualifier: temperature)."),
    ("REFERS_TO", "an entity is defined by / determined in accordance with a LEGAL_REF. "
                  "Ex.: (characteristics of olive oils, REFERS_TO, Commission Implementing Regulation (EU) 2022/2105)."),
    ("Several subjects", "If the provision lists several products ('extra virgin olive oil and virgin olive oil'), "
                         "annotate one entity per product and one relation per product."),
    ("Unsure?", "Write your doubt in 'notes' and annotate your best choice. Do not leave a unit unannotated: "
                "if a unit has no entity, add one row with unit_id and type 'NONE'."),
    ("Time", "About 4-5 minutes per unit. Please record your total time in the 'Log' sheet."),
]


def style_header(ws, widths):
    for i, w in enumerate(widths, start=1):
        c = ws.cell(row=1, column=i)
        c.font, c.fill, c.alignment = HEAD, HEAD_FILL, WRAP
        ws.column_dimensions[c.column_letter].width = w
    ws.freeze_panes = "A2"


def add_list_validation(ws, col, values, rows=2000):
    dv = DataValidation(type="list", formula1='"' + ",".join(values) + '"', allow_blank=True,
                        showErrorMessage=True, errorTitle="Invalid value",
                        error="Choose a value from the list.")
    ws.add_data_validation(dv)
    dv.add(f"{col}2:{col}{rows}")


def add_ref_validation(ws, col, ref, rows=2000):
    dv = DataValidation(type="list", formula1=ref, allow_blank=True, showErrorMessage=True,
                        errorTitle="Unknown unit", error="Choose a unit_id from the 'Units' sheet.")
    ws.add_data_validation(dv)
    dv.add(f"{col}2:{col}{rows}")


def guidelines_sheet(wb):
    ws = wb.active
    ws.title = "Guidelines"
    ws.column_dimensions["A"].width = 28
    ws.column_dimensions["B"].width = 120
    for r, (k, v) in enumerate(GUIDELINES, start=1):
        a = ws.cell(row=r, column=1, value=k)
        a.alignment = WRAP
        if v is None:
            a.font = Font(bold=True, size=12 if r > 1 else 14, color="2F5496")
        else:
            a.font = Font(bold=True)
            ws.cell(row=r, column=2, value=v).alignment = WRAP


def annotation_workbook(units, path, annotator):
    wb = Workbook()
    guidelines_sheet(wb)

    ws = wb.create_sheet("Units")
    ws.append(["unit_id", "article", "title", "text"])
    for u in units:
        ws.append([u["unit_id"], u["article"], u["title"], u["text"]])
    style_header(ws, [14, 10, 34, 130])
    for row in ws.iter_rows(min_row=2):
        for c in row:
            c.alignment = WRAP
    n = len(units) + 1
    unit_ref = f"=Units!$A$2:$A${n}"

    ents = wb.create_sheet("Entities")
    ents.append(["unit_id", "entity_text (copy exactly)", "type", "occurrence", "notes"])
    style_header(ents, [14, 70, 16, 12, 50])
    add_ref_validation(ents, "A", unit_ref)
    add_list_validation(ents, "C", ENTITY_TYPES + ["NONE"])

    rels = wb.create_sheet("Relations")
    rels.append(["unit_id", "subject_text", "relation", "object_text", "qualifier (PARAMETER, optional)", "notes"])
    style_header(rels, [14, 50, 16, 50, 30, 40])
    add_ref_validation(rels, "A", unit_ref)
    add_list_validation(rels, "C", RELATION_TYPES)

    log = wb.create_sheet("Log")
    log.append(["field", "value"])
    for k in ["annotator name", "role / expertise", "date(s)", "total time (minutes)", "comments"]:
        log.append([k, ""])
    style_header(log, [28, 80])
    if annotator == "A1":
        log.append(["", ""])
        log.append(["note", "A1 annotates ALL units (dev + test)."])
    else:
        log.append(["", ""])
        log.append(["note", "A2 annotates only these units, independently of A1."])

    wb.save(path)


# --------------------------------------------------------------------------
# Requirements checklist (drafted from the text; to be validated by the legal expert)
# Columns: id, provision, applies_to, type, requirement, on-chain
#   on-chain: DATA   = checkable from data submitted to the contract
#             ORACLE = needs certified off-chain input (lab, panel, sensor, inspector)
#             OFF    = not meaningfully enforceable on-chain (physical / visual / legal)
# --------------------------------------------------------------------------
CONSUMER = "consumer oils (EVOO, virgin, composed oil, olive-pomace oil)"
LIMITS = {
    "extra virgin olive oil": [("acidity (%)", "≤ 0,80"), ("peroxide value (mEq O2/kg)", "≤ 20,0"),
                               ("K232", "≤ 2,50"), ("K268 or K270", "≤ 0,22"), ("ΔK", "≤ 0,01"),
                               ("median of defect (Md)", "= 0,0"), ("fruity median (Mf)", "> 0,0"),
                               ("fatty acid ethyl esters (mg/kg)", "≤ 35")],
    "virgin olive oil": [("acidity (%)", "≤ 2,0"), ("peroxide value (mEq O2/kg)", "≤ 20,0"),
                         ("K232", "≤ 2,60"), ("K268 or K270", "≤ 0,25"), ("ΔK", "≤ 0,01"),
                         ("median of defect (Md)", "≤ 3,5"), ("fruity median (Mf)", "> 0,0")],
    "olive oil composed of refined olive oils and virgin olive oils": [
        ("acidity (%)", "≤ 1,00"), ("peroxide value (mEq O2/kg)", "≤ 15,0"),
        ("K268 or K270", "≤ 1,15"), ("ΔK", "≤ 0,15")],
    "olive-pomace oil": [("acidity (%)", "≤ 1,00"), ("peroxide value (mEq O2/kg)", "≤ 15,0"),
                         ("K268 or K270", "≤ 1,70"), ("ΔK", "≤ 0,18")],
}

OTHER_REQS = [
    ("Art. 2(1), Annex I", "all 8 categories", "DEFINITION",
     "An oil belongs to a category only if it meets the characteristics of that category in Annex I, Tables A and B.", "ORACLE"),
    ("Art. 3(1)", CONSUMER, "CONSTRAINT",
     "Only consumer oils may be part of blends of olive oil and other vegetable oils.", "DATA"),
    ("Art. 3(2)", CONSUMER, "CONSTRAINT", "Only consumer oils may be incorporated in other foodstuffs.", "DATA"),
    ("Art. 4(1)", CONSUMER, "OBLIGATION",
     "Packaging presented to the final consumer has a maximum capacity of five litres.", "DATA"),
    ("Art. 4(1)", CONSUMER, "OBLIGATION",
     "Packaging is fitted with an opening system that can no longer be sealed after first opening.", "OFF"),
    ("Art. 4(2)", CONSUMER, "PERMISSION",
     "For restaurants, hospitals, canteens and similar establishments, Member States may allow packaging above five litres.", "DATA"),
    ("Art. 5(1)", CONSUMER, "OBLIGATION",
     "The particulars of Articles 6 to 9 (legal name/category, storage, origin, packaging plant) are mandatory on the label.", "DATA"),
    ("Art. 5(2)", CONSUMER, "OBLIGATION",
     "Legal name and place of origin are grouped in the principal field of vision, in full and in a homogeneous body of text.", "OFF"),
    ("Art. 6(2)", CONSUMER, "OBLIGATION",
     "The label bears the category information text prescribed for the category (e.g. EVOO: 'superior category olive oil "
     "obtained directly from olives and solely by mechanical means').", "DATA"),
    ("Art. 7", CONSUMER, "OBLIGATION",
     "The container or label states that the oil must be stored away from light and heat.", "DATA"),
    ("Art. 8(1)", "extra virgin olive oil, virgin olive oil", "OBLIGATION", "The place of origin appears on the label.", "DATA"),
    ("Art. 8(2)", "composed oil, olive-pomace oil", "PROHIBITION", "No place of origin is displayed on the label.", "DATA"),
    ("Art. 8(3)", "extra virgin olive oil, virgin olive oil", "CONSTRAINT",
     "Place of origin is only: a Member State / the Union / a third country; one of the three prescribed blend indications; "
     "or a PDO/PGI.", "DATA"),
    ("Art. 8(6)", "extra virgin olive oil, virgin olive oil", "OBLIGATION",
     "An origin naming a Member State or the Union corresponds to where olives were harvested and where the mill is.", "ORACLE"),
    ("Art. 8(7)", "extra virgin olive oil, virgin olive oil", "OBLIGATION",
     "If harvest country differs from mill country, the origin uses the prescribed wording "
     "('... obtained in X from olives harvested in Y').", "DATA"),
    ("Art. 8(5)", "imported oils", "OBLIGATION",
     "For imports, origin is determined under Articles 59-63 of Regulation (EU) No 952/2013.", "OFF"),
    ("Art. 9", CONSUMER, "OBLIGATION",
     "Where applicable, the label bears the alphanumeric identification of the approved packaging plant.", "DATA"),
    ("Art. 10(a)", "extra virgin olive oil, virgin olive oil", "CONDITIONAL PERMISSION",
     "'First cold pressing' only if obtained below 27 °C by a first mechanical pressing with hydraulic presses.", "ORACLE"),
    ("Art. 10(b)", "extra virgin olive oil, virgin olive oil", "CONDITIONAL PERMISSION",
     "'Cold extraction' only if obtained below 27 °C by percolation or centrifugation.", "ORACLE"),
    ("Art. 10(c)", "extra virgin olive oil, virgin olive oil", "CONDITIONAL PERMISSION",
     "Organoleptic indications only for EVOO/virgin and only if based on the official panel assessment (Annex II ranges).", "ORACLE"),
    ("Art. 10(d)", CONSUMER, "CONDITIONAL OBLIGATION",
     "If maximum acidity is indicated, maximum peroxide, wax content and UV absorption values are shown alongside.", "DATA"),
    ("Art. 11(1)", "extra virgin olive oil, virgin olive oil", "RESTRICTED PERMISSION",
     "Only EVOO and virgin olive oil may bear the harvesting year.", "DATA"),
    ("Art. 11(2)", "extra virgin olive oil, virgin olive oil", "CONDITIONAL PERMISSION",
     "Harvesting year only if 100 % of the contents come from that harvest; shown as marketing year or month + year.", "ORACLE"),
    ("Art. 11(3)", "Member States", "PERMISSION",
     "Member States may make the harvesting year mandatory for domestic production sold on their national market.", "DATA"),
    ("Art. 12(1)", "blends", "CONDITIONAL OBLIGATION",
     "If olive oil is highlighted outside the ingredients list, the blend bears the prescribed trade description and the percentage.", "DATA"),
    ("Art. 12(2)", "blends", "CONDITIONAL PERMISSION",
     "Images or graphics may highlight olive oil only if it is more than 50 % of the blend.", "DATA"),
    ("Art. 12(3)", "foodstuffs", "CONDITIONAL OBLIGATION",
     "If olive oil is highlighted outside the ingredients list, the food name is followed by the percentage of oil.", "DATA"),
    ("Art. 12(6)", "foodstuffs", "PROHIBITION",
     "Oils with other foodstuffs added do not bear any of the legal names of Article 6.", "DATA"),
]


def checklist_rows():
    rows, n = [], 1
    for cat, lims in LIMITS.items():
        for param, val in lims:
            rows.append([f"R{n:02d}", "Annex I, Table A", cat, "LIMIT", f"{param} {val}", "ORACLE"])
            n += 1
    for prov, applies, typ, req, chain in OTHER_REQS:
        rows.append([f"R{n:02d}", prov, applies, typ, req, chain])
        n += 1
    return rows


def checklist_workbook(path):
    wb = Workbook()
    ws = wb.active
    ws.title = "Instructions"
    ws.column_dimensions["A"].width = 140
    for r, line in enumerate([
        "Reg2Chain - requirements checklist (gold reference for KG coverage and contract coverage)",
        "Scope: the four categories sold to the final consumer (Art. 1(b)) and the labelling/marketing rules of Articles 2-12.",
        "This list is a DRAFT prepared from the regulation text. It does not come from the Reg2Chain system.",
        "For each row, choose in 'expert_decision': KEEP, EDIT (write the corrected text in 'corrected_requirement'), or DELETE.",
        "Check also the 'on_chain' column: DATA = checkable from submitted data; ORACLE = needs certified off-chain input "
        "(lab, panel, sensor, inspector); OFF = not meaningfully enforceable on-chain.",
        "Add any missing requirement at the bottom with expert_decision = ADD.",
        "Record your total time in the 'Log' sheet. These numbers are reported in the paper (expert effort).",
    ], start=1):
        c = ws.cell(row=r, column=1, value=line)
        c.alignment = WRAP
        if r == 1:
            c.font = Font(bold=True, size=14, color="2F5496")

    cl = wb.create_sheet("Checklist")
    cl.append(["req_id", "provision", "applies_to", "type", "requirement", "on_chain",
               "expert_decision", "corrected_requirement", "corrected_on_chain", "reason / comment"])
    for row in checklist_rows():
        cl.append(row + ["", "", "", ""])
    style_header(cl, [8, 18, 34, 22, 80, 10, 16, 60, 16, 40])
    for row in cl.iter_rows(min_row=2):
        for c in row:
            c.alignment = WRAP
    add_list_validation(cl, "G", ["KEEP", "EDIT", "DELETE", "ADD"], rows=300)
    add_list_validation(cl, "F", ["DATA", "ORACLE", "OFF"], rows=300)
    add_list_validation(cl, "I", ["DATA", "ORACLE", "OFF"], rows=300)

    log = wb.create_sheet("Log")
    log.append(["field", "value"])
    for k in ["expert name", "expertise", "date(s)", "total time (minutes)", "comments"]:
        log.append([k, ""])
    style_header(log, [28, 80])
    wb.save(path)
    return len(checklist_rows())


def main(units_csv, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    units = list(csv.DictReader(open(units_csv, encoding="utf-8")))
    a2 = [u for u in units if u["double_annotation"] == "Y"]
    annotation_workbook(units, os.path.join(out_dir, "Reg2Chain_annotation_A1.xlsx"), "A1")
    annotation_workbook(a2, os.path.join(out_dir, "Reg2Chain_annotation_A2.xlsx"), "A2")
    n = checklist_workbook(os.path.join(out_dir, "Reg2Chain_requirements_checklist.xlsx"))
    print(f"A1: {len(units)} units | A2: {len(a2)} units | checklist: {n} requirements -> {out_dir}")


if __name__ == "__main__":
    main(*sys.argv[1:3])


In [ ]:
%%writefile scripts/check_and_build_gold.py
"""
Step 1 - Check annotations, build the gold standard and compute inter-annotator agreement.

  python check_and_build_gold.py units.csv Reg2Chain_annotation_A1.xlsx [Reg2Chain_annotation_A2.xlsx] OUT_DIR

Checks (printed as a list of problems to fix in the workbook):
  - unit_id exists; entity text found in the unit (exact copy) at the given occurrence
  - entity type / relation type valid; entities do not overlap
  - relation subject/object (and qualifier) are annotated entities of the same unit
  - every unit has at least one row (or type NONE)

Outputs:
  gold_dev.jsonl, gold_test.jsonl : {"unit_id","sentence","entities":[{text,label,start,end}],
                                     "relations":[{subject,predicate,object,qualifier}]}
  agreement.txt                   : A1 vs A2 on the double-annotated units
     - entity span agreement (strict and untyped F1, A1 as reference)
     - Cohen's kappa at token level (entity type per token, 'O' outside entities)
     - relation agreement (strict F1)
"""
import csv
import json
import os
import re
import sys
from collections import Counter

from openpyxl import load_workbook

ENTITY_TYPES = {"PRODUCT", "PARAMETER", "VALUE", "LABEL_ITEM", "ACTOR", "LEGAL_REF", "PROCESS"}
RELATION_TYPES = {"OBLIGATION", "PERMISSION", "PROHIBITION", "LIMIT", "CONDITION", "REFERS_TO"}


def cell(v):
    return "" if v is None else str(v).strip()


def norm(s):
    return re.sub(r"\s+", " ", s.strip()).lower()


def read_sheet(wb, name):
    ws = wb[name]
    rows = list(ws.iter_rows(values_only=True))
    header = [cell(h) for h in rows[0]]
    return [dict(zip(header, [cell(v) for v in r])) for r in rows[1:] if any(v is not None for v in r)], header


def find_occurrence(text, span, occ, placed):
    """occ-th occurrence of span that does not overlap an already placed (longer) entity."""
    found, start = 0, -1
    while True:
        start = text.find(span, start + 1)
        if start < 0:
            return None
        end = start + len(span)
        # skip occurrences inside a longer, different entity; same-text entities still count
        if all(end <= e["start"] or start >= e["end"] or e["text"] == span for e in placed):
            found += 1
            if found == max(1, occ):
                return start


def load_annotations(path, units):
    wb = load_workbook(path, data_only=True)
    ents_rows, eh = read_sheet(wb, "Entities")
    rels_rows, rh = read_sheet(wb, "Relations")
    e_text, e_type, e_occ = eh[1], eh[2], eh[3]
    r_subj, r_rel, r_obj, r_q = rh[1], rh[2], rh[3], rh[4]
    problems = []
    ents = {u: [] for u in units}
    rels = {u: [] for u in units}
    seen_units = set()

    # longest spans first, so a short entity is never placed inside a longer one
    order = sorted(enumerate(ents_rows, start=2), key=lambda x: -len(x[1][e_text]))
    for i, r in order:
        uid, span, typ = r["unit_id"], r[e_text], r[e_type].upper()
        if uid not in units:
            problems.append(f"Entities row {i}: unknown unit_id '{uid}'")
            continue
        seen_units.add(uid)
        if typ == "NONE":
            continue
        if typ not in ENTITY_TYPES:
            problems.append(f"Entities row {i} ({uid}): invalid type '{typ}'")
            continue
        try:
            occ = int(float(r[e_occ])) if r[e_occ] else 1
        except ValueError:
            occ = 1
        start = find_occurrence(units[uid]["text"], span, occ, ents[uid])
        if start is None:
            problems.append(f"Entities row {i} ({uid}): text not found exactly in unit: '{span}'")
            continue
        ents[uid].append({"text": span, "label": typ, "start": start, "end": start + len(span)})

    for uid, lst in ents.items():
        lst.sort(key=lambda e: e["start"])
        for a, b in zip(lst, lst[1:]):
            if b["start"] < a["end"]:
                problems.append(f"{uid}: overlapping entities '{a['text']}' and '{b['text']}'")

    for i, r in enumerate(rels_rows, start=2):
        uid, rel = r["unit_id"], r[r_rel].upper()
        if uid not in units:
            problems.append(f"Relations row {i}: unknown unit_id '{uid}'")
            continue
        seen_units.add(uid)
        if rel not in RELATION_TYPES:
            problems.append(f"Relations row {i} ({uid}): invalid relation '{rel}'")
            continue
        names = {norm(e["text"]) for e in ents[uid]}
        for role, val in (("subject", r[r_subj]), ("object", r[r_obj]), ("qualifier", r[r_q])):
            if role == "qualifier" and not val:
                continue
            if norm(val) not in names:
                problems.append(f"Relations row {i} ({uid}): {role} '{val}' is not an annotated entity of this unit")
        rels[uid].append({"subject": r[r_subj], "predicate": rel, "object": r[r_obj], "qualifier": r[r_q]})

    for uid in units:
        if uid not in seen_units:
            problems.append(f"{uid}: no annotation row (add a row with type NONE if the unit has no entity)")
    return ents, rels, problems


# ---------------------------------------------------------------- agreement
def prf(a, b):
    """F1 of set/multiset b against reference a."""
    ca, cb = Counter(a), Counter(b)
    tp = sum((ca & cb).values())
    p = tp / max(1, sum(cb.values()))
    r = tp / max(1, sum(ca.values()))
    return tp, sum(ca.values()), sum(cb.values()), (2 * p * r / (p + r) if p + r else 0.0)


def token_labels(text, ents):
    labels = []
    for m in re.finditer(r"\w+|[^\w\s]", text):
        lab = "O"
        for e in ents:
            if e["start"] <= m.start() < e["end"]:
                lab = e["label"]
                break
        labels.append(lab)
    return labels


def cohen_kappa(x, y):
    n = len(x)
    po = sum(a == b for a, b in zip(x, y)) / n
    cx, cy = Counter(x), Counter(y)
    pe = sum(cx[k] * cy[k] for k in set(cx) | set(cy)) / (n * n)
    return (po - pe) / (1 - pe) if pe < 1 else 1.0


def agreement(units, e1, r1, e2, r2, ids):
    lines = [f"Double-annotated units: {len(ids)}"]
    s1 = [(u, e["start"], e["end"], e["label"]) for u in ids for e in e1[u]]
    s2 = [(u, e["start"], e["end"], e["label"]) for u in ids for e in e2[u]]
    tp, n1, n2, f = prf(s1, s2)
    lines.append(f"Entities strict (span+type): A1={n1} A2={n2} agreed={tp} F1={f:.3f}")
    tp, n1, n2, f = prf([x[:3] for x in s1], [x[:3] for x in s2])
    lines.append(f"Entities untyped (span only): agreed={tp} F1={f:.3f}")
    t1, t2 = [], []
    for u in ids:
        t1 += token_labels(units[u]["text"], e1[u])
        t2 += token_labels(units[u]["text"], e2[u])
    lines.append(f"Cohen's kappa (token level, {len(t1)} tokens): {cohen_kappa(t1, t2):.3f}")
    key = lambda r: (norm(r["subject"]), r["predicate"], norm(r["object"]), norm(r["qualifier"]))
    q1 = [(u,) + key(r) for u in ids for r in r1[u]]
    q2 = [(u,) + key(r) for u in ids for r in r2[u]]
    tp, n1, n2, f = prf(q1, q2)
    lines.append(f"Relations strict: A1={n1} A2={n2} agreed={tp} F1={f:.3f}")
    return "\n".join(lines)


def main(units_csv, a1_path, *rest):
    a2_path = rest[0] if len(rest) == 2 else None
    out_dir = rest[-1]
    os.makedirs(out_dir, exist_ok=True)
    rows = list(csv.DictReader(open(units_csv, encoding="utf-8")))
    units = {r["unit_id"]: r for r in rows}

    e1, r1, p1 = load_annotations(a1_path, units)
    print(f"A1: {sum(map(len, e1.values()))} entities, {sum(map(len, r1.values()))} relations, {len(p1)} problems")
    for p in p1:
        print("  A1 -", p)

    for split in ("dev", "test"):
        with open(os.path.join(out_dir, f"gold_{split}.jsonl"), "w", encoding="utf-8") as f:
            for r in rows:
                if r["split"] == split:
                    f.write(json.dumps({"unit_id": r["unit_id"], "sentence": r["text"],
                                        "entities": e1[r["unit_id"]], "relations": r1[r["unit_id"]]},
                                       ensure_ascii=False) + "\n")
    labels = Counter(e["label"] for u in e1 for e in e1[u] if units[u]["split"] == "test")
    rels = Counter(r["predicate"] for u in r1 for r in r1[u] if units[u]["split"] == "test")
    print("Test entities per type:", dict(labels.most_common()))
    print("Test relations per type:", dict(rels.most_common()))

    if a2_path:
        a2_units = {u: units[u] for u in units if units[u]["double_annotation"] == "Y"}
        e2, r2, p2 = load_annotations(a2_path, a2_units)
        print(f"A2: {sum(map(len, e2.values()))} entities, {sum(map(len, r2.values()))} relations, {len(p2)} problems")
        for p in p2:
            print("  A2 -", p)
        rep = agreement(units, e1, r1, e2, r2, sorted(a2_units))
        open(os.path.join(out_dir, "agreement.txt"), "w").write(rep + "\n")
        print(rep)


if __name__ == "__main__":
    main(*sys.argv[1:])


In [ ]:
%%writefile scripts/compare_with_preannotation.py
"""
Step 1 - Measure how much the annotator corrected the LLM pre-annotation (reported in the paper).

  python compare_with_preannotation.py units.csv preannotation_claude_A1.xlsx Reg2Chain_annotation_A1.xlsx OUT_DIR

Counts, for entities (span + type) and relations (subject, relation, object, qualifier):
  kept      : present in both
  deleted   : in the pre-annotation, removed by the annotator
  added     : added by the annotator
  retyped   : same entity span, different type (entities only)
Writes correction_report.txt and a per-unit CSV of changes.
"""
import csv
import os
import sys
from collections import Counter

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from check_and_build_gold import load_annotations, norm  # noqa: E402


def main(units_csv, pre_path, final_path, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    units = {r["unit_id"]: r for r in csv.DictReader(open(units_csv, encoding="utf-8"))}
    pe, pr, _ = load_annotations(pre_path, units)
    fe, fr, probs = load_annotations(final_path, units)
    if probs:
        print(f"{len(probs)} problems in the final workbook: fix them first (run check_and_build_gold.py)")

    rows, tot = [], Counter()
    for u in units:
        P = Counter((e["start"], e["end"], e["label"]) for e in pe[u])
        F = Counter((e["start"], e["end"], e["label"]) for e in fe[u])
        kept, dele, add = P & F, P - F, F - P
        spans_d = Counter((s, e) for s, e, _ in dele.elements())
        spans_a = Counter((s, e) for s, e, _ in add.elements())
        retyped = sum((spans_d & spans_a).values())
        tot["ent_pre"] += sum(P.values()); tot["ent_final"] += sum(F.values())
        tot["ent_kept"] += sum(kept.values()); tot["ent_retyped"] += retyped
        tot["ent_deleted"] += sum(dele.values()) - retyped; tot["ent_added"] += sum(add.values()) - retyped
        key = lambda r: (norm(r["subject"]), r["predicate"], norm(r["object"]), norm(r["qualifier"]))
        RP, RF = Counter(map(key, pr[u])), Counter(map(key, fr[u]))
        tot["rel_pre"] += sum(RP.values()); tot["rel_final"] += sum(RF.values())
        tot["rel_kept"] += sum((RP & RF).values())
        tot["rel_deleted"] += sum((RP - RF).values()); tot["rel_added"] += sum((RF - RP).values())
        changed = (P != F) or (RP != RF)
        tot["units_changed"] += changed
        rows.append([u, units[u]["split"], sum(P.values()), sum(F.values()), retyped,
                     sum((RP - RF).values()), sum((RF - RP).values()), "Y" if changed else "N"])

    with open(os.path.join(out_dir, "corrections_per_unit.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["unit_id", "split", "entities_pre", "entities_final", "entities_retyped",
                    "relations_deleted", "relations_added", "changed"])
        w.writerows(rows)

    pct = lambda a, b: f"{100 * a / b:.1f} %" if b else "n/a"
    ent_changes = tot["ent_deleted"] + tot["ent_added"] + tot["ent_retyped"]
    rel_changes = tot["rel_deleted"] + tot["rel_added"]
    rep = "\n".join([
        f"Units changed by the annotator: {tot['units_changed']}/{len(units)} ({pct(tot['units_changed'], len(units))})",
        f"Entities: pre-annotated {tot['ent_pre']}, final {tot['ent_final']}",
        f"  kept {tot['ent_kept']} | retyped {tot['ent_retyped']} | deleted {tot['ent_deleted']} | added {tot['ent_added']}",
        f"  correction rate = (retyped + deleted + added) / final = {pct(ent_changes, tot['ent_final'])}",
        f"Relations: pre-annotated {tot['rel_pre']}, final {tot['rel_final']}",
        f"  kept {tot['rel_kept']} | deleted {tot['rel_deleted']} | added {tot['rel_added']}",
        f"  correction rate = (deleted + added) / final = {pct(rel_changes, tot['rel_final'])}",
    ])
    open(os.path.join(out_dir, "correction_report.txt"), "w").write(rep + "\n")
    print(rep)


if __name__ == "__main__":
    main(*sys.argv[1:5])


In [ ]:
%%writefile scripts/preannotation_claude.py
"""
Pre-annotation of the 63 units (drafted by Claude, Anthropic) - to be reviewed and corrected by annotator A1.
NOT given to annotator A2 (independent annotation).

E(text, type, occurrence=1)      R(subject, relation, object, qualifier="")
"""

def E(t, ty, occ=1, note=""):
    return (t, ty, occ, note)


def R(s, r, o, q="", note=""):
    return (s, r, o, q, note)


P1B = "oils referred to in Article 1, point (b)"
REG1308 = "Regulation (EU) No 1308/2013"


def art2(point, product, part):
    ref1 = f"point {point} of Tables A and B of Annex I to this Regulation"
    ref2 = f"Part VIII, {part}, of Annex VII to {REG1308}"
    return ([E(ref1, "LEGAL_REF"), E(product, "PRODUCT"), E(ref2, "LEGAL_REF")],
            [R(product, "REFERS_TO", ref1), R(product, "REFERS_TO", ref2)])


def annex(product, pairs):
    ents = [E(product, "PRODUCT")]
    rels = []
    for param, val in pairs:
        ents += [E(param, "PARAMETER"), E(val, "VALUE")]
        rels.append(R(product, "LIMIT", val, param))
    return ents, rels


A = {}

A["ART1"] = ([
    E("characteristics", "PARAMETER"),
    E(f"olive oils referred to in Part VIII, points 1 to 6, of Annex VII to {REG1308}", "PRODUCT"),
    E(f"olive oils referred to in Part VIII, points 1(a) and (b) and points 3 and 6, of Annex VII to {REG1308}", "PRODUCT"),
    E("final consumer", "ACTOR"),
    E("foodstuff", "PRODUCT"),
], [], "scope provision: no deontic relation")

A["ART2.1.lead"] = ([E("Olive oils", "PRODUCT"), E("characteristics", "PARAMETER")], [], "")
for k, (pt, prod, part) in {
    "a": (1, "extra virgin olive oil", "point 1(a)"), "b": (2, "virgin olive oil", "point 1(b)"),
    "c": (3, "lampante olive oil", "point 1(c)"), "d": (4, "refined olive oil", "point 2"),
    "e": (5, "olive oil composed of refined olive oils and virgin olive oils", "point 3"),
    "f": (6, "crude olive-pomace oil", "point 4"), "g": (7, "refined olive-pomace oil", "point 5"),
    "h": (8, "olive-pomace oil", "point 6"),
}.items():
    e, r = art2(pt, prod, part)
    A[f"ART2.1.{k}"] = (e, r, "")

A["ART2.2"] = ([E("characteristics of olive oils laid down in Annex I", "PARAMETER"),
                E("Commission Implementing Regulation (EU) 2022/2105", "LEGAL_REF")],
               [R("characteristics of olive oils laid down in Annex I", "REFERS_TO",
                  "Commission Implementing Regulation (EU) 2022/2105")], "")

A["ART3.1"] = ([E(P1B, "PRODUCT"), E("blends of olive oil and other vegetable oils", "PRODUCT")],
               [R(P1B, "PERMISSION", "blends of olive oil and other vegetable oils")], "")
A["ART3.2"] = ([E(P1B, "PRODUCT"), E("other foodstuffs", "PRODUCT")],
               [R(P1B, "PERMISSION", "other foodstuffs")], "")
A["ART3.3"] = ([E("Member States", "ACTOR"),
                E("blends of olive oil and other vegetable oils referred to in paragraph 1", "PRODUCT"),
                E("production", "PROCESS", 1), E("marketing", "PROCESS", 1), E("production", "PROCESS", 2),
                E("such blends", "PRODUCT", 1), E("such blends", "PRODUCT", 2)],
               [], "rule on Member States' powers: no relation of the schema fits")

A["ART4.1"] = ([E("Oils referred to in Article 1, point (b)", "PRODUCT"), E("final consumer", "ACTOR"), E("maximum capacity", "PARAMETER"),
                E("five litres", "VALUE"),
                E("opening system that can no longer be sealed after the first time it is opened", "PROCESS")],
               [R("Oils referred to in Article 1, point (b)", "LIMIT", "five litres", "maximum capacity"),
                R("Oils referred to in Article 1, point (b)", "OBLIGATION",
                  "opening system that can no longer be sealed after the first time it is opened")], "")
A["ART4.2"] = ([E(P1B, "PRODUCT"),
                E("restaurants, hospitals, canteens and other similar collective establishments", "ACTOR"),
                E("Member States", "ACTOR"), E("maximum capacity", "PARAMETER"),
                E("exceeding five litres", "VALUE")],
               [R("Member States", "PERMISSION", "exceeding five litres", "maximum capacity")], "")

A["ART5.1"] = ([E("particulars referred to in Articles 6 to 9", "LABEL_ITEM")], [], "obligation without subject entity")
A["ART5.2"] = ([E("legal name referred to in Article 6(1)", "LABEL_ITEM"),
                E("place of origin referred to in Article 8(1)", "LABEL_ITEM"),
                E("Article 2(2), point (l), of Regulation (EU) No 1169/2011", "LEGAL_REF")], [], "")
A["ART5.3"] = ([E("particulars referred to in Articles 10, 11 and 12", "LABEL_ITEM")], [], "")
A["ART6.1"] = ([E(f"description of the {P1B}", "LABEL_ITEM"), E("legal name", "LABEL_ITEM"),
                E("Article 2(2), point (n), of Regulation (EU) No 1169/2011", "LEGAL_REF")],
               [R("legal name", "REFERS_TO", "Article 2(2), point (n), of Regulation (EU) No 1169/2011")], "")
A["ART6.2.lead"] = ([E("those oils", "PRODUCT"), E("description referred to in paragraph 1", "LABEL_ITEM"),
                     E("information on the category of oil", "LABEL_ITEM")],
                    [R("those oils", "OBLIGATION", "information on the category of oil")], "")
for k, prod, txt in [
    ("a", "extra virgin olive oil", "‘superior category olive oil obtained directly from olives and solely by mechanical means’"),
    ("b", "virgin olive oil", "‘olive oil obtained directly from olives and solely by mechanical means’"),
    ("c", "olive oil composed of refined olive oils and virgin olive oils",
     "‘oil comprising exclusively olive oils that have undergone refining and oils obtained directly from olives’"),
]:
    A[f"ART6.2.{k}"] = ([E(prod, "PRODUCT"), E(txt, "LABEL_ITEM")], [R(prod, "OBLIGATION", txt)], "")
T1 = "‘oil comprising exclusively oils obtained by treating the product obtained after the extraction of olive oil and oils obtained directly from olives’"
T2 = "‘oil comprising exclusively oils obtained by processing olive pomace and oils obtained directly from olives’"
A["ART6.2.d"] = ([E("olive-pomace oil", "PRODUCT"), E(T1, "LABEL_ITEM"), E(T2, "LABEL_ITEM")],
                 [R("olive-pomace oil", "OBLIGATION", T1, note="alternative (i)"),
                  R("olive-pomace oil", "OBLIGATION", T2, note="alternative (ii)")], "")
A["ART7"] = ([E("Information on the special storage conditions", "LABEL_ITEM"), E(P1B, "PRODUCT")],
             [R(P1B, "OBLIGATION", "Information on the special storage conditions")], "")

REF8 = f"Part VIII, points 1(a) and (b), of Annex VII to {REG1308}"
A["ART8.1"] = ([E("extra virgin olive oil", "PRODUCT"), E("virgin olive oil", "PRODUCT"), E(REF8, "LEGAL_REF"),
                E("place of origin", "LABEL_ITEM")],
               [R("extra virgin olive oil", "OBLIGATION", "place of origin"),
                R("virgin olive oil", "OBLIGATION", "place of origin"),
                R("extra virgin olive oil", "REFERS_TO", REF8), R("virgin olive oil", "REFERS_TO", REF8)], "")
P82 = f"oils referred to in Part VIII, points 3 and 6, of Annex VII to {REG1308}"
A["ART8.2"] = ([E(P82, "PRODUCT"), E("place of origin", "LABEL_ITEM")],
               [R(P82, "PROHIBITION", "place of origin")], "")
A["ART8.3.lead"] = ([E("place of origin referred to in paragraph 1", "LABEL_ITEM")], [], "")
L83a = "reference to that Member State, the Union or the third country"
A["ART8.3.a"] = ([E("olive oils", "PRODUCT"), E("paragraphs 6 and 7", "LEGAL_REF"), E(L83a, "LABEL_ITEM")],
                 [R("olive oils", "PERMISSION", L83a)], "'shall only consist of' = restricted permission")
B = ["‘blend of olive oils of European Union origin’", "reference to the Union",
     "‘blend of olive oils not of European Union origin’", "reference to origin outside the Union",
     "‘blend of olive oils of European Union origin and not of European Union origin’",
     "reference to origin within the Union and outside the Union"]
A["ART8.3.b"] = ([E("blends of olive oils", "PRODUCT"), E("paragraphs 6 and 7", "LEGAL_REF")]
                 + [E(b, "LABEL_ITEM") for b in B],
                 [R("blends of olive oils", "PERMISSION", b) for b in B], "")
A["ART8.3.c"] = ([E("protected designation of origin", "LABEL_ITEM"), E("protected geographical indication", "LABEL_ITEM"),
                  E("Regulation (EU) No 1151/2012", "LEGAL_REF")],
                 [R("protected designation of origin", "REFERS_TO", "Regulation (EU) No 1151/2012"),
                  R("protected geographical indication", "REFERS_TO", "Regulation (EU) No 1151/2012")], "")
A["ART8.4"] = ([E("names of brands or firms", "LABEL_ITEM"), E("Directive 89/104/EEC", "LEGAL_REF"),
                E("Council Regulation (EC) No 40/94", "LEGAL_REF"), E("place of origin", "LABEL_ITEM")],
               [R("names of brands or firms", "PROHIBITION", "place of origin", note="cannot count as place of origin"),
                R("names of brands or firms", "REFERS_TO", "Directive 89/104/EEC"),
                R("names of brands or firms", "REFERS_TO", "Council Regulation (EC) No 40/94")], "")
A["ART8.5"] = ([E("place of origin", "LABEL_ITEM"), E("Articles 59 to 63 of Regulation (EU) No 952/2013", "LEGAL_REF")],
               [R("place of origin", "REFERS_TO", "Articles 59 to 63 of Regulation (EU) No 952/2013")], "")
A["ART8.6"] = ([E("place of origin mentioning a Member State or the Union", "LABEL_ITEM"), E("mill", "ACTOR")], [], "")
W87 = ("‘(extra) virgin olive oil obtained in (the Union or the name of the Member State or third country concerned) "
       "from olives harvested in (the Union or the name of the Member State or third country concerned)’")
A["ART8.7"] = ([E("mill", "ACTOR"), E("place of origin", "LABEL_ITEM"), E(W87, "LABEL_ITEM")],
               [R("place of origin", "OBLIGATION", W87)], "")
A["ART9"] = ([E(P1B, "PRODUCT"), E("alphanumeric identification of the packaging plant", "LABEL_ITEM"),
              E("Article 6 of Commission Implementing Regulation (EU) 2022/2105", "LEGAL_REF")],
             [R(P1B, "OBLIGATION", "alphanumeric identification of the packaging plant"),
              R("alphanumeric identification of the packaging plant", "REFERS_TO",
                "Article 6 of Commission Implementing Regulation (EU) 2022/2105")], "")
P10 = f"{P1B}, of this Regulation"
A["ART10.lead"] = ([E("optional reserved terms", "LABEL_ITEM"), E(f"Annex IX to {REG1308}", "LEGAL_REF"),
                    E(P10, "PRODUCT")],
                   [R("optional reserved terms", "REFERS_TO", f"Annex IX to {REG1308}"),
                    R(P10, "PERMISSION", "optional reserved terms")], "")
EV = "extra virgin or virgin olive oils"
A["ART10.a"] = ([E("‘first cold pressing’", "LABEL_ITEM"), E(EV, "PRODUCT"), E("temperature", "PARAMETER"),
                 E("below 27 °C", "VALUE"), E("first mechanical pressing of the olive paste", "PROCESS"),
                 E("traditional extraction system using hydraulic presses", "PROCESS")],
                [R(EV, "PERMISSION", "‘first cold pressing’"),
                 R("‘first cold pressing’", "CONDITION", "below 27 °C", "temperature"),
                 R("‘first cold pressing’", "CONDITION", "first mechanical pressing of the olive paste"),
                 R("‘first cold pressing’", "CONDITION", "traditional extraction system using hydraulic presses")], "")
A["ART10.b"] = ([E("‘cold extraction’", "LABEL_ITEM"), E(EV, "PRODUCT"), E("temperature", "PARAMETER"),
                 E("below 27 ° C", "VALUE"), E("percolation", "PROCESS"), E("centrifugation", "PROCESS")],
                [R(EV, "PERMISSION", "‘cold extraction’"),
                 R("‘cold extraction’", "CONDITION", "below 27 ° C", "temperature"),
                 R("‘cold extraction’", "CONDITION", "percolation"),
                 R("‘cold extraction’", "CONDITION", "centrifugation")], "")
IND = "indications of organoleptic characteristics referring to taste or smell"
OC = f"organoleptic characteristics as defined in Annex IX to {REG1308}"
ASS = "assessment carried out following the method referred to in Annex I, point 5, of Commission Implementing Regulation (EU) 2022/2105"
A["ART10.c"] = ([E(IND, "LABEL_ITEM"), E("extra virgin and virgin olive oils", "PRODUCT"), E(OC, "LABEL_ITEM"),
                 E(ASS, "PROCESS"), E("Annex II to this Regulation", "LEGAL_REF")],
                [R("extra virgin and virgin olive oils", "PERMISSION", IND), R(OC, "CONDITION", ASS)], "")
MA = "indication of the maximum acidity expected by the date of minimum durability"
MV = "maximum values for peroxide, wax content and ultraviolet absorption"
A["ART10.d"] = ([E(MA, "LABEL_ITEM"), E("Article 9(1), point (f), of Regulation (EU) No 1169/2011", "LEGAL_REF"),
                 E(MV, "LABEL_ITEM"), E("Commission Implementing Regulation (EU) 2022/2105", "LEGAL_REF")],
                [R(MA, "CONDITION", MV), R(MV, "REFERS_TO", "Commission Implementing Regulation (EU) 2022/2105")], "")
A["ART11.1"] = ([E("extra virgin olive oil", "PRODUCT"), E("virgin olive oil", "PRODUCT"), E(REF8, "LEGAL_REF"),
                 E("indication of the harvesting year", "LABEL_ITEM")],
                [R("extra virgin olive oil", "PERMISSION", "indication of the harvesting year"),
                 R("virgin olive oil", "PERMISSION", "indication of the harvesting year"),
                 R("extra virgin olive oil", "REFERS_TO", REF8), R("virgin olive oil", "REFERS_TO", REF8)], "")
A["ART11.2"] = ([E("harvesting year", "LABEL_ITEM"), E("100 %", "VALUE"), E("contents of the container", "PARAMETER"),
                 E("relevant marketing year", "LABEL_ITEM"),
                 E(f"Article 6, point (f), of {REG1308}", "LEGAL_REF"),
                 E("month and the year of the harvest", "LABEL_ITEM")],
                [R("harvesting year", "CONDITION", "100 %", "contents of the container"),
                 R("relevant marketing year", "REFERS_TO", f"Article 6, point (f), of {REG1308}")], "")
A["ART11.3"] = ([E("Member States", "ACTOR"), E("harvesting year referred to in paragraph 1", "LABEL_ITEM"),
                 E("olive oils referred to in that paragraph", "PRODUCT")],
                [R("Member States", "PERMISSION", "harvesting year referred to in paragraph 1")], "")
A["ART11.4"] = ([E("olive oils labelled prior to the date on which that decision takes effect", "PRODUCT")], [], "")
A["ART11.5"] = ([E("Member States", "ACTOR"), E("Commission", "ACTOR"),
                 E("Delegated Regulation (EU) 2017/1183", "LEGAL_REF")],
                [R("Member States", "OBLIGATION", "Commission", note="obligation to notify the Commission")], "")
TD = "‘Blend of vegetable oils (or the specific names of the vegetable oils concerned) and olive oil’"
A["ART12.1"] = ([E(P1B, "PRODUCT"), E("blend with other vegetable oils", "PRODUCT"), E("blend concerned", "PRODUCT"),
                 E(TD, "LABEL_ITEM"), E("percentage of such oils in the blend", "LABEL_ITEM")],
                [R("blend concerned", "OBLIGATION", TD),
                 R("blend concerned", "OBLIGATION", "percentage of such oils in the blend")], "")
A["ART12.2"] = ([E(P1B, "PRODUCT"), E("images or graphics", "LABEL_ITEM"),
                 E("blend referred to in paragraph 1", "PRODUCT"), E("more than 50 %", "VALUE")],
                [R("blend referred to in paragraph 1", "PERMISSION", "images or graphics"),
                 R("images or graphics", "CONDITION", "more than 50 %")], "")
PO = "percentage of oil relative to the total net weight of the foodstuff"
A["ART12.3"] = ([E("solid foodstuffs preserved exclusively in olive oil", "PRODUCT"),
                 E("Council Regulations (EEC) No 2136/89 (12) and (EEC) No 1536/92 (13)", "LEGAL_REF"),
                 E(f"{P1B}, of this Regulation", "PRODUCT"), E("name of the food", "LABEL_ITEM"), E(PO, "LABEL_ITEM")],
                [R("name of the food", "OBLIGATION", PO)], "")
PA1 = "percentage of added oils referred to in Article 1, point (b), relative to the total net weight of the foodstuff referred to in paragraph 3 of this Article"
PA2 = "percentage of added oil relative to the total weight of fats"
A["ART12.4"] = ([E(PA1, "LABEL_ITEM"), E(PA2, "LABEL_ITEM"), E("‘percentage of fats’", "LABEL_ITEM")],
                [R(PA1, "PERMISSION", PA2, note="may be replaced by")], "")
A["ART12.5"] = ([E("descriptions referred to in Article 6(1)", "LABEL_ITEM"), E("‘olive oil’", "LABEL_ITEM", 1),
                 E("products referred to in paragraphs 1 and 3 of this Article", "PRODUCT"),
                 E("olive-pomace oil", "PRODUCT"), E("‘olive oil’", "LABEL_ITEM", 2),
                 E("‘olive-pomace oil’", "LABEL_ITEM")],
                [R("products referred to in paragraphs 1 and 3 of this Article", "PERMISSION", "‘olive oil’"),
                 R("olive-pomace oil", "OBLIGATION", "‘olive-pomace oil’")], "")
A["ART12.6"] = ([E("other foodstuffs", "PRODUCT"), E(P1B, "PRODUCT"), E("resulting foodstuff", "PRODUCT"),
                 E("legal names referred to in Article 6", "LABEL_ITEM")],
                [R("resulting foodstuff", "PROHIBITION", "legal names referred to in Article 6")], "")
A["ART13"] = ([E("Regulation (EEC) No 2568/91", "LEGAL_REF"), E("Implementing Regulation (EU) No 29/2012", "LEGAL_REF"),
               E("Commission Implementing Regulation (EU) 2022/2105", "LEGAL_REF"),
               E("correlation table in Annex III to this Regulation", "LEGAL_REF")], [],
              "repeal: no relation of the schema fits")
A["ART14"] = ([], [], "no entity")

ANX = {
    "1": ("extra virgin olive oil", [("Acidity (%)", "≤ 0,80"), ("Peroxide value (mEq O2/kg)", "≤ 20,0"),
                                     ("K232", "≤ 2,50"), ("K268 or K270", "≤ 0,22"), ("ΔK", "≤ 0,01"),
                                     ("Median of defect (Md)", "= 0,0"), ("Fruity median (Mf)", "> 0,0"),
                                     ("Fatty acid ethyl esters (mg/kg)", "≤ 35")]),
    "2": ("virgin olive oil", [("Acidity (%)", "≤ 2,0"), ("Peroxide value (mEq O2/kg)", "≤ 20,0"),
                               ("K232", "≤ 2,60"), ("K268 or K270", "≤ 0,25"), ("ΔK", "≤ 0,01"),
                               ("Median of defect (Md)", "≤ 3,5"), ("Fruity median (Mf)", "> 0,0")]),
    "3": ("lampante olive oil", [("Acidity (%)", "> 2,0"), ("Median of defect (Md)", "> 3,5")]),
    "4": ("refined olive oil", [("Acidity (%)", "≤ 0,30"), ("Peroxide value (mEq O2/kg)", "≤ 5,0"),
                                ("K268 or K270", "≤ 1,25"), ("ΔK", "≤ 0,16")]),
    "5": ("olive oil composed of refined olive oil and virgin olive oils",
          [("Acidity (%)", "≤ 1,00"), ("Peroxide value (mEq O2/kg)", "≤ 15,0"), ("K268 or K270", "≤ 1,15"), ("ΔK", "≤ 0,15")]),
    "6": ("crude olive-pomace oil", []),
    "7": ("refined olive-pomace oil", [("Acidity (%)", "≤ 0,30"), ("Peroxide value (mEq O2/kg)", "≤ 5,0"),
                                       ("K268 or K270", "≤ 2,00"), ("ΔK", "≤ 0,20")]),
    "8": ("olive-pomace oil", [("Acidity (%)", "≤ 1,00"), ("Peroxide value (mEq O2/kg)", "≤ 15,0"),
                               ("K268 or K270", "≤ 1,70"), ("ΔK", "≤ 0,18")]),
}
for k, (prod, pairs) in ANX.items():
    e, r = annex(prod, pairs)
    A[f"ANX1.A.{k}"] = (e, r, "")


### 1.3 Segment the regulation into provision units
Articles 1–14 (one unit per numbered paragraph; long paragraphs split into lettered points) + Annex I, Table A (one unit per category). Recitals excluded. A fixed seed makes the dev/test split reproducible.

Expected: dev 16, test 47, of which 14 double-annotated.

In [ ]:
!python scripts/segment_regulation.py data/regulation_text.txt step1_gold/units.csv

import pandas as pd
units = pd.read_csv('step1_gold/units.csv')
print(units.groupby(['split', 'double_annotation']).size())
units.head(10)

### 1.4 Annotation workbooks and requirements checklist
- `Reg2Chain_annotation_A1.xlsx`: annotator 1, all units
- `Reg2Chain_annotation_A2.xlsx`: legal expert, double-annotated units only, without access to A1
- `Reg2Chain_requirements_checklist.xlsx`: legal expert and blockchain developer (KEEP / EDIT / DELETE / ADD)

Note: running this cell again overwrites the annotated workbooks.

In [ ]:
!python scripts/build_annotation_workbooks.py step1_gold/units.csv step1_gold
print(sorted(os.listdir('step1_gold')))

### 1.5 Check the annotations and compute inter-annotator agreement (before calibration)
A1 (`step1_gold/annotated/Reg2Chain_annotation_A1.xlsx`): LLM pre-annotation reviewed by the first annotator. A2 (`step1_gold/annotated/Reg2Chain_annotation_A2.xlsx`): independent annotation by the legal expert, from scratch, without access to A1 or to the pre-annotation.

Outputs in `step1_gold/gold_precalibration/` (including `agreement.txt`).

In [ ]:
A1 = 'step1_gold/annotated/Reg2Chain_annotation_A1.xlsx'
A2 = 'step1_gold/annotated/Reg2Chain_annotation_A2.xlsx'
if os.path.exists(A2):
    !python scripts/check_and_build_gold.py step1_gold/units.csv {A1} {A2} step1_gold/gold_precalibration
else:
    !python scripts/check_and_build_gold.py step1_gold/units.csv {A1} step1_gold/gold_precalibration

### 1.6 Correction rate of the pre-annotation
Compares the original pre-annotation (`step1_gold/preannotation_claude_A1.xlsx`, locked) with the reviewed A1 workbook.

In [ ]:
!python scripts/compare_with_preannotation.py step1_gold/units.csv step1_gold/preannotation_claude_A1.xlsx step1_gold/annotated/Reg2Chain_annotation_A1.xlsx step1_gold/gold_precalibration

### 1.7 Calibration (resolution of disagreements) and final gold standard
The two annotators jointly resolved the disagreements in `step1_gold/Reg2Chain_calibration.xlsx` (sheet *Rule*: Rule 1 = A2, Rule 2 = A1; sheet *Disagreements*: KEEP/DROP). Rule 1 is applied to every PRODUCT mention of all 63 units. Inter-annotator agreement (cell 1.5) is the value before calibration.

Output: `step1_gold/gold/gold_dev.jsonl`, `step1_gold/gold/gold_test.jsonl`, the reference used in all evaluations.

In [ ]:
%%writefile scripts/apply_calibration.py
"""
Step 1 - Apply the calibration decisions and build the FINAL gold standard.

  python apply_calibration.py units.csv A1.xlsx A2.xlsx calibration.xlsx OUT_DIR

Calibration decisions (sheet 'Rule'):
  Rule 1 = A2 : a PRODUCT mention of the form '<product> referred to in <reference>' is split into
                PRODUCT '<product>' + LEGAL_REF '<reference>' + relation REFERS_TO(product, reference).
                Relations that used the long span now use the product span.
                (Applied to PRODUCT entities only: the legal expert kept label items whole.)
  Rule 2 = A1 : the header of Annex units is not annotated.

Double-annotated units: final = A1 items not marked DROP + A2-only items marked KEEP
(sheet 'Disagreements'). Other units: A1 + automatic application of Rule 1.

Outputs: gold_dev.jsonl, gold_test.jsonl, calibration_report.txt
"""
import csv
import json
import os
import re
import sys
from collections import Counter

from openpyxl import load_workbook

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from check_and_build_gold import load_annotations, norm  # noqa: E402

SEP = " referred to in "


def apply_rule1(text, ents, rels):
    ents, rels = [dict(e) for e in ents], [dict(r) for r in rels]
    new_e, renamed = [], {}
    for e in ents:
        i = e["text"].find(SEP)
        if e["label"] == "PRODUCT" and i > 0:
            head, ref = e["text"][:i], e["text"][i + len(SEP):]
            hs, rs = e["start"], e["start"] + i + len(SEP)
            assert text[hs:hs + len(head)] == head and text[rs:rs + len(ref)] == ref
            new_e.append({"text": head, "label": "PRODUCT", "start": hs, "end": hs + len(head)})
            new_e.append({"text": ref, "label": "LEGAL_REF", "start": rs, "end": rs + len(ref)})
            renamed[norm(e["text"])] = head
            rels.append({"subject": head, "predicate": "REFERS_TO", "object": ref, "qualifier": ""})
        else:
            new_e.append(e)
    for r in rels:
        for k in ("subject", "object"):
            if norm(r[k]) in renamed:
                r[k] = renamed[norm(r[k])]
    return sorted(new_e, key=lambda e: e["start"]), rels


def parse_item(cell):
    """'text  [LABEL]' -> (text, LABEL) ; 'subj —REL→ obj  (qualifier: q)' -> tuple."""
    m = re.match(r"^(.*)  \[([A-Z_]+)\]$", cell)
    if m:
        return ("E", m.group(1), m.group(2))
    m = re.match(r"^(.*) —([A-Z_]+)→ (.*?)(?:  \(qualifier: (.*)\))?$", cell)
    return ("R", m.group(1), m.group(2), m.group(3), m.group(4) or "")


def main(units_csv, a1, a2, calib, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    rows = list(csv.DictReader(open(units_csv, encoding="utf-8")))
    units = {r["unit_id"]: r for r in rows}
    dbl = {u: units[u] for u in units if units[u]["double_annotation"] == "Y"}
    e1, r1, p1 = load_annotations(a1, units)
    e2, r2, p2 = load_annotations(a2, dbl)
    assert not p1 and not p2, (p1, p2)

    wb = load_workbook(calib, data_only=True)
    rules = {i: (row[3] or "").strip().upper()
             for i, row in enumerate(wb["Rule"].iter_rows(min_row=2, values_only=True), start=1) if row[0]}
    assert rules.get(1) == "A2" and rules.get(2) == "A1", rules
    decisions = []
    for row in wb["Disagreements"].iter_rows(min_row=2, values_only=True):
        if row[0]:
            src = "A1" if row[2] else "A2"
            decisions.append((row[0], src, parse_item(row[2] or row[3]), (row[4] or "").strip().upper()))
    undecided = [d for d in decisions if d[3] not in ("KEEP", "DROP")]
    assert not undecided, undecided

    final_e, final_r, report = {}, {}, []
    for u in units:
        if u in dbl:
            ents = [dict(e) for e in e1[u]]
            rels = [dict(r) for r in r1[u]]
            for uid, src, item, dec in decisions:
                if uid != u:
                    continue
                if src == "A1" and dec == "DROP":
                    if item[0] == "E":
                        ents = [e for e in ents if not (e["text"] == item[1] and e["label"] == item[2])]
                    else:
                        rels = [r for r in rels if not (r["subject"] == item[1] and r["predicate"] == item[2]
                                                         and r["object"] == item[3])]
                if src == "A2" and dec == "KEEP":
                    if item[0] == "E":
                        ents += [e for e in e2[u] if e["text"] == item[1] and e["label"] == item[2]][:1]
                    else:
                        rels.append({"subject": item[1], "predicate": item[2], "object": item[3],
                                     "qualifier": item[4]})
            # consistency check: Rule 1 applied to A1 must give the same product/reference entities
            auto_e, _ = apply_rule1(units[u]["text"], e1[u], r1[u])
            if {(e["text"], e["label"]) for e in auto_e if e["label"] in ("PRODUCT", "LEGAL_REF")} - \
               {(e["text"], e["label"]) for e in ents}:
                report.append(f"note {u}: rule-1 output differs from the joint decisions (decisions kept)")
            final_e[u], final_r[u] = sorted(ents, key=lambda e: e["start"]), rels
        else:
            final_e[u], final_r[u] = apply_rule1(units[u]["text"], e1[u], r1[u])

    # validate: no overlap, relations refer to entities
    for u in units:
        names = {norm(e["text"]) for e in final_e[u]}
        for a, b in zip(final_e[u], final_e[u][1:]):
            assert b["start"] >= a["end"], (u, a, b)
        for r in final_r[u]:
            assert norm(r["subject"]) in names and norm(r["object"]) in names, (u, r)

    for split in ("dev", "test"):
        with open(os.path.join(out_dir, f"gold_{split}.jsonl"), "w", encoding="utf-8") as f:
            for r in rows:
                if r["split"] == split:
                    f.write(json.dumps({"unit_id": r["unit_id"], "sentence": r["text"],
                                        "entities": final_e[r["unit_id"]], "relations": final_r[r["unit_id"]]},
                                       ensure_ascii=False) + "\n")
    lines = [f"Rules: {rules}", f"Decisions applied: {len(decisions)}"] + report
    for split in ("dev", "test"):
        ids = [r["unit_id"] for r in rows if r["split"] == split]
        E = Counter(e["label"] for u in ids for e in final_e[u])
        R = Counter(r["predicate"] for u in ids for r in final_r[u])
        lines.append(f"{split}: {len(ids)} units, {sum(E.values())} entities {dict(E.most_common())}, "
                     f"{sum(R.values())} relations {dict(R.most_common())}")
    open(os.path.join(out_dir, "calibration_report.txt"), "w").write("\n".join(lines) + "\n")
    print("\n".join(lines))


if __name__ == "__main__":
    main(*sys.argv[1:6])


In [ ]:
!python scripts/apply_calibration.py step1_gold/units.csv step1_gold/annotated/Reg2Chain_annotation_A1.xlsx step1_gold/annotated/Reg2Chain_annotation_A2.xlsx step1_gold/Reg2Chain_calibration.xlsx step1_gold/gold

## 2. Preprocessing Agent
Cleans and normalises each provision unit and adds linguistic annotations (sentences, tokens, lemmas, POS, dependency labels, stop-word flags). The original text and character offsets are preserved, so that every extraction can be aligned with the gold standard; legal references are never altered and obligation words (*shall, must, may, not, no, only*) are never treated as stop words.

Output: `step2_extraction/preprocessed_units.json`

In [ ]:
# session setup (needed after a reconnection or a change of runtime)
from google.colab import drive
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Reg2Chain_revision')

In [ ]:
!pip install -q spacy
!python -m spacy download en_core_web_sm

In [ ]:
%%writefile scripts/preprocessing_agent.py
"""
Section 2 - Preprocessing Agent.

Cleans and normalises each provision unit and adds linguistic annotations (sentences,
tokens, lemmas, POS, stop-word flags) while keeping the ORIGINAL text and character
offsets, so that every downstream extraction can be aligned with the gold standard.
Legal references (e.g. 'Article 8(1)', 'Annex I') are kept intact: no token is removed
from the text itself; stop-word removal is only a flag.

  python preprocessing_agent.py units.csv preprocessed_units.json
"""
import csv
import json
import re
import sys
import unicodedata

import spacy

MODALS = {"shall", "must", "may"}


def normalise(text):
    t = unicodedata.normalize("NFKC", text)
    return re.sub(r"\s+", " ", t).strip()


def main(units_csv, out_path):
    nlp = spacy.load("en_core_web_sm")
    units = list(csv.DictReader(open(units_csv, encoding="utf-8")))
    out = []
    for u in units:
        doc = nlp(u["text"])
        out.append({
            "unit_id": u["unit_id"], "article": u["article"], "split": u["split"],
            "original": u["text"], "normalised": normalise(u["text"]),
            "sentences": [{"text": s.text, "start": s.start_char, "end": s.end_char} for s in doc.sents],
            "tokens": [{"text": t.text, "lemma": t.lemma_, "pos": t.pos_, "dep": t.dep_,
                        "start": t.idx, "end": t.idx + len(t.text),
                        # obligation / condition words are never treated as stop words
                        "is_stop": bool(t.is_stop and t.lower_ not in MODALS | {"not", "no", "only"})}
                       for t in doc],
        })
    json.dump(out, open(out_path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    n_tok = sum(len(u["tokens"]) for u in out)
    print(f"Preprocessing Agent: {len(out)} units, {n_tok} tokens -> {out_path}")


if __name__ == "__main__":
    main(*sys.argv[1:3])


In [ ]:
!python scripts/preprocessing_agent.py step1_gold/units.csv step2_extraction/preprocessed_units.json

## 3. Entity Extraction Agent and Evaluation 1 (Table 1)
Four systems are compared on the 47 test units of the gold standard:

| System | Description |
|---|---|
| Baseline | spaCy `en_core_web_sm` NER (labels mapped to our scheme) and dependency-based relation extraction (deontic type from the modal verb) |
| Rules | gazetteers built from the development annotations only, domain vocabulary defined outside the test set (the 8 legal oil categories, Annex I parameters), patterns for legal references, values and quoted wordings, relation rules |
| LLM | Qwen2.5-7B-Instruct (4-bit, greedy decoding), few-shot with 6 development units, JSON output aligned to the text |
| Hybrid (Reg2Chain) | rules and LLM, fusion strategy selected on the development units |

Relation extraction is evaluated end-to-end (predicted entities) and with gold entities. We report strict, relaxed and untyped matching, with 95 % bootstrap confidence intervals over units.

All design choices (gazetteers, few-shot examples, fusion strategy) use the development units only; the test units are processed once, with the code frozen. Section 3.4 needs a GPU runtime (about 1 h on a T4).

In [ ]:
# session setup (needed after a reconnection or a change of runtime)
from google.colab import drive
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Reg2Chain_revision')

Mounted at /content/drive


In [ ]:
# A change of runtime (e.g. to a T4 GPU) resets Colab: reconnect Drive and the project folder
from google.colab import drive
import os, torch
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Reg2Chain_revision')
print('Working in', os.getcwd())
!pip install -q -U transformers accelerate bitsandbytes spacy
!python -m spacy download en_core_web_sm
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — switch to a T4 GPU runtime')
os.makedirs('step2_extraction/preds', exist_ok=True); os.makedirs('step2_extraction/logs', exist_ok=True)

### 3.1 Scripts of the Entity Extraction Agent and of the evaluation

In [ ]:
%%writefile scripts/extraction_common.py
"""Shared helpers for the Entity Extraction Agent and its evaluation (Section 3)."""
import json
import re
import unicodedata

ENTITY_TYPES = ["PRODUCT", "PARAMETER", "VALUE", "LABEL_ITEM", "ACTOR", "LEGAL_REF", "PROCESS"]
RELATION_TYPES = ["OBLIGATION", "PERMISSION", "PROHIBITION", "LIMIT", "CONDITION", "REFERS_TO"]


def norm(s):
    s = unicodedata.normalize("NFKC", s or "").lower()
    s = re.sub(r"\s+", " ", s).strip(" .;:,")
    return re.sub(r"^(the|a|an) ", "", s)


def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]


def save_jsonl(rows, path):
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")


def select_non_overlapping(cands):
    """Greedy: higher priority first, then longer span, then earlier start."""
    chosen = []
    for c in sorted(cands, key=lambda e: (-e.get("prio", 0), -(e["end"] - e["start"]), e["start"])):
        if all(c["end"] <= o["start"] or c["start"] >= o["end"] for o in chosen):
            chosen.append(c)
    return sorted(({k: v for k, v in c.items() if k != "prio"} for c in chosen), key=lambda e: e["start"])


def locate(text, span, taken=()):
    """Character offsets of `span` in `text` (exact, then case-insensitive), avoiding taken spans."""
    for hay, needle in ((text, span), (text.lower(), span.lower())):
        start = hay.find(needle)
        while start >= 0:
            end = start + len(needle)
            if all(end <= a or start >= b for a, b in taken):
                return start, end
            start = hay.find(needle, start + 1)
    return None


def dedup_relations(rels):
    seen, out = set(), []
    for r in rels:
        key = (norm(r["subject"]), r["predicate"], norm(r["object"]), norm(r.get("qualifier", "")))
        if key not in seen:
            seen.add(key)
            out.append({"subject": r["subject"], "predicate": r["predicate"], "object": r["object"],
                        "qualifier": r.get("qualifier", "")})
    return out


Overwriting scripts/extraction_common.py


In [ ]:
%%writefile scripts/extract_baseline.py
"""
Section 3 - Baseline: spaCy NER (labels mapped to the annotation scheme) + dependency-based RE.

spaCy's generic labels are mapped to the closest types of our scheme so that the comparison
is fair (the 'untyped' score in the evaluation additionally ignores types altogether).
Relations: for each verb with a subject and an object, the deontic type is derived from the
modal verb (shall/must -> OBLIGATION, may -> PERMISSION, negation -> PROHIBITION).

  python extract_baseline.py gold_test.jsonl pred_baseline.jsonl
"""
import sys

import spacy

import os
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from extraction_common import dedup_relations, load_jsonl, save_jsonl

LABEL_MAP = {"ORG": "ACTOR", "GPE": "ACTOR", "NORP": "ACTOR", "PERSON": "ACTOR",
             "LAW": "LEGAL_REF", "PRODUCT": "PRODUCT",
             "PERCENT": "VALUE", "QUANTITY": "VALUE", "CARDINAL": "VALUE", "MONEY": "VALUE"}


def chunk_of(tok, doc):
    for nc in doc.noun_chunks:
        if nc.start <= tok.i < nc.end:
            return nc
    return doc[tok.i:tok.i + 1]


def deontic(verb):
    kids = list(verb.children)
    neg = any(k.dep_ == "neg" for k in kids) or any(t.lower_ == "no" for t in verb.subtree if t.i < verb.i)
    modal = {k.lower_ for k in kids if k.dep_ == "aux"}
    if neg and modal & {"shall", "must", "may"}:
        return "PROHIBITION"
    if modal & {"shall", "must"}:
        return "OBLIGATION"
    if "may" in modal:
        return "PERMISSION"
    return None


def relations(doc):
    out = []
    for v in doc:
        if v.pos_ not in ("VERB", "AUX"):
            continue
        rel = deontic(v)
        if not rel:
            continue
        subj = [c for c in v.children if c.dep_ in ("nsubj", "nsubjpass")]
        objs = [c for c in v.children if c.dep_ in ("dobj", "attr")]
        objs += [p for c in v.children if c.dep_ == "prep" for p in c.children if p.dep_ == "pobj"]
        for s in subj:
            for o in objs:
                out.append({"subject": chunk_of(s, doc).text, "predicate": rel,
                            "object": chunk_of(o, doc).text, "qualifier": ""})
    return dedup_relations(out)


def main(gold_path, out_path):
    nlp = spacy.load("en_core_web_sm")
    preds = []
    for g in load_jsonl(gold_path):
        doc = nlp(g["sentence"])
        ents = [{"text": e.text, "label": LABEL_MAP[e.label_], "start": e.start_char, "end": e.end_char}
                for e in doc.ents if e.label_ in LABEL_MAP]
        preds.append({"unit_id": g["unit_id"], "entities": ents, "relations": relations(doc)})
    save_jsonl(preds, out_path)
    print(f"baseline: {sum(len(p['entities']) for p in preds)} entities, "
          f"{sum(len(p['relations']) for p in preds)} relations -> {out_path}")


if __name__ == "__main__":
    main(*sys.argv[1:3])


In [ ]:
%%writefile scripts/extract_rules.py
"""
Section 3 - Rule-based component of the Entity Extraction Agent.

Entities: gazetteers built automatically from the DEVELOPMENT annotations only, complemented
by domain vocabulary defined outside the test set (the eight legal oil categories and the
Annex I analytical parameters), plus patterns for legal references, numeric values and quoted
label wordings. Non-overlapping spans are selected by priority, then length.

Relations (also usable on gold entities with --gold-entities):
  REFERS_TO  : entity followed by 'referred to in' / 'within the meaning of' /
               'in accordance with' / 'laid down in' + LEGAL_REF
  LIMIT      : Annex units, PRODUCT + (PARAMETER, VALUE) pairs
  OBLIGATION / PERMISSION / PROHIBITION : modal verb of the clause; subjects = PRODUCT/ACTOR
               entities before the modal, object = first LABEL_ITEM/PROCESS/PRODUCT after it
  CONDITION  : 'only if' / 'only where' / 'below' ... VALUE attached to the LABEL_ITEM

  python extract_rules.py gold_test.jsonl pred_rules.jsonl --dev=gold_dev.jsonl [--gold-entities]
"""
import re
import sys

import os
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from extraction_common import dedup_relations, load_jsonl, save_jsonl, select_non_overlapping

# Domain vocabulary defined OUTSIDE the test set:
#  - the eight legal categories of olive oils (Regulation (EU) No 1308/2013, Annex VII, Part VIII)
#  - the analytical parameters of the quality characteristics (column headings of Annex I, Table A)
DOMAIN = {
    "PRODUCT": ["extra virgin olive oil", "virgin olive oil", "lampante olive oil", "refined olive oil",
                "olive oil composed of refined olive oils and virgin olive oils", "crude olive-pomace oil",
                "refined olive-pomace oil", "olive-pomace oil"],
    "PARAMETER": ["Acidity (%)", "Peroxide value (mEq O2/kg)", "K232", "K268 or K270", "ΔK",
                  "Median of defect (Md)", "Fruity median (Mf)", "Fatty acid ethyl esters (mg/kg)"],
}
GAZ_TYPES = ["PARAMETER", "LABEL_ITEM", "PRODUCT", "ACTOR", "PROCESS"]


def build_gazetteers(dev_path):
    """Gazetteers = domain vocabulary + entity texts annotated in the DEVELOPMENT units only."""
    gaz = {t: set(DOMAIN.get(t, [])) for t in GAZ_TYPES}
    if dev_path:
        for g in load_jsonl(dev_path):
            for e in g["entities"]:
                if e["label"] in gaz and not QUOTE.fullmatch(e["text"]):
                    gaz[e["label"]].add(e["text"])
    return {t: sorted(v, key=len, reverse=True) for t, v in gaz.items()}


PRIO = {"LABEL_ITEM_Q": 9, "LEGAL_REF": 8, "VALUE": 7, "PARAMETER": 6, "LABEL_ITEM": 5,
        "PRODUCT": 4, "ACTOR": 3, "PROCESS": 2}

QUOTE = re.compile(r"‘[^’]{2,}’")
VALUE_RES = [
    re.compile(r"(?:≤|≥|<|>|=)\s?\d+(?:,\d+)?"),
    re.compile(r"\b(?:more than|less than|at least|at most|below|above|exceeding|not exceeding)\s\d+(?:,\d+)?\s?(?:%|° ?C|litres|kg|mg/kg)", re.I),
    re.compile(r"\b\d+(?:,\d+)?\s?(?:%|° ?C|litres)"),
]
REF_START = re.compile(r"\b(?:Articles?|Annex|Part|points?|paragraphs?|Directive|"
                       r"(?:Commission |Council )?(?:Delegated |Implementing )?Regulations?)\b")
REF_TOKEN = re.compile(r"\s*(?:\(\w{1,4}\)|\d+(?:/\d+)*(?:\(\d+\))?|[IVXL]+\b|[a-h]\b|and\b|or\b|to\b|of\b|in\b|,|"
                       r"Annex\b|Part\b|points?\b|paragraphs?\b|Tables?\b|[AB]\b|Articles?\b|No\b|"
                       r"(?:Commission |Council )?(?:Delegated |Implementing )?Regulations?\b|"
                       r"\((?:EU|EEC|EC)\)|this Regulation\b|this Article\b|that paragraph\b|"
                       r"\d{2,}/\d+/EEC\b|EEC\b)")
TRAIL = re.compile(r"(?:\s*(?:,|and|or|to|of|in))+$")


def legal_refs(text):
    out, pos = [], 0
    for m in REF_START.finditer(text):
        if m.start() < pos:
            continue
        end = m.end()
        while True:
            t = REF_TOKEN.match(text, end)
            if not t or t.end() == end:
                break
            end = t.end()
        span = TRAIL.sub("", text[m.start():end])
        if re.search(r"\d|[IVX]{1,4}\b|this Regulation|that paragraph", span) and len(span) > 4:
            out.append({"text": span, "label": "LEGAL_REF", "start": m.start(), "end": m.start() + len(span),
                        "prio": PRIO["LEGAL_REF"]})
            pos = m.start() + len(span)
    return out


def gazetteer(text, terms, label):
    out = []
    for t in terms:
        for m in re.finditer(r"(?<![\w-])" + re.escape(t) + r"(?![\w-])", text, re.I):
            out.append({"text": text[m.start():m.end()], "label": label, "start": m.start(), "end": m.end(),
                        "prio": PRIO[label]})
    return out


def extract_entities(text, gaz):
    c = [{"text": m.group(0), "label": "LABEL_ITEM", "start": m.start(), "end": m.end(),
          "prio": PRIO["LABEL_ITEM_Q"]} for m in QUOTE.finditer(text)]
    c += legal_refs(text)
    for rx in VALUE_RES:
        c += [{"text": m.group(0), "label": "VALUE", "start": m.start(), "end": m.end(),
               "prio": PRIO["VALUE"]} for m in rx.finditer(text)]
    if text.startswith("Annex I, Table A"):            # calibration rule: header not annotated
        hdr = text.index(":")
        c = [e for e in c if e["start"] > hdr or e["label"] == "PRODUCT"]
    for label in GAZ_TYPES:
        c += gazetteer(text, gaz[label], label)
    if text.startswith("Annex I, Table A"):
        hdr = text.index(":")
        c = [e for e in c if not (e["start"] < hdr and e["label"] != "PRODUCT")]
    return select_non_overlapping(c)


# ------------------------------------------------------------------ relations
LINKERS = re.compile(r"^\s*,?\s*(?:as\s+)?(?:referred to in|within the meaning of|in accordance with|laid down in|"
                     r"as defined in|determined in accordance with)\s*$", re.I)
NEG = re.compile(r"\b(?:shall not|may not|must not|no\b[^.;:]*\bshall)", re.I)
MODAL = re.compile(r"\b(shall|must|may)\b", re.I)


def extract_relations(text, ents):
    ents = sorted(ents, key=lambda e: e["start"])
    rels = []
    # REFERS_TO
    for i, e in enumerate(ents):
        if e["label"] == "LEGAL_REF":
            continue
        for f in ents[i + 1:i + 3]:
            if f["label"] == "LEGAL_REF" and LINKERS.match(text[e["end"]:f["start"]]):
                rels.append({"subject": e["text"], "predicate": "REFERS_TO", "object": f["text"]})
                break
    # LIMIT (Annex Table A units)
    if text.startswith("Annex I, Table A"):
        prod = next((e for e in ents if e["label"] == "PRODUCT"), None)
        for a, b in zip(ents, ents[1:]):
            if prod and a["label"] == "PARAMETER" and b["label"] == "VALUE":
                rels.append({"subject": prod["text"], "predicate": "LIMIT", "object": b["text"],
                             "qualifier": a["text"]})
        return dedup_relations(rels)
    # deontic relations, clause by clause
    for clause in re.finditer(r"[^.;:]+", text):
        cs, ce = clause.start(), clause.end()
        m = MODAL.search(text, cs, ce)
        if not m:
            continue
        pred = "PROHIBITION" if NEG.search(text[cs:ce]) else (
            "PERMISSION" if m.group(1).lower() == "may" else "OBLIGATION")
        inside = [e for e in ents if cs <= e["start"] < ce]
        subjects = [e for e in inside if e["start"] < m.start() and e["label"] in ("PRODUCT", "ACTOR")]
        after = [e for e in inside if e["start"] > m.start()]
        obj = next((e for e in after if e["label"] in ("LABEL_ITEM", "PROCESS")), None) or \
            next((e for e in after if e["label"] == "PRODUCT"), None)
        if obj is None and subjects and pred != "PROHIBITION":
            # 'X may bear Y' where Y precedes the modal is rare; nothing to link
            continue
        if obj is None:
            continue
        for s in subjects:
            if s is not obj:
                rels.append({"subject": s["text"], "predicate": pred, "object": obj["text"]})
        # CONDITION: value or process after 'only if/where', 'below', 'more than' in the same clause
        if re.search(r"\bonly (?:if|where|when)\b|\bbelow\b|\bmore than\b", text[cs:ce], re.I):
            target = obj if obj["label"] == "LABEL_ITEM" else None
            for v in after:
                if target and v["label"] == "VALUE":
                    q = next((p["text"] for p in inside if p["label"] == "PARAMETER"), "")
                    rels.append({"subject": target["text"], "predicate": "CONDITION", "object": v["text"],
                                 "qualifier": q})
    return dedup_relations(rels)


def main(gold_path, out_path, *flags):
    use_gold = "--gold-entities" in flags
    dev = next((f.split("=", 1)[1] for f in flags if f.startswith("--dev=")), None)
    gaz = build_gazetteers(dev)
    preds = []
    for g in load_jsonl(gold_path):
        ents = g["entities"] if use_gold else extract_entities(g["sentence"], gaz)
        preds.append({"unit_id": g["unit_id"], "entities": ents,
                      "relations": extract_relations(g["sentence"], ents)})
    save_jsonl(preds, out_path)
    print(f"rules{' (gold entities)' if use_gold else ''}: {sum(len(p['entities']) for p in preds)} entities, "
          f"{sum(len(p['relations']) for p in preds)} relations -> {out_path}")


if __name__ == "__main__":
    main(*sys.argv[1:])


In [ ]:
%%writefile scripts/extract_llm.py
"""
Section 3 - LLM component of the Entity Extraction Agent (open model, runs on a Colab GPU).

Model: Qwen2.5-7B-Instruct, 4-bit (bitsandbytes), greedy decoding (deterministic).
Prompt: the annotation guidelines (types, relations, conventions) + few-shot examples taken
from the DEVELOPMENT units only + the unit to annotate. The model answers in JSON; spans are
aligned to the unit text (exact, then case-insensitive); spans not found in the text are
discarded and logged. Every prompt and raw response is saved (prompt/response log).

  python extract_llm.py gold_test.jsonl gold_dev.jsonl pred_llm.jsonl log.jsonl [--gold-entities]

--gold-entities : the gold entities are given in the prompt and only relations are asked
                  (relation extraction evaluated independently of entity errors).
--pipeline      : production run on all units (used to build the ontology / knowledge graph; not evaluated).
"""
import json
import os
import re
import sys
import time

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from extraction_common import (ENTITY_TYPES, RELATION_TYPES, dedup_relations, load_jsonl, locate,  # noqa: E402
                               norm, save_jsonl, select_non_overlapping)

MODEL_ID = os.environ.get("REG2CHAIN_LLM", "Qwen/Qwen2.5-7B-Instruct")
MAX_NEW_TOKENS = 1200
FEW_SHOT_IDS = ["ART3.1", "ART8.2", "ART7", "ART11.3", "ART10.d", "ANX1.A.8"]   # development units only

GUIDELINES = """You annotate provisions of Regulation (EU) 2022/2104 on olive oil marketing standards.

ENTITY TYPES (copy each entity EXACTLY as it appears in the text; no overlap; longest span with one type;
no leading 'the/a/an'; no final punctuation):
- PRODUCT: oil categories and other products (extra virgin olive oil, oils, blends, foodstuffs...). If a product
  is followed by 'referred to in <reference>', annotate the product alone and the reference as LEGAL_REF.
- PARAMETER: a measurable characteristic (acidity, peroxide value, K232, temperature, maximum capacity...).
- VALUE: a numeric limit or quantity with its operator/unit ('≤ 0,80', 'below 27 °C', 'more than 50 %').
- LABEL_ITEM: something that must/may/must not appear on the label (place of origin, harvesting year, legal name,
  a quoted mandatory wording kept with its quotes...).
- ACTOR: an organisation or role (Member States, Commission, final consumer, mill...).
- LEGAL_REF: a reference to a legal act or provision (Article 6(1), Annex I, Regulation (EU) No 1308/2013...).
- PROCESS: a production, extraction or packaging method or feature (percolation, centrifugation...).
Do not annotate the words 'label' or 'container', nor the header 'Annex I, Table A, quality characteristics of'.

RELATION TYPES (between two annotated entities of the same provision):
- OBLIGATION: subject must have/do/show object ('shall', 'must').
- PERMISSION: subject may have/do/show object, including restricted permissions ('may', 'may only').
- PROHIBITION: subject must not have/show object ('shall not', 'no ... shall').
- LIMIT: a PRODUCT has a limit VALUE for a PARAMETER (put the PARAMETER text in "qualifier").
- CONDITION: a permission/obligation on the subject applies only if the object (VALUE or PROCESS) holds
  (PARAMETER in "qualifier" if any).
- REFERS_TO: an entity is defined by / referred to in / determined in accordance with a LEGAL_REF.
If several products share a rule, write one relation per product.

Answer with JSON only:
{"entities": [{"text": "...", "type": "..."}],
 "relations": [{"subject": "...", "relation": "...", "object": "...", "qualifier": ""}]}"""

GOLD_ENT_INSTR = ("The entities of the provision are given below. Use ONLY these entities as subjects and objects "
                  "and return the relations (copy the entity texts exactly). Return {\"relations\": [...]}.")


def example_answer(g, relations_only=False):
    rels = [{"subject": r["subject"], "relation": r["predicate"], "object": r["object"],
             "qualifier": r.get("qualifier", "")} for r in g["relations"]]
    if relations_only:
        return json.dumps({"relations": rels}, ensure_ascii=False)
    ents = [{"text": e["text"], "type": e["label"]} for e in g["entities"]]
    return json.dumps({"entities": ents, "relations": rels}, ensure_ascii=False)


def user_msg(g, relations_only):
    msg = f"Provision:\n{g['sentence']}"
    if relations_only:
        ents = json.dumps([{"text": e["text"], "type": e["label"]} for e in g["entities"]], ensure_ascii=False)
        msg += f"\n\n{GOLD_ENT_INSTR}\nEntities: {ents}"
    return msg


def build_messages(g, shots, relations_only):
    msgs = [{"role": "system", "content": GUIDELINES}]
    for s in shots:
        msgs.append({"role": "user", "content": user_msg(s, relations_only)})
        msgs.append({"role": "assistant", "content": example_answer(s, relations_only)})
    msgs.append({"role": "user", "content": user_msg(g, relations_only)})
    return msgs


def parse_json(raw):
    raw = re.sub(r"^```(?:json)?|```$", "", raw.strip(), flags=re.M).strip()
    m = re.search(r"\{.*\}", raw, re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None


def to_prediction(g, obj, relations_only):
    text, dropped = g["sentence"], []
    if relations_only:
        ents = g["entities"]
    else:
        cands = []
        for e in (obj or {}).get("entities", []):
            t, ty = str(e.get("text", "")).strip(), str(e.get("type", "")).upper()
            if ty not in ENTITY_TYPES or not t:
                dropped.append(("bad_type", t, ty))
                continue
            loc = locate(text, t, [(c["start"], c["end"]) for c in cands if c["text"] == t])
            if loc is None:
                dropped.append(("not_in_text", t, ty))
                continue
            cands.append({"text": text[loc[0]:loc[1]], "label": ty, "start": loc[0], "end": loc[1]})
        ents = select_non_overlapping(cands)
    names = {norm(e["text"]) for e in ents}
    rels = []
    for r in (obj or {}).get("relations", []):
        rel = str(r.get("relation", r.get("predicate", ""))).upper()
        s, o = str(r.get("subject", "")), str(r.get("object", ""))
        if rel in RELATION_TYPES and norm(s) in names and norm(o) in names:
            rels.append({"subject": s, "predicate": rel, "object": o, "qualifier": str(r.get("qualifier", "") or "")})
        else:
            dropped.append(("bad_relation", s, rel, o))
    return {"unit_id": g["unit_id"], "entities": ents, "relations": dedup_relations(rels)}, dropped


def load_model():
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, device_map="auto",
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                                               bnb_4bit_quant_type="nf4"))
    model.eval()
    return tok, model


def generate(tok, model, messages):
    import torch
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, temperature=None,
                             top_p=None, top_k=None, pad_token_id=tok.eos_token_id)
    return prompt, tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


def main(gold_path, dev_path, out_path, log_path, *flags):
    relations_only = "--gold-entities" in flags
    dev = {g["unit_id"]: g for g in load_jsonl(dev_path)}
    shots = [dev[i] for i in FEW_SHOT_IDS]
    gold = load_jsonl(gold_path)
    if "--pipeline" not in flags:   # evaluation runs: the few-shot units must never be evaluated
        assert not set(FEW_SHOT_IDS) & {g["unit_id"] for g in gold}, "few-shot examples must not be evaluated"
    preds = load_jsonl(out_path) if os.path.exists(out_path) else []      # resume after a disconnection
    logs = load_jsonl(log_path) if os.path.exists(log_path) else []
    done = {p["unit_id"] for p in preds}
    if done:
        print(f"Resuming: {len(done)} units already done")
    if len(done) == len(gold):
        print(f"all units already done -> {out_path}")
        return
    tok, model = load_model()
    t0 = time.time()
    for k, g in enumerate(gold, 1):
        if g["unit_id"] in done:
            continue
        prompt, raw = generate(tok, model, build_messages(g, shots, relations_only))
        obj = parse_json(raw)
        pred, dropped = to_prediction(g, obj, relations_only)
        preds.append(pred)
        logs.append({"unit_id": g["unit_id"], "model": MODEL_ID, "decoding": "greedy",
                     "max_new_tokens": MAX_NEW_TOKENS, "relations_only": relations_only,
                     "prompt": prompt, "response": raw, "parsed": obj is not None, "dropped": dropped})
        print(f"[{k}/{len(gold)}] {g['unit_id']}: {len(pred['entities'])} ent, {len(pred['relations'])} rel, "
              f"{len(dropped)} dropped, {time.time() - t0:.0f}s", flush=True)
        save_jsonl(preds, out_path)          # saved after every unit (safe if the session stops)
        save_jsonl(logs, log_path)
    print(f"LLM ({MODEL_ID}) -> {out_path}; log -> {log_path}")


if __name__ == "__main__":
    main(*sys.argv[1:])


In [ ]:
%%writefile scripts/combine_hybrid.py
"""
Section 3 - Hybrid Entity Extraction Agent (Reg2Chain) = rules + LLM.

Entities: the precise pattern-based types (LEGAL_REF, VALUE, quoted LABEL_ITEM) and the
domain-gazetteer PARAMETERs come from the rules; every other type comes from the LLM;
rule entities are then added wherever they do not overlap an LLM entity.
Relations: union of the LLM relations and of the rule relations recomputed on the merged
entities (deduplicated); relations whose arguments are not merged entities are discarded.

  python combine_hybrid.py gold.jsonl pred_rules.jsonl pred_llm.jsonl pred_hybrid.jsonl --strategy=patterns-first|llm-first

The strategy is selected on the development units (never on the test units).
"""
import os
import re
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from extraction_common import dedup_relations, load_jsonl, norm, save_jsonl, select_non_overlapping  # noqa: E402
from extract_rules import extract_relations  # noqa: E402

RULE_FIRST = {"LEGAL_REF", "VALUE", "PARAMETER"}
QUOTED = re.compile(r"^‘.*’$")


def merge(rule_ents, llm_ents, strategy):
    """patterns-first: rule LEGAL_REF/VALUE/PARAMETER/quoted spans win over the LLM, the LLM wins elsewhere.
    llm-first: LLM spans win everywhere; rule spans only fill the gaps."""
    cands = []
    for e in rule_ents:
        strong = e["label"] in RULE_FIRST or QUOTED.match(e["text"])
        prio = 3 if (strategy == "patterns-first" and strong) else 1
        cands.append(dict(e, prio=prio))
    for e in llm_ents:
        cands.append(dict(e, prio=2))
    return select_non_overlapping(cands)


def main(gold_path, rules_path, llm_path, out_path, *flags):
    strategy = next((f.split("=", 1)[1] for f in flags if f.startswith("--strategy=")), "patterns-first")
    assert strategy in ("patterns-first", "llm-first"), strategy
    rules = {p["unit_id"]: p for p in load_jsonl(rules_path)}
    llm = {p["unit_id"]: p for p in load_jsonl(llm_path)}
    out = []
    for g in load_jsonl(gold_path):
        u = g["unit_id"]
        r, l = rules.get(u, {"entities": [], "relations": []}), llm.get(u, {"entities": [], "relations": []})
        ents = merge(r["entities"], l["entities"], strategy)
        names = {norm(e["text"]) for e in ents}
        rels = [x for x in l["relations"] + extract_relations(g["sentence"], ents)
                if norm(x["subject"]) in names and norm(x["object"]) in names]
        out.append({"unit_id": u, "entities": ents, "relations": dedup_relations(rels)})
    save_jsonl(out, out_path)
    print(f"hybrid ({strategy}): {sum(len(p['entities']) for p in out)} entities, "
          f"{sum(len(p['relations']) for p in out)} relations -> {out_path}")


if __name__ == "__main__":
    main(*sys.argv[1:])


In [ ]:
%%writefile scripts/evaluate_extraction.py
"""
Section 3 - Evaluation of entity and relation extraction against the gold standard (new Table 1).

Entities (micro-averaged over units, one-to-one matching):
  strict   : same start/end offsets and same type
  relaxed  : overlapping spans and same type
  untyped  : overlapping spans, type ignored (fair to systems with another label set)
Relations:
  strict   : same subject, relation type and object (normalised text); LIMIT also needs the qualifier
  relaxed  : same relation type, subject and object overlap (>= 50 % of tokens or containment)
95 % confidence intervals for F1 by bootstrap over units (1,000 resamples, fixed seed).

  python evaluate_extraction.py gold_test.jsonl OUT_DIR name=pred.jsonl [name=pred.jsonl ...]
"""
import csv
import os
import random
import re
import sys
from collections import Counter

import os
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from extraction_common import load_jsonl, norm

TOK = re.compile(r"\w+|[^\w\s]")


def toks(s):
    return set(TOK.findall(norm(s)))


def text_overlap(a, b):
    na, nb = norm(a), norm(b)
    if not na or not nb:
        return False
    if na == nb or na in nb or nb in na:
        return True
    ta, tb = toks(a), toks(b)
    return len(ta & tb) / max(1, min(len(ta), len(tb))) >= 0.5


def span_overlap(a, b):
    return a["start"] < b["end"] and b["start"] < a["end"]


ENT = {
    "strict": lambda g, p: g["start"] == p["start"] and g["end"] == p["end"] and g["label"] == p["label"],
    "relaxed": lambda g, p: span_overlap(g, p) and g["label"] == p["label"],
    "untyped": lambda g, p: span_overlap(g, p),
}
REL = {
    "strict": lambda g, p: (norm(g["subject"]), g["predicate"], norm(g["object"])) ==
                           (norm(p["subject"]), p["predicate"], norm(p["object"])) and
                           (g["predicate"] != "LIMIT" or norm(g.get("qualifier")) == norm(p.get("qualifier"))),
    "relaxed": lambda g, p: g["predicate"] == p["predicate"] and text_overlap(g["subject"], p["subject"])
                            and text_overlap(g["object"], p["object"]),
}


def match(gold, pred, same):
    used, tp = [False] * len(pred), 0
    for g in gold:
        for j, p in enumerate(pred):
            if not used[j] and same(g, p):
                used[j] = True
                tp += 1
                break
    return tp


def per_unit(gold, preds, key, same):
    out = []
    for g in gold:
        p = preds.get(g["unit_id"], {}).get(key, [])
        out.append((match(g[key], p, same), len(g[key]), len(p)))
    return out


def prf(counts):
    tp = sum(c[0] for c in counts); ng = sum(c[1] for c in counts); np_ = sum(c[2] for c in counts)
    p = tp / np_ if np_ else 0.0
    r = tp / ng if ng else 0.0
    return tp, np_ - tp, ng - tp, p, r, (2 * p * r / (p + r) if p + r else 0.0)


def bootstrap_ci(counts, n=1000, seed=2104):
    rnd = random.Random(seed)
    f1s = sorted(prf([counts[rnd.randrange(len(counts))] for _ in counts])[5] for _ in range(n))
    return f1s[int(0.025 * n)], f1s[int(0.975 * n) - 1]


def main(gold_path, out_dir, *specs):
    os.makedirs(out_dir, exist_ok=True)
    gold = load_jsonl(gold_path)
    print(f"Gold: {len(gold)} units, {sum(len(g['entities']) for g in gold)} entities, "
          f"{sum(len(g['relations']) for g in gold)} relations")
    rows = []
    for spec in specs:
        name, path = spec.split("=", 1)
        preds = {p["unit_id"]: p for p in load_jsonl(path)}
        missing = [g["unit_id"] for g in gold if g["unit_id"] not in preds]
        if missing:
            print(f"{name}: {len(missing)} units without prediction (counted as FN)")
        for task, key, settings in (("NER", "entities", ENT), ("RE", "relations", REL)):
            if task == "NER" and name.endswith("+gold-ents"):
                continue
            for s, fn in settings.items():
                c = per_unit(gold, preds, key, fn)
                tp, fp, fn_, p, r, f = prf(c)
                lo, hi = bootstrap_ci(c)
                rows.append([name, task, s, tp, fp, fn_, round(p, 3), round(r, 3), round(f, 3),
                             f"[{lo:.2f}, {hi:.2f}]"])
    hdr = ["system", "task", "setting", "TP", "FP", "FN", "P", "R", "F1", "F1 95% CI"]
    with open(os.path.join(out_dir, "table1_extraction.csv"), "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerows([hdr] + rows)
    print(f"\n{'system':<22}{'task':<5}{'setting':<9}{'TP':>5}{'FP':>5}{'FN':>5}{'P':>7}{'R':>7}{'F1':>7}  CI")
    for r in rows:
        print(f"{r[0]:<22}{r[1]:<5}{r[2]:<9}{r[3]:>5}{r[4]:>5}{r[5]:>5}{r[6]:>7.3f}{r[7]:>7.3f}{r[8]:>7.3f}  {r[9]}")
    with open(os.path.join(out_dir, "table1_extraction.tex"), "w", encoding="utf-8") as f:
        f.write("% system & task & setting & TP & FP & FN & P & R & F1 & 95% CI\n")
        for r in rows:
            f.write(" & ".join(str(x) for x in r).replace("_", "\\_") + " \\\\\n")

    # per-type F1 (strict entities) for each system
    types = sorted({e["label"] for g in gold for e in g["entities"]})
    per_type = []
    for spec in specs:
        name, path = spec.split("=", 1)
        if name.endswith("+gold-ents"):
            continue
        preds = {p["unit_id"]: p for p in load_jsonl(path)}
        row = [name]
        for t in types:
            c = [(match([e for e in g["entities"] if e["label"] == t],
                        [e for e in preds.get(g["unit_id"], {}).get("entities", []) if e["label"] == t],
                        ENT["relaxed"]),
                  sum(e["label"] == t for e in g["entities"]),
                  sum(e["label"] == t for e in preds.get(g["unit_id"], {}).get("entities", [])))
                 for g in gold]
            row.append(round(prf(c)[5], 3))
        per_type.append(row)
    with open(os.path.join(out_dir, "ner_f1_per_type_relaxed.csv"), "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerows([["system"] + types] + per_type)
    print("\nRelaxed NER F1 per type:")
    print(f"{'system':<22}" + "".join(f"{t[:9]:>11}" for t in types))
    for row in per_type:
        print(f"{row[0]:<22}" + "".join(f"{v:>11.3f}" for v in row[1:]))


if __name__ == "__main__":
    main(*sys.argv[1:])


In [ ]:
%%writefile scripts/select_hybrid_strategy.py
"""
Section 3 - Select the hybrid fusion strategy on the DEVELOPMENT units (never on the test units).

Score = mean of relaxed NER F1 and relaxed RE F1 on the dev units that are not few-shot examples.

  python select_hybrid_strategy.py gold_dev_eval.jsonl pred_rules_dev.jsonl pred_llm_dev.jsonl OUT_DIR
"""
import os
import subprocess
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from evaluate_extraction import ENT, REL, per_unit, prf  # noqa: E402
from extraction_common import load_jsonl  # noqa: E402

HERE = os.path.dirname(os.path.abspath(__file__))


def main(gold_path, rules_path, llm_path, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    gold, scores = load_jsonl(gold_path), {}
    for st in ("patterns-first", "llm-first"):
        out = os.path.join(out_dir, f"pred_hybrid_dev_{st}.jsonl")
        subprocess.run([sys.executable, os.path.join(HERE, "combine_hybrid.py"), gold_path, rules_path, llm_path,
                        out, f"--strategy={st}"], check=True)
        preds = {p["unit_id"]: p for p in load_jsonl(out)}
        ner = prf(per_unit(gold, preds, "entities", ENT["relaxed"]))[5]
        re_ = prf(per_unit(gold, preds, "relations", REL["relaxed"]))[5]
        scores[st] = (ner + re_) / 2
        print(f"dev {st}: NER relaxed F1 = {ner:.3f}, RE relaxed F1 = {re_:.3f}, score = {scores[st]:.3f}")
    best = max(scores, key=scores.get)
    open(os.path.join(out_dir, "hybrid_strategy.txt"), "w").write(best + "\n")
    print(f"selected strategy (on dev): {best}")


if __name__ == "__main__":
    main(*sys.argv[1:5])


### 3.2 Development check (development units only, no test data)
The 10 development units that are not few-shot examples are used to check the LLM output and to select the hybrid fusion strategy. About 10 min on a T4.

In [ ]:
import json
FEW_SHOT = ["ART3.1", "ART8.2", "ART7", "ART11.3", "ART10.d", "ANX1.A.8"]
with open('step1_gold/gold/gold_dev.jsonl') as f, open('step2_extraction/gold_dev_eval.jsonl', 'w') as g:
    for line in f:
        if json.loads(line)['unit_id'] not in FEW_SHOT:
            g.write(line)
G = 'step1_gold/gold'; P = 'step2_extraction/preds'; L = 'step2_extraction/logs'
!python scripts/extract_rules.py step2_extraction/gold_dev_eval.jsonl {P}/rules_dev.jsonl --dev={G}/gold_dev.jsonl
!python scripts/extract_llm.py step2_extraction/gold_dev_eval.jsonl {G}/gold_dev.jsonl {P}/llm_dev.jsonl {L}/llm_dev_log.jsonl
!python scripts/select_hybrid_strategy.py step2_extraction/gold_dev_eval.jsonl {P}/rules_dev.jsonl {P}/llm_dev.jsonl step2_extraction

### 3.3 Baseline and rules on the test units

In [ ]:
!python scripts/extract_baseline.py {G}/gold_test.jsonl {P}/baseline_test.jsonl
!python scripts/extract_rules.py {G}/gold_test.jsonl {P}/rules_test.jsonl --dev={G}/gold_dev.jsonl
!python scripts/extract_rules.py {G}/gold_test.jsonl {P}/rules_test_gold_ents.jsonl --gold-entities

### 3.4 LLM on the test units (GPU)
Two runs: entities and relations (about 30 min), then relations with the gold entities (about 25 min). Predictions and the full prompt/response log are saved after every unit: if Colab disconnects, run the cell again and it resumes where it stopped.

In [ ]:
!python scripts/extract_llm.py {G}/gold_test.jsonl {G}/gold_dev.jsonl {P}/llm_test.jsonl {L}/llm_test_log.jsonl

In [ ]:
!python scripts/extract_llm.py {G}/gold_test.jsonl {G}/gold_dev.jsonl {P}/llm_test_gold_ents.jsonl {L}/llm_test_gold_ents_log.jsonl --gold-entities

### 3.5 Hybrid Entity Extraction Agent (Reg2Chain) on the test units
Uses the fusion strategy selected on the development units in 3.2.

In [ ]:
strategy = open('step2_extraction/hybrid_strategy.txt').read().strip()
print('strategy selected on dev:', strategy)
!python scripts/combine_hybrid.py {G}/gold_test.jsonl {P}/rules_test.jsonl {P}/llm_test.jsonl {P}/hybrid_test.jsonl --strategy={strategy}

### 3.6 Evaluation 1 (Table 1)
Outputs in `step2_extraction/results/`: `table1_extraction.csv`, `table1_extraction.tex` (LaTeX rows), `ner_f1_per_type_relaxed.csv`.

In [ ]:
!python scripts/evaluate_extraction.py {G}/gold_test.jsonl step2_extraction/results \
    baseline={P}/baseline_test.jsonl rules={P}/rules_test.jsonl llm={P}/llm_test.jsonl hybrid={P}/hybrid_test.jsonl \
    rules+gold-ents={P}/rules_test_gold_ents.jsonl llm+gold-ents={P}/llm_test_gold_ents.jsonl

### 3.7 Statistical significance and LLM error analysis
Paired bootstrap over the test units (10,000 resamples): F1 differences between systems with 95 % CI and p-values. Analysis of the LLM logs: hallucinated spans (not found in the text), invalid outputs.
Outputs: `step2_extraction/results/significance_paired_bootstrap.csv`, `llm_error_analysis.txt`.

In [ ]:
%%writefile scripts/significance_and_errors.py
"""
Section 3 - Statistical significance (paired bootstrap) and LLM error analysis.

1. Paired bootstrap over test units (10,000 resamples, fixed seed): for each comparison
   (hybrid vs rules, hybrid vs llm, hybrid vs baseline) and each setting, the F1 difference,
   its 95 % CI and a two-sided p-value (share of resamples where the difference changes sign).
2. LLM log analysis: entities/relations proposed by the model, kept, and discarded
   (span not found in the text = hallucinated span, invalid type, invalid relation),
   and the number of responses that were not valid JSON.

  python significance_and_errors.py gold_test.jsonl OUT_DIR preds_dir logs_dir
"""
import csv
import json
import os
import random
import sys
from collections import Counter

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from evaluate_extraction import ENT, REL, per_unit, prf  # noqa: E402
from extraction_common import load_jsonl  # noqa: E402

N_BOOT, SEED = 10000, 2104
SETTINGS = [("NER", "entities", "strict", ENT["strict"]), ("NER", "entities", "relaxed", ENT["relaxed"]),
            ("RE", "relations", "strict", REL["strict"]), ("RE", "relations", "relaxed", REL["relaxed"])]
COMPARISONS = [("hybrid", "rules"), ("hybrid", "llm"), ("hybrid", "baseline"), ("llm", "rules")]


def paired_bootstrap(ca, cb):
    rnd, n = random.Random(SEED), len(ca)
    obs = prf(ca)[5] - prf(cb)[5]
    diffs = []
    for _ in range(N_BOOT):
        idx = [rnd.randrange(n) for _ in range(n)]
        diffs.append(prf([ca[i] for i in idx])[5] - prf([cb[i] for i in idx])[5])
    diffs.sort()
    lo, hi = diffs[int(0.025 * N_BOOT)], diffs[int(0.975 * N_BOOT) - 1]
    p = 2 * min(sum(d <= 0 for d in diffs), sum(d >= 0 for d in diffs)) / N_BOOT
    return obs, lo, hi, min(1.0, p)


def main(gold_path, out_dir, preds_dir, logs_dir):
    os.makedirs(out_dir, exist_ok=True)
    gold = load_jsonl(gold_path)
    files = {"baseline": "baseline_test.jsonl", "rules": "rules_test.jsonl", "llm": "llm_test.jsonl",
             "hybrid": "hybrid_test.jsonl"}
    preds = {k: {p["unit_id"]: p for p in load_jsonl(os.path.join(preds_dir, v))} for k, v in files.items()}
    rows = []
    print(f"Paired bootstrap ({N_BOOT} resamples over {len(gold)} test units)")
    print(f"{'comparison':<22}{'task':<5}{'setting':<9}{'ΔF1':>8}{'95% CI':>18}{'p':>8}")
    for a, b in COMPARISONS:
        for task, key, setting, fn in SETTINGS:
            d, lo, hi, p = paired_bootstrap(per_unit(gold, preds[a], key, fn), per_unit(gold, preds[b], key, fn))
            rows.append([f"{a} vs {b}", task, setting, round(d, 3), round(lo, 3), round(hi, 3), round(p, 4)])
            print(f"{a + ' vs ' + b:<22}{task:<5}{setting:<9}{d:>8.3f}   [{lo:6.3f}, {hi:6.3f}]{p:>8.4f}")
    with open(os.path.join(out_dir, "significance_paired_bootstrap.csv"), "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerows([["comparison", "task", "setting", "delta_F1", "CI_low", "CI_high", "p_value"]] + rows)

    print("\nLLM output analysis (prompt/response logs)")
    report = []
    for name in ("llm_test_log.jsonl", "llm_test_gold_ents_log.jsonl"):
        path = os.path.join(logs_dir, name)
        if not os.path.exists(path):
            continue
        logs = load_jsonl(path)
        reasons = Counter(d[0] for l in logs for d in l["dropped"])
        proposed_e = kept_e = 0
        for l in logs:
            try:
                obj = json.loads(l["response"][l["response"].index("{"):l["response"].rindex("}") + 1])
                proposed_e += len(obj.get("entities", []))
            except (ValueError, json.JSONDecodeError):
                pass
        if "gold_ents" not in name:
            p = {x["unit_id"]: x for x in load_jsonl(os.path.join(preds_dir, "llm_test.jsonl"))}
            kept_e = sum(len(x["entities"]) for x in p.values())
        line = (f"{name}: {len(logs)} responses, {sum(not l['parsed'] for l in logs)} not valid JSON; "
                f"entities proposed {proposed_e}, kept {kept_e}, span not found in text (hallucinated) "
                f"{reasons.get('not_in_text', 0)}, invalid type {reasons.get('bad_type', 0)}; "
                f"relations discarded (argument not an entity / invalid type) {reasons.get('bad_relation', 0)}")
        report.append(line)
        print(line)
        if reasons.get("not_in_text"):
            ex = [d[1] for l in logs for d in l["dropped"] if d[0] == "not_in_text"][:8]
            print("   examples of discarded spans:", ex)
    open(os.path.join(out_dir, "llm_error_analysis.txt"), "w", encoding="utf-8").write("\n".join(report) + "\n")


if __name__ == "__main__":
    main(*sys.argv[1:5])


In [ ]:
G = 'step1_gold/gold'; P = 'step2_extraction/preds'
!python scripts/significance_and_errors.py {G}/gold_test.jsonl step2_extraction/results {P} step2_extraction/logs

### 3.8 Relation extraction of the hybrid agent with the gold entities

Why this cell: Table 1 evaluates relation extraction end to end, so a relation error can come either from a wrong entity or from the relation step itself. Section 3.6 already scores the rules and the LLM with the gold entities (0.678 and 0.679 strict F1), but not the hybrid agent. This cell fills that gap. With the gold entities, the fusion of the hybrid agent reduces to the union of the LLM relations and the rule relations (deduplicated, arguments restricted to the entities), so the score shows whether the end-to-end gain of the hybrid in relation extraction comes from better entities or from combining the two relation sets.

It reuses the predictions of 3.3 and 3.4 (`*_test_gold_ents.jsonl`); no LLM call, about 10 s.
Output: `step2_extraction/results/re_gold_entities.csv`.

In [ ]:
# Same fusion rule as the hybrid agent (union of LLM relations and rule relations, deduplicated,
# arguments restricted to the entities), applied to the gold entities. Then scores and paired
# bootstrap against each component, with the same functions as 3.6 and 3.7.
# The cell sets its own working folder, so it can be run alone in a new session.
import os, sys, csv
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Reg2Chain_revision')
sys.path.insert(0, os.path.abspath('scripts'))

from extraction_common import load_jsonl, save_jsonl, norm, dedup_relations
from evaluate_extraction import REL, per_unit, prf, bootstrap_ci
from significance_and_errors import paired_bootstrap

G = 'step1_gold/gold'; P = 'step2_extraction/preds'; R = 'step2_extraction/results'
gold = load_jsonl(f'{G}/gold_test.jsonl')
rules = {p['unit_id']: p for p in load_jsonl(f'{P}/rules_test_gold_ents.jsonl')}
llm = {p['unit_id']: p for p in load_jsonl(f'{P}/llm_test_gold_ents.jsonl')}

hyb = []
for g in gold:
    u = g['unit_id']
    names = {norm(e['text']) for e in g['entities']}
    rels = [x for x in llm.get(u, {}).get('relations', []) + rules.get(u, {}).get('relations', [])
            if norm(x['subject']) in names and norm(x['object']) in names]
    hyb.append({'unit_id': u, 'entities': g['entities'], 'relations': dedup_relations(rels)})
save_jsonl(hyb, f'{P}/hybrid_test_gold_ents.jsonl')

systems = {'rules': rules, 'llm': llm, 'hybrid': {p['unit_id']: p for p in hyb}}
rows = []
print(f"{'system (gold entities)':<24}{'setting':<9}{'TP':>4}{'FP':>4}{'FN':>4}{'P':>7}{'R':>7}{'F1':>7}  95% CI")
for name, preds in systems.items():
    for s in ('strict', 'relaxed'):
        c = per_unit(gold, preds, 'relations', REL[s])
        tp, fp, fn, p, r, f = prf(c); lo, hi = bootstrap_ci(c)
        rows.append([name, s, tp, fp, fn, round(p, 3), round(r, 3), round(f, 3), f'[{lo:.2f}, {hi:.2f}]'])
        print(f"{name:<24}{s:<9}{tp:>4}{fp:>4}{fn:>4}{p:>7.3f}{r:>7.3f}{f:>7.3f}  [{lo:.2f}, {hi:.2f}]")

print('\nPaired bootstrap, gold entities')
for a, b in (('hybrid', 'rules'), ('hybrid', 'llm'), ('llm', 'rules')):
    for s in ('strict', 'relaxed'):
        d, lo, hi, p = paired_bootstrap(per_unit(gold, systems[a], 'relations', REL[s]),
                                        per_unit(gold, systems[b], 'relations', REL[s]))
        rows.append([f'{a} vs {b}', s, '', '', '', '', '', round(d, 3), f'[{lo:.3f}, {hi:.3f}] p={p:.4f}'])
        print(f"{a + ' vs ' + b:<18}{s:<9} dF1={d:+.3f}  [{lo:.3f}, {hi:.3f}]  p={p:.4f}")

with open(f'{R}/re_gold_entities.csv', 'w', newline='', encoding='utf-8') as f:
    csv.writer(f).writerows([['system', 'setting', 'TP', 'FP', 'FN', 'P', 'R', 'F1', 'CI / test']] + rows)
print(f'\nSaved {R}/re_gold_entities.csv')

system (gold entities)  setting    TP  FP  FN      P      R     F1  95% CI
rules                   strict     59   7  49  0.894  0.546  0.678  [0.53, 0.79]
rules                   relaxed    60   6  48  0.909  0.556  0.690  [0.55, 0.80]
llm                     strict     72  32  36  0.692  0.667  0.679  [0.55, 0.79]
llm                     relaxed    72  32  36  0.692  0.667  0.679  [0.55, 0.79]
hybrid                  strict     79  38  29  0.675  0.731  0.702  [0.59, 0.80]
hybrid                  relaxed    79  38  29  0.675  0.731  0.702  [0.59, 0.80]

Paired bootstrap, gold entities
hybrid vs rules   strict    dF1=+0.024  [-0.048, 0.114]  p=0.5516
hybrid vs rules   relaxed   dF1=+0.013  [-0.061, 0.104]  p=0.7662
hybrid vs llm     strict    dF1=+0.023  [-0.015, 0.065]  p=0.2438
hybrid vs llm     relaxed   dF1=+0.023  [-0.015, 0.065]  p=0.2438
llm vs rules      strict    dF1=+0.001  [-0.092, 0.104]  p=0.9702
llm vs rules      relaxed   dF1=-0.010  [-0.104, 0.096]  p=0.8470

Saved ste

## 4. Ontology Agent and expert refinement
The Ontology Agent follows the Text2Onto approach (term extraction, concept formation, taxonomy by modifier stripping, instance and non-taxonomic relation learning), re-implemented in Python on the output of the Entity Extraction Agent for all 63 provision units. Each normative relation becomes a `ComplianceRule` individual linked to its source provision. The learned ontology is exported to OWL (Turtle) and to a review workbook.

The legal expert and the ontology researcher then refine it (KEEP / EDIT / MERGE / DELETE / ADD); the number and type of corrections and the time spent are recorded.

Section 4.1 needs a GPU for a few minutes (the 6 few-shot units; the 57 other units reuse the predictions of Section 3).

In [ ]:
# session setup (needed after a reconnection or a change of runtime)
from google.colab import drive
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Reg2Chain_revision')

In [ ]:
from google.colab import drive
import os, json, csv, shutil
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Reg2Chain_revision')
!pip install -q -U rdflib transformers accelerate bitsandbytes spacy
!python -m spacy download en_core_web_sm
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — switch to a T4 GPU runtime')
for d in ['step3_ontology', 'step3_ontology/learned', 'step3_ontology/review', 'step3_ontology/refined']:
    os.makedirs(d, exist_ok=True)
G = 'step1_gold/gold'; P = 'step2_extraction/preds'; L = 'step2_extraction/logs'; O = 'step3_ontology'

### 4.1 Entity Extraction Agent on all 63 units (pipeline run)
Rules + LLM + hybrid fusion (strategy selected on dev in 3.2). The LLM predictions already computed in Section 3 are reused; only the 6 few-shot units are processed now. This run builds the ontology; it is not an evaluation.

In [ ]:
# all units, in the input format of the extraction scripts
with open(f'{O}/all_units.jsonl', 'w', encoding='utf-8') as f:
    for u in csv.DictReader(open('step1_gold/units.csv', encoding='utf-8')):
        f.write(json.dumps({'unit_id': u['unit_id'], 'sentence': u['text'], 'entities': [], 'relations': []}, ensure_ascii=False) + '\n')
# reuse the LLM predictions of Section 3 (dev check + test run)
if not os.path.exists(f'{O}/llm_all.jsonl'):
    with open(f'{O}/llm_all.jsonl', 'w') as out, open(f'{O}/llm_all_log.jsonl', 'w') as log:
        for a, b in [(f'{P}/llm_dev.jsonl', f'{L}/llm_dev_log.jsonl'), (f'{P}/llm_test.jsonl', f'{L}/llm_test_log.jsonl')]:
            out.write(open(a).read()); log.write(open(b).read())
!python scripts/extract_llm.py {O}/all_units.jsonl {G}/gold_dev.jsonl {O}/llm_all.jsonl {O}/llm_all_log.jsonl --pipeline
!python scripts/extract_rules.py {O}/all_units.jsonl {O}/rules_all.jsonl --dev={G}/gold_dev.jsonl
strategy = open('step2_extraction/hybrid_strategy.txt').read().strip()
!python scripts/combine_hybrid.py {O}/all_units.jsonl {O}/rules_all.jsonl {O}/llm_all.jsonl {O}/extraction_all.jsonl --strategy={strategy}

### 4.2 Ontology learning (Text2Onto approach), OWL export and review workbook

In [ ]:
%%writefile scripts/ontology_agent.py
"""
Section 4 - Ontology Agent.

Follows the Text2Onto approach (Cimiano & Völker, 2005), re-implemented in Python on top of the
outputs of the Entity Extraction Agent on all provision units:
  1. Term extraction   : entity mentions -> terms (lemmatised, lower-cased), with frequency and
                         TF-IDF-style relevance (number of units mentioning the term).
  2. Concept formation : terms with the same lemma are merged into one concept; each concept is
                         placed under the top class of its entity type.
  3. Taxonomy          : 'vertical relations' by modifier stripping (e.g. 'refined olive oil'
                         -> 'olive oil'): the parent is the longest concept that is a head-suffix.
  4. Instances         : legal references become individuals of LegalReference; each normative
                         relation becomes a ComplianceRule individual linked to its subject, object,
                         threshold value / parameter and source provision (traceability).
  5. Non-taxonomic relations : one object property per relation type, domain/range from arguments.
The ontology is exported to OWL (Turtle) and to a review workbook for expert refinement.

  python ontology_agent.py extraction_all.jsonl units.csv OUT_DIR
"""
import csv
import json
import math
import os
import re
import sys
from collections import Counter, defaultdict

import spacy
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from extraction_common import load_jsonl, norm  # noqa: E402

TOP = {"PRODUCT": "Product", "PARAMETER": "QualityParameter", "LABEL_ITEM": "LabelItem", "ACTOR": "Actor",
       "PROCESS": "Process", "LEGAL_REF": "LegalReference"}
PROPS = {"OBLIGATION": "hasObligation", "PERMISSION": "hasPermission", "PROHIBITION": "hasProhibition",
         "LIMIT": "hasLimit", "CONDITION": "hasCondition", "REFERS_TO": "refersTo"}
NORMATIVE = {"OBLIGATION", "PERMISSION", "PROHIBITION", "LIMIT", "CONDITION"}
NS = "http://example.org/reg2chain/olive-oil#"

_NLP = None


def lemma_key(text):
    global _NLP
    if _NLP is None:                       # loaded only when terms are lemmatised (not needed for 4.4)
        _NLP = spacy.load("en_core_web_sm", disable=["ner", "parser"])
    nlp = _NLP
    return " ".join(t.lemma_.lower() for t in nlp(norm(text)) if not t.is_punct or t.text in "()%/")


def camel(label):
    words = re.findall(r"[A-Za-z0-9ΔΣ]+", label)
    return "".join(w[:1].upper() + w[1:] for w in words) or "Concept"


def source_of(unit_id):
    m = re.match(r"ART(\d+)(?:\.(\d+))?(?:\.([a-h]|lead))?", unit_id)
    if m:
        s = f"Article {m.group(1)}" + (f"({m.group(2)})" if m.group(2) else "")
        return s + (f", point ({m.group(3)})" if m.group(3) and m.group(3) != "lead" else "")
    m = re.match(r"ANX1\.A\.(\d)", unit_id)
    return f"Annex I, Table A, row {m.group(1)}" if m else unit_id


def build(extraction, units):
    texts = {u["unit_id"]: u["text"] for u in units}
    n_units = len(texts)
    # 1-2. terms and concepts
    concepts, mention2concept = {}, {}
    for p in extraction:
        for e in p["entities"]:
            if e["label"] not in TOP:
                continue
            key = (e["label"], lemma_key(e["text"]))
            c = concepts.setdefault(key, {"type": e["label"], "variants": Counter(), "units": set()})
            c["variants"][e["text"]] += 1
            c["units"].add(p["unit_id"])
            mention2concept[(p["unit_id"], norm(e["text"]))] = key
    for key, c in concepts.items():
        c["label"] = c["variants"].most_common(1)[0][0]
        c["frequency"] = sum(c["variants"].values())
        c["relevance"] = round(c["frequency"] * math.log(1 + n_units / len(c["units"])), 3)
    names, ids = set(), {}
    for key, c in sorted(concepts.items(), key=lambda kv: (-kv[1]["frequency"], kv[1]["label"])):
        base = camel(c["label"]) if c["type"] != "LEGAL_REF" else "Ref_" + camel(c["label"])[:40]
        name, k = base, 2
        while name in names:
            name, k = f"{base}{k}", k + 1
        names.add(name)
        ids[key] = name
        c["id"] = name
    # 3. taxonomy by modifier stripping (head-suffix), within the same top class
    for key, c in concepts.items():
        c["parent"] = TOP[c["type"]]
        if c["type"] == "LEGAL_REF":
            continue
        words = key[1].split()
        for i in range(1, len(words)):
            cand = (c["type"], " ".join(words[i:]))
            if cand in concepts:
                c["parent"] = ids[cand]
                break
    # 4-5. rules and properties
    rules, refs, prop_use = [], [], defaultdict(lambda: [Counter(), Counter()])
    for p in extraction:
        for r in p["relations"]:
            s = mention2concept.get((p["unit_id"], norm(r["subject"])))
            o = mention2concept.get((p["unit_id"], norm(r["object"])))
            if r["predicate"] == "REFERS_TO":
                if s and o:
                    refs.append({"subject": ids[s], "reference": ids[o], "source": source_of(p["unit_id"])})
                    prop_use["REFERS_TO"][0][concepts[s]["type"]] += 1
                    prop_use["REFERS_TO"][1][concepts[o]["type"]] += 1
                continue
            if r["predicate"] not in NORMATIVE or not s:
                continue
            value = r["object"] if (o is None) else ""
            rules.append({"rule_id": f"R{len(rules) + 1:03d}", "type": r["predicate"], "subject": ids[s],
                          "object": ids[o] if o else "", "value": value, "qualifier": r.get("qualifier", ""),
                          "unit_id": p["unit_id"], "source": source_of(p["unit_id"]),
                          "provision": texts.get(p["unit_id"], "")})
            prop_use[r["predicate"]][0][concepts[s]["type"]] += 1
            prop_use[r["predicate"]][1][concepts[o]["type"] if o else "VALUE"] += 1
    props = [{"property": PROPS[t], "relation_type": t,
              "domain": TOP.get(prop_use[t][0].most_common(1)[0][0], "Product") if prop_use[t][0] else "Product",
              "range": (TOP.get(prop_use[t][1].most_common(1)[0][0], "xsd:string") if prop_use[t][1] else "")
              if t != "LIMIT" else "Threshold (value + parameter)",
              "uses": sum(prop_use[t][0].values())} for t in PROPS]
    return {"concepts": sorted(concepts.values(), key=lambda c: (c["type"], c["parent"], c["label"])),
            "properties": props, "rules": rules, "references": refs}


def to_owl(onto, path):
    from rdflib import RDF, RDFS, OWL, XSD, Graph, Literal, Namespace
    ex = Namespace(NS)
    g = Graph()
    g.bind("r2c", ex)
    g.bind("owl", OWL)
    for top in list(TOP.values()) + ["ComplianceRule", "Threshold"]:
        g.add((ex[top], RDF.type, OWL.Class))
    for t, cls in (("OBLIGATION", "Obligation"), ("PERMISSION", "Permission"), ("PROHIBITION", "Prohibition"),
                   ("LIMIT", "QualityLimit"), ("CONDITION", "Condition")):
        g.add((ex[cls], RDF.type, OWL.Class))
        g.add((ex[cls], RDFS.subClassOf, ex.ComplianceRule))
    for c in onto["concepts"]:
        node = ex[c["id"]]
        if c["type"] == "LEGAL_REF":
            g.add((node, RDF.type, ex.LegalReference))
        else:
            g.add((node, RDF.type, OWL.Class))
            g.add((node, RDFS.subClassOf, ex[c["parent"]]))
        g.add((node, RDFS.label, Literal(c["label"], lang="en")))
    for p in onto["properties"]:
        g.add((ex[p["property"]], RDF.type, OWL.ObjectProperty))
    for dp in ("thresholdValue", "onParameter", "sourceProvision", "provisionText", "appliesTo", "concerns"):
        g.add((ex[dp], RDF.type, OWL.DatatypeProperty if dp in ("thresholdValue", "sourceProvision",
                                                                  "provisionText") else OWL.ObjectProperty))
    cls_of = {"OBLIGATION": "Obligation", "PERMISSION": "Permission", "PROHIBITION": "Prohibition",
              "LIMIT": "QualityLimit", "CONDITION": "Condition"}
    for r in onto["rules"]:
        node = ex[r["rule_id"]]
        g.add((node, RDF.type, ex[cls_of[r["type"]]]))
        for subj in r["subject"].split(" OR "):
            g.add((node, ex.appliesTo, ex[subj.strip()]))
        if r["object"]:
            g.add((node, ex.concerns, ex[r["object"]]))
        if r.get("threshold"):
            t = r["threshold"]
            g.add((node, ex.thresholdOperator, Literal(t.get("operator", ""))))
            g.add((node, ex.thresholdNumeric, Literal(t.get("numeric_value", ""))))
            g.add((node, ex.thresholdUnit, Literal(t.get("unit", ""))))
        if r.get("condition"):
            g.add((node, ex.conditionExpression, Literal(r["condition"].get("expression", ""))))
        if r["value"]:
            g.add((node, ex.thresholdValue, Literal(r["value"], datatype=XSD.string)))
        if r["qualifier"]:
            g.add((node, ex.onParameter, Literal(r["qualifier"])))
        g.add((node, ex.sourceProvision, Literal(r["source"])))
    for ref in onto["references"]:
        g.add((ex[ref["subject"]], ex.refersTo, ex[ref["reference"]]))
    g.serialize(path, format="turtle")
    return len(g)


HEAD = Font(bold=True, color="FFFFFF")
FILL = PatternFill("solid", fgColor="2F5496")
WRAP = Alignment(wrap_text=True, vertical="top")


def sheet(wb, title, header, rows, widths, decision_col, choices):
    ws = wb.create_sheet(title)
    ws.append(header)
    for r in rows:
        ws.append(r)
    for i, w in enumerate(widths, 1):
        ws.column_dimensions[ws.cell(row=1, column=i).column_letter].width = w
    for c in ws[1]:
        c.font, c.fill, c.alignment = HEAD, FILL, WRAP
    for row in ws.iter_rows(min_row=2):
        for c in row:
            c.alignment = WRAP
            if isinstance(c.value, str) and c.value.startswith("="):   # e.g. '= 0,0' must stay text
                c.data_type = "s"
    dv = DataValidation(type="list", formula1='"' + ",".join(choices) + '"', allow_blank=True)
    ws.add_data_validation(dv)
    dv.add(f"{decision_col}2:{decision_col}{len(rows) + 200}")
    ws.freeze_panes = "A2"


def review_workbook(onto, path):
    wb = Workbook()
    ws = wb.active
    ws.title = "Instructions"
    for i, line in enumerate([
        "Reg2Chain - expert refinement of the automatically learned ontology",
        "The ontology was learned automatically (Text2Onto approach) from the Entity Extraction Agent output on all 63 provision units.",
        "For each row, choose a decision. Concepts: KEEP / EDIT (new label and/or new parent) / MERGE (give the concept id to merge into) / DELETE.",
        "Rules: KEEP / EDIT (corrected type, subject, object or value) / DELETE. Add missing concepts or rules at the bottom with decision ADD.",
        "Parent = the superclass of the concept (top classes: Product, QualityParameter, LabelItem, Actor, Process, LegalReference).",
        "Record your total time in the 'Log' sheet. The number of corrections and the time are reported in the paper (expert effort).",
    ], 1):
        c = ws.cell(row=i, column=1, value=line)
        c.alignment = WRAP
        if i == 1:
            c.font = Font(bold=True, size=13, color="2F5496")
    ws.column_dimensions["A"].width = 140
    sheet(wb, "Concepts",
          ["concept_id", "label", "top_class", "parent", "frequency", "n_units", "variants", "decision",
           "new_label", "new_parent", "merge_into", "comment"],
          [[c["id"], c["label"], TOP[c["type"]], c["parent"], c["frequency"], len(c["units"]),
            "; ".join(v for v in c["variants"] if v != c["label"]), "", "", "", "", ""] for c in onto["concepts"]],
          [28, 40, 16, 26, 10, 9, 30, 12, 30, 24, 20, 30], "H", ["KEEP", "EDIT", "MERGE", "DELETE", "ADD"])
    sheet(wb, "Properties",
          ["property", "relation_type", "domain", "range", "uses", "decision", "new_domain", "new_range", "comment"],
          [[p["property"], p["relation_type"], p["domain"], p["range"], p["uses"], "", "", "", ""]
           for p in onto["properties"]],
          [20, 16, 18, 26, 8, 12, 18, 18, 30], "F", ["KEEP", "EDIT", "DELETE", "ADD"])
    sheet(wb, "Rules",
          ["rule_id", "type", "subject", "object", "value", "qualifier", "source", "provision", "decision",
           "corrected_type", "corrected_subject", "corrected_object", "corrected_value", "comment"],
          [[r["rule_id"], r["type"], r["subject"], r["object"], r["value"], r["qualifier"], r["source"],
            r["provision"], "", "", "", "", "", ""] for r in onto["rules"]],
          [8, 13, 26, 26, 12, 22, 22, 70, 12, 14, 22, 22, 14, 28], "I", ["KEEP", "EDIT", "DELETE", "ADD"])
    log = wb.create_sheet("Log")
    log.append(["field", "value"])
    for k in ["expert name", "expertise", "date(s)", "total time (minutes)", "comments"]:
        log.append([k, ""])
    log.column_dimensions["A"].width = 26
    log.column_dimensions["B"].width = 80
    for c in log[1]:
        c.font, c.fill = HEAD, FILL
    wb.save(path)


def main(extraction_path, units_csv, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    units = list(csv.DictReader(open(units_csv, encoding="utf-8")))
    onto = build(load_jsonl(extraction_path), units)
    serial = json.loads(json.dumps(onto, default=lambda o: sorted(o) if isinstance(o, set) else dict(o)))
    json.dump(serial, open(os.path.join(out_dir, "ontology_learned.json"), "w", encoding="utf-8"),
              ensure_ascii=False, indent=1)
    n = to_owl(onto, os.path.join(out_dir, "ontology_learned.ttl"))
    review_workbook(onto, os.path.join(out_dir, "Reg2Chain_ontology_review.xlsx"))
    by_type = Counter(TOP[c["type"]] for c in onto["concepts"])
    print(f"Ontology Agent: {len(onto['concepts'])} concepts {dict(by_type)}, "
          f"{sum(c['parent'] not in TOP.values() for c in onto['concepts'])} learned subclass links, "
          f"{len(onto['properties'])} properties, {len(onto['rules'])} rules, "
          f"{len(onto['references'])} refersTo links, {n} OWL triples -> {out_dir}")


if __name__ == "__main__":
    main(*sys.argv[1:4])


In [ ]:
!python scripts/ontology_agent.py {O}/extraction_all.jsonl step1_gold/units.csv {O}/learned

### 4.3 Expert refinement (legal expert, about 1.5 to 2 h)
Give the legal expert `step3_ontology/learned/Reg2Chain_ontology_review.xlsx`. She copies it to `step3_ontology/review/Reg2Chain_ontology_review.xlsx` and fills in, for every row:
- Concepts: KEEP / EDIT (new label, new parent) / MERGE (concept to merge into) / DELETE, and adds missing concepts with ADD;
- Properties: KEEP / EDIT / DELETE;
- Rules: KEEP / EDIT (corrected type, subject, object or value) / DELETE, and adds missing rules with ADD;
- Log: name, expertise, date, total time.

Typical corrections expected from a Text2Onto-style taxonomy: legal categories wrongly nested by modifier stripping (e.g. *extra virgin olive oil* under *virgin olive oil*), duplicate concepts to merge, generic terms to delete.

### 4.4 Apply the refinement and measure the expert effort
Outputs in `step3_ontology/refined/`: `ontology_refined.ttl` (OWL), `ontology_refined.json`, `refinement_report.txt` / `.csv` (number and type of corrections, expert time).

In [ ]:
%%writefile scripts/apply_ontology_refinement.py
"""
Section 4 - Apply the expert refinement to the learned ontology and measure the expert effort.

Reads the reviewed workbook and the learned ontology and applies the expert decisions:
  Concepts   : KEEP / EDIT (new label, new parent) / MERGE (merge_into) / DELETE / ADD
  Properties : KEEP / EDIT (new_label = new property name, new_domain, new_range) / DELETE
  Rules      : KEEP / EDIT (corrected type / subject / object / value) / DELETE / ADD
               - a subject written 'A OR B OR C' means that the rule applies to each category
               - an object 'Condition_x' / 'Threshold_x' refers to the Conditions / Thresholds sheets
Optional sheets added by the experts are integrated:
  Thresholds  : structured limits (parameter, operator, numeric value, unit, product scope) per rule
  Conditions  : condition expressions per rule
  Schema gaps : provisions the schema cannot express (reported, kept in the output)
Concepts referenced in rules but absent from the Concepts sheet are created and counted as
implicit additions. Rules added by the experts are linked to their provision via units.csv.

Outputs: ontology_refined.json, ontology_refined.ttl, refinement_report.txt / .csv

  python apply_ontology_refinement.py ontology_learned.json review.xlsx OUT_DIR [units.csv]
"""
import csv
import json
import os
import re
import sys
from collections import Counter

from openpyxl import load_workbook

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from ontology_agent import TOP, camel, source_of, to_owl  # noqa: E402
from extraction_common import norm  # noqa: E402

TYPE_OF_TOP = {v: k for k, v in TOP.items()}
IDENT = re.compile(r"^[A-Za-z][A-Za-z0-9_]*$")


def _clean(v):
    """Cells such as '= 0,0' may come back as '#ERROR!' or empty after editing in a spreadsheet tool."""
    return "" if (not v or v.startswith("#")) else v


def canon_source(s):
    """Experts sometimes wrote 'Article 6(2)(a)', 'Article 10(a)' or added ', first sentence'; map to the
    source format used by the pipeline ('Article 6(2), point (a)') so the rule is linked to its provision."""
    s = re.sub(r",\s*(read with|first|second|third|last)\b.*$", "", (s or "").strip())
    s = re.sub(r"^(Article \d+(?:\(\d+\))?)\(([a-h])\)$", r"\1, point (\2)", s)
    return s


def rows(wb, name):
    if name not in wb.sheetnames:
        return []
    data = list(wb[name].iter_rows(values_only=True))
    if not data:
        return []
    hdr = [str(h).strip() if h is not None else f"col{i}" for i, h in enumerate(data[0])]
    return [dict(zip(hdr, [("" if v is None else str(v).strip()) for v in r])) for r in data[1:]
            if any(v not in (None, "") for v in r)]


def minutes(text):
    t = (text or "").lower().replace(",", ".")
    h = re.search(r"(\d+(?:\.\d+)?)\s*h", t)
    m = re.search(r"(\d+)\s*(?:min|m\b)", t)
    if h or m:
        return round(float(h.group(1)) * 60 if h else 0) + (int(m.group(1)) if m else 0)
    return int(float(t)) if re.fullmatch(r"\d+(?:\.\d+)?", t) else None


def parts(v):
    return [p.strip() for p in (v or "").split(" OR ") if p.strip()]


def main(learned_path, review_path, out_dir, units_csv=None):
    os.makedirs(out_dir, exist_ok=True)
    onto = json.load(open(learned_path, encoding="utf-8"))
    wb = load_workbook(review_path, data_only=True)
    stats = {s: Counter() for s in ("Concepts", "Properties", "Rules")}
    unit_of_source = {}
    if units_csv:
        for u in csv.DictReader(open(units_csv, encoding="utf-8")):
            unit_of_source.setdefault(source_of(u["unit_id"]), u["unit_id"])

    # ---------------- concepts
    by_id = {c["id"]: c for c in onto["concepts"]}
    merged, deleted, concepts = {}, set(), {}
    for r in rows(wb, "Concepts"):
        d = (r.get("decision") or "").upper() or "UNDECIDED"
        stats["Concepts"][d] += 1
        cid = r.get("concept_id") or camel(r.get("new_label") or r.get("label", ""))
        if d == "DELETE":
            deleted.add(cid)
            continue
        if d == "MERGE" and r.get("merge_into"):
            merged[cid] = r["merge_into"]
            continue
        c = dict(by_id.get(cid, {"id": cid, "type": TYPE_OF_TOP.get(r.get("top_class"), "PRODUCT"),
                                  "frequency": 0, "units": [], "variants": {}}))
        c["id"] = cid
        c["label"] = r.get("new_label") or r.get("label") or c.get("label", cid)
        c["parent"] = r.get("new_parent") or r.get("parent") or c.get("parent") or TOP[c["type"]]
        c["origin"] = "expert" if d == "ADD" else "learned"
        concepts[cid] = c

    def resolve(cid):
        seen = set()
        while cid in merged and cid not in seen:
            seen.add(cid)
            cid = merged[cid]
        return None if cid in deleted else cid

    # ---------------- thresholds, conditions, schema gaps (sheets added by the experts)
    thresholds = {t["rule_id"]: t for t in rows(wb, "Thresholds") if t.get("rule_id")}
    threshold_ids = {t.get("threshold_id"): t for t in rows(wb, "Thresholds") if t.get("threshold_id")}
    conditions = {c["condition_id"]: c for c in rows(wb, "Conditions") if c.get("condition_id")}
    cond_by_rule = {c["rule_id"]: c for c in conditions.values() if c.get("rule_id")}
    gaps_rows = list(wb["Schema gaps"].iter_rows(values_only=True)) if "Schema gaps" in wb.sheetnames else []
    gaps = [[("" if v is None else str(v)) for v in r] for r in gaps_rows if any(v not in (None, "") for v in r)]

    # ---------------- properties
    props = []
    for r in rows(wb, "Properties"):
        d = (r.get("decision") or "").upper() or "UNDECIDED"
        stats["Properties"][d] += 1
        if d == "DELETE":
            continue
        name = r.get("new_label") if IDENT.match(r.get("new_label") or "") else r["property"]
        props.append({"property": name, "relation_type": r["relation_type"],
                      "domain": r.get("new_domain") or r["domain"], "range": r.get("new_range") or r["range"],
                      "learned_name": r["property"]})

    # ---------------- rules
    by_rule = {x["rule_id"]: x for x in onto["rules"]}
    rules, implicit = [], Counter()

    def ensure(cid, role):
        """Concept referenced by a rule: resolve merges, create it if the experts introduced it."""
        if not cid or cid.startswith(("Condition_", "Threshold_")):
            return cid
        rid = resolve(cid)
        if rid is None:
            return None
        if rid not in concepts:
            concepts[rid] = {"id": rid, "type": "PRODUCT" if role == "subject" else "LABEL_ITEM",
                             "label": re.sub(r"(?<!^)(?=[A-Z])", " ", rid).lower(), "frequency": 0,
                             "units": [], "variants": {}, "origin": "expert (implicit)",
                             "parent": TOP["PRODUCT" if role == "subject" else "LABEL_ITEM"]}
            implicit[role] += 1
        return rid

    edit_kind = Counter()
    renumbered = 0
    restored_values = 0
    for r in rows(wb, "Rules"):
        d = (r.get("decision") or "").upper() or "UNDECIDED"
        stats["Rules"][d] += 1
        if d == "EDIT":
            changed_fields = [k for k in ("corrected_type", "corrected_subject", "corrected_object", "corrected_value")
                              if r.get(k)]
            edit_kind["content corrected" if changed_fields else "enriched only (threshold / condition / comment)"] += 1
        if d == "DELETE":
            continue
        # The review row is self-contained (source + provision). The learned rule with the same id is
        # reused only if it comes from the same provision: if the learned ontology was regenerated after
        # the review, rule ids may have been renumbered, and the row's own source must win.
        learned = by_rule.get(r["rule_id"])
        if learned and r.get("source") and learned.get("source") != r.get("source"):
            learned = None
            renumbered += 1
        base = dict(learned or {"rule_id": r["rule_id"] or f"X{len(rules) + 1:03d}",
                                "unit_id": "", "provision": r.get("provision", "")})
        subj = [s for s in (ensure(p, "subject") for p in parts(r.get("corrected_subject") or r.get("subject")))
                if s]
        obj = ensure(r.get("corrected_object") or r.get("object", ""), "object") or ""
        base.update({"type": r.get("corrected_type") or r["type"], "subject": " OR ".join(subj), "object": obj,
                     "value": r.get("corrected_value") or _clean(r.get("value", "")) or base.get("value", ""),
                     "qualifier": r.get("qualifier", "") or base.get("qualifier", ""),
                     "source": r.get("source") or base.get("source", ""), "decision": d})
        src = canon_source(base["source"])
        if src in unit_of_source:
            base["unit_id"] = unit_of_source[src]
        t = thresholds.get(base["rule_id"]) or threshold_ids.get(obj)
        if base["type"] == "LIMIT" and not base.get("value"):
            # a cell typed '= 0,0' is read as a formula and becomes '#ERROR!': take the value of the learned
            # rule of the same provision and parameter, or rebuild '= <numeric value>' from the Thresholds sheet
            same = [x.get("value", "") for x in onto["rules"] if x.get("source") == base["source"]
                    and norm(x.get("qualifier", "")) == norm(base.get("qualifier", "")) and x.get("value")]
            if same:
                base["value"] = same[0]
            elif t and t.get("numeric_value"):
                base["value"] = "= " + str(t["numeric_value"])
            restored_values += 1 if base.get("value") else 0
        if t and _clean(str(t.get("operator") or "")) == "" and base.get("value", "").startswith("="):
            t = dict(t, operator="=")
        if t:
            base["threshold"] = {k: t.get(k, "") for k in ("parameter", "operator", "numeric_value", "unit",
                                                             "product_scope")}
        c = conditions.get(obj) or cond_by_rule.get(base["rule_id"])
        if c:
            base["condition"] = {"id": c["condition_id"], "expression": c.get("condition_expression", "")}
        if subj:
            rules.append(base)

    for c in concepts.values():
        if c["parent"] not in TOP.values():
            c["parent"] = resolve(c["parent"]) or TOP[c["type"]]
            if c["parent"] not in concepts and c["parent"] not in TOP.values():
                c["parent"] = TOP[c["type"]]

    refs = [{"subject": resolve(x["subject"]), "reference": resolve(x["reference"]), "source": x["source"]}
            for x in onto["references"]]
    refs = [x for x in refs if x["subject"] in concepts and x["reference"] in concepts]
    refined = {"concepts": list(concepts.values()), "properties": props, "rules": rules, "references": refs,
               "conditions": list(conditions.values()), "thresholds": list(threshold_ids.values()),
               "schema_gaps": gaps}
    json.dump(refined, open(os.path.join(out_dir, "ontology_refined.json"), "w", encoding="utf-8"),
              ensure_ascii=False, indent=1)
    n = to_owl(refined, os.path.join(out_dir, "ontology_refined.ttl"))

    log = {r["field"]: r.get("value", "") for r in rows(wb, "Log")}
    lines, table = [], []
    for s, cnt in stats.items():
        total = sum(cnt.values())
        learned = total - cnt.get("ADD", 0)
        changed = sum(v for k, v in cnt.items() if k in ("EDIT", "MERGE", "DELETE"))
        lines.append(f"{s}: {learned} learned items | " + ", ".join(f"{k} {v}" for k, v in sorted(cnt.items())) +
                     f" | learned items corrected: {changed}/{learned} ({100 * changed / max(1, learned):.1f} %), "
                     f"items added by the experts: {cnt.get('ADD', 0)}")
        for k, v in sorted(cnt.items()):
            table.append([s, k, v])
    lines.append(f"Rules marked EDIT: {dict(edit_kind)}")
    lines.append(f"Rules linked to their provision: {sum(1 for r in rules if r.get('unit_id'))}/{len(rules)} "
                 f"(review rows re-aligned on their own source: {renumbered}); "
                 f"limit values restored after a spreadsheet formula error: {restored_values}")
    lines.append(f"Concepts introduced implicitly in rules: {sum(implicit.values())} {dict(implicit)}")
    lines.append(f"Structured thresholds: {len(threshold_ids)}, conditions: {len(conditions)}, "
                 f"schema gaps identified: {max(0, len(gaps))}")
    t_min = minutes(log.get("total time (minutes)", ""))
    lines.append(f"Experts: {log.get('expert name', '')} ({log.get('expertise', '')}); "
                 f"time: {log.get('total time (minutes)', '')}" + (f" = {t_min} min" if t_min else ""))
    lines.append(f"Refined ontology: {len(concepts)} concepts, {len(props)} properties, {len(rules)} rules "
                 f"({sum(1 for r in rules if ' OR ' in r['subject'])} with several product categories), "
                 f"{len(refs)} refersTo links, {n} OWL triples")
    open(os.path.join(out_dir, "refinement_report.txt"), "w", encoding="utf-8").write("\n".join(lines) + "\n")
    with open(os.path.join(out_dir, "refinement_report.csv"), "w", newline="", encoding="utf-8") as f:
        csv.writer(f).writerows([["sheet", "decision", "count"]] + table)
    if gaps:
        with open(os.path.join(out_dir, "schema_gaps.csv"), "w", newline="", encoding="utf-8") as f:
            csv.writer(f).writerows(gaps)
    print("\n".join(lines))


if __name__ == "__main__":
    main(*sys.argv[1:5])


In [ ]:
!python scripts/apply_ontology_refinement.py {O}/learned/ontology_learned.json {O}/review/Reg2Chain_ontology_review.xlsx {O}/refined step1_gold/units.csv

## 5. Graph Construction Agent and Evaluation 2 (Table 2)
The refined ontology is turned into the regulatory knowledge graph: concepts, legal references, compliance rules (thresholds attached to the rule of the category they apply to) and the provisions they derive from (`DERIVED_FROM`). The graph is saved as JSON and loaded into Neo4j Aura when `NEO4J_URI`, `NEO4J_USER` (or `NEO4J_USERNAME`) and `NEO4J_PASSWORD` are set in the Secrets panel.

Evaluation 2 compares three graphs, built from the spaCy baseline extraction, from the Reg2Chain extraction before refinement (learned) and after (refined):
- structure: nodes, edges, average degree, isolated nodes, components, traceable rules;
- quality against the gold standard (test units): precision and recall of the rules, precision of the concepts;
- coverage: share of the 55 validated checklist requirements represented in the graph, as judged by the legal expert.

In [ ]:
# session setup (needed after a reconnection or a change of runtime)
from google.colab import drive
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Reg2Chain_revision')

In [ ]:
from google.colab import drive, userdata
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Reg2Chain_revision')
!pip install -q networkx neo4j rdflib openpyxl spacy
!python -m spacy download en_core_web_sm
# read from the Secrets panel, never printed; accepts the names of the Aura credentials file (NEO4J_USERNAME)
for k, names in (('NEO4J_URI', ['NEO4J_URI']), ('NEO4J_USER', ['NEO4J_USER', 'NEO4J_USERNAME']),
                 ('NEO4J_PASSWORD', ['NEO4J_PASSWORD'])):
    for name in names:
        try:
            os.environ[k] = userdata.get(name)
            break
        except Exception:
            pass
    else:
        print(f'Secret {k} not set: the graph will be built without Neo4j')
for d in ['step4_kg', 'step4_kg/baseline', 'step4_kg/coverage']:
    os.makedirs(d, exist_ok=True)
G = 'step1_gold/gold'; O = 'step3_ontology'; K = 'step4_kg'

### 5.1 Build the knowledge graphs
The refined graph is the Reg2Chain knowledge graph (loaded into Neo4j). The *learned* and *baseline* graphs are built only for the comparison of Evaluation 2.

In [ ]:
%%writefile scripts/kg_builder.py
"""
Section 5 - Graph Construction Agent.

Builds the regulatory knowledge graph from an ontology (learned or expert-refined):
  (:Concept {id, label, top_class})            one node per concept (Product, LabelItem, ...)
  (:LegalReference {id, label})                legal references
  (:Rule {id, type, value, parameter, source}) one node per compliance rule
  (:Provision {id, source, text})              the provision unit each rule comes from
  edges: SUBCLASS_OF, APPLIES_TO (rule -> subject concept), CONCERNS (rule -> object concept),
         DERIVED_FROM (rule -> provision), REFERS_TO (concept -> legal reference)
Each threshold is attached to the rule of the product category it applies to, and each rule is
traceable to its source provision.

The graph is written as JSON (nodes/edges) and, if Neo4j credentials are available
(environment variables NEO4J_URI, NEO4J_USER, NEO4J_PASSWORD - in Colab from the Secrets panel),
loaded into Neo4j (the database is cleared first).

  python kg_builder.py ontology.json units.csv kg.json [--neo4j]
"""
import csv
import json
import os
import sys

TOPS = {"Product", "QualityParameter", "LabelItem", "Actor", "Process", "LegalReference"}


def build_graph(onto, units):
    texts = {u["unit_id"]: u["text"] for u in units}
    nodes, edges = {}, []
    for t in TOPS - {"LegalReference"}:
        nodes[t] = {"id": t, "kind": "Concept", "label": t, "top_class": t}
    for c in onto["concepts"]:
        top = {"PRODUCT": "Product", "PARAMETER": "QualityParameter", "LABEL_ITEM": "LabelItem",
               "ACTOR": "Actor", "PROCESS": "Process", "LEGAL_REF": "LegalReference"}.get(c["type"], "Product")
        kind = "LegalReference" if top == "LegalReference" else "Concept"
        nodes[c["id"]] = {"id": c["id"], "kind": kind, "label": c["label"], "top_class": top}
        if kind == "Concept" and c.get("parent") and c["parent"] != c["id"]:
            edges.append((c["id"], "SUBCLASS_OF", c["parent"]))
    for r in onto["rules"]:
        rid = r["rule_id"]
        t = r.get("threshold") or {}
        nodes[rid] = {"id": rid, "kind": "Rule", "type": r["type"], "value": r.get("value", ""),
                      "parameter": t.get("parameter") or r.get("qualifier", ""), "source": r.get("source", ""),
                      "operator": t.get("operator", ""), "numeric_value": t.get("numeric_value", ""),
                      "unit": t.get("unit", ""), "condition": (r.get("condition") or {}).get("expression", "")}
        for subj in (r.get("subject") or "").split(" OR "):
            if subj.strip():
                edges.append((rid, "APPLIES_TO", subj.strip()))
        obj = r.get("object") or ""
        if obj.startswith("Condition_"):
            nodes.setdefault(obj, {"id": obj, "kind": "Condition", "label": obj,
                                   "expression": (r.get("condition") or {}).get("expression", "")})
        elif obj.startswith("Threshold_"):
            nodes.setdefault(obj, {"id": obj, "kind": "Threshold", "label": obj, **{k: str(v) for k, v in t.items()}})
        if obj:
            edges.append((rid, "CONCERNS", obj))
        uid = r.get("unit_id") or ""
        if uid:
            pid = "P_" + uid
            nodes.setdefault(pid, {"id": pid, "kind": "Provision", "source": r.get("source", ""),
                                   "text": texts.get(uid, r.get("provision", ""))})
            edges.append((rid, "DERIVED_FROM", pid))
    for x in onto.get("references", []):
        edges.append((x["subject"], "REFERS_TO", x["reference"]))
    edges = [e for e in dict.fromkeys(edges) if e[0] in nodes and e[2] in nodes]
    return {"nodes": list(nodes.values()), "edges": [{"source": a, "type": t, "target": b} for a, t, b in edges]}


def load_neo4j(graph):
    from neo4j import GraphDatabase
    uri, pwd = os.environ.get("NEO4J_URI"), os.environ.get("NEO4J_PASSWORD")
    user = os.environ.get("NEO4J_USER") or os.environ.get("NEO4J_USERNAME")
    if not (uri and user and pwd):
        print("Neo4j credentials not set (NEO4J_URI / NEO4J_USER / NEO4J_PASSWORD): Neo4j loading skipped")
        return
    with GraphDatabase.driver(uri, auth=(user, pwd)) as driver, driver.session() as s:
        s.run("MATCH (n) DETACH DELETE n")
        for kind in ("Concept", "LegalReference", "Rule", "Provision", "Condition", "Threshold"):
            batch = [n for n in graph["nodes"] if n["kind"] == kind]
            s.run(f"UNWIND $rows AS r MERGE (n:{kind} {{id: r.id}}) SET n += r", rows=batch)
        for et in sorted({e["type"] for e in graph["edges"]}):
            batch = [e for e in graph["edges"] if e["type"] == et]
            s.run(f"UNWIND $rows AS r MATCH (a {{id: r.source}}), (b {{id: r.target}}) MERGE (a)-[:{et}]->(b)",
                  rows=batch)
        n = s.run("MATCH (n) RETURN count(n) AS c").single()["c"]
        m = s.run("MATCH ()-[r]->() RETURN count(r) AS c").single()["c"]
    print(f"loaded into Neo4j: {n} nodes, {m} relationships")


def main(onto_path, units_csv, out_path, *flags):
    onto = json.load(open(onto_path, encoding="utf-8"))
    units = list(csv.DictReader(open(units_csv, encoding="utf-8")))
    graph = build_graph(onto, units)
    json.dump(graph, open(out_path, "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    kinds = {}
    for n in graph["nodes"]:
        kinds[n["kind"]] = kinds.get(n["kind"], 0) + 1
    print(f"Graph Construction Agent: {len(graph['nodes'])} nodes {kinds}, {len(graph['edges'])} edges -> {out_path}")
    if "--neo4j" in flags:
        load_neo4j(graph)


if __name__ == "__main__":
    main(*sys.argv[1:])


In [ ]:
# Reg2Chain knowledge graph (expert-refined ontology) -> JSON + Neo4j
!python scripts/kg_builder.py {O}/refined/ontology_refined.json step1_gold/units.csv {K}/kg_refined.json --neo4j
# before expert refinement
!python scripts/kg_builder.py {O}/learned/ontology_learned.json step1_gold/units.csv {K}/kg_learned.json
# baseline: same pipeline on the spaCy baseline extraction (all units)
!python scripts/extract_baseline.py {O}/all_units.jsonl {K}/baseline/baseline_all.jsonl
!python scripts/ontology_agent.py {K}/baseline/baseline_all.jsonl step1_gold/units.csv {K}/baseline
!python scripts/kg_builder.py {K}/baseline/ontology_learned.json step1_gold/units.csv {K}/kg_baseline.json

### 5.2 Evaluation 2: structure and semantic quality (Table 2)
Output: `step4_kg/table2_kg_quality.csv`.

In [ ]:
%%writefile scripts/kg_evaluation.py
"""
Section 5 - Evaluation 2: quality of the knowledge graph (replaces the former Table 2).

Structure, semantic quality against the gold standard, and coverage of the requirements:

  metrics       structural metrics of each graph (nodes, edges, average degree, isolated nodes,
                connected components, traceable rules) AND semantic quality against the gold
                standard, on the TEST units only: precision / recall of the rules (relaxed match
                with the gold normative relations of the same provision) and of the concepts.
  coverage-sheet  coverage workbook: for each requirement of the validated checklist, candidate
                rules of the graph (same provision; same product + parameter + value for limits);
                the expert marks COVERED / PARTIAL / NOT_COVERED.
  coverage-score  coverage of the regulation's requirements (overall and by on-chain class).

  python kg_evaluation.py metrics gold_test.jsonl OUT_DIR name=kg.json [name=kg.json ...]
  python kg_evaluation.py coverage-sheet checklist.xlsx kg.json coverage.xlsx
  python kg_evaluation.py coverage-score coverage_reviewed.xlsx OUT_DIR
"""
import csv
import json
import os
import re
import sys
from collections import Counter

import networkx as nx
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from evaluate_extraction import text_overlap  # noqa: E402
from extraction_common import load_jsonl, norm  # noqa: E402

NORMATIVE = {"OBLIGATION", "PERMISSION", "PROHIBITION", "LIMIT", "CONDITION"}
TOP2TYPE = {"Product": "PRODUCT", "QualityParameter": "PARAMETER", "LabelItem": "LABEL_ITEM", "Actor": "ACTOR",
            "Process": "PROCESS", "LegalReference": "LEGAL_REF"}


# ------------------------------------------------------------------ metrics
def structure(graph):
    g = nx.Graph()
    g.add_nodes_from(n["id"] for n in graph["nodes"])
    g.add_edges_from((e["source"], e["target"]) for e in graph["edges"])
    comps = list(nx.connected_components(g))
    rules = [n for n in graph["nodes"] if n["kind"] == "Rule"]
    traced = {e["source"] for e in graph["edges"] if e["type"] == "DERIVED_FROM"}
    return {"nodes": g.number_of_nodes(), "edges": g.number_of_edges(),
            "avg_degree": round(2 * g.number_of_edges() / max(1, g.number_of_nodes()), 2),
            "isolated_nodes": sum(1 for _, d in g.degree() if d == 0),
            "components": len(comps),
            "largest_component_share": round(max((len(c) for c in comps), default=0) / max(1, g.number_of_nodes()), 3),
            "concepts": sum(n["kind"] == "Concept" for n in graph["nodes"]),
            "rules": len(rules),
            "traceable_rules_share": round(sum(r["id"] in traced for r in rules) / max(1, len(rules)), 3)}


def rule_views(graph):
    label = {n["id"]: n.get("label", n["id"]) for n in graph["nodes"]}
    cond = {n["id"]: n.get("expression", "") for n in graph["nodes"] if n["kind"] == "Condition"}
    out_e = {}
    for e in graph["edges"]:
        out_e.setdefault(e["source"], {}).setdefault(e["type"], []).append(e["target"])
    rules = []
    for n in graph["nodes"]:
        if n["kind"] != "Rule":
            continue
        ed = out_e.get(n["id"], {})
        prov = ed.get("DERIVED_FROM", [""])[0]
        rules.append({"id": n["id"], "type": n["type"], "unit_id": prov[2:] if prov.startswith("P_") else "",
                      "subject": " | ".join(label.get(x, x) for x in (ed.get("APPLIES_TO") or [""])),
                      "subjects": [label.get(x, x) for x in (ed.get("APPLIES_TO") or [])],
                      "object": label.get((ed.get("CONCERNS") or [""])[0], "") or n.get("value", ""),
                      "value": n.get("value", ""), "parameter": n.get("parameter", ""),
                      "condition": n.get("condition", "") or cond.get((ed.get("CONCERNS") or [""])[0], "")})
    return rules


def compact(s):
    """'PeroxideValueMEqO2Kg' and 'Peroxide value (mEq O2/kg)' -> 'peroxidevaluemeqo2kg' (identifier vs text form)."""
    return re.sub(r"[^0-9a-zδ]", "", (s or "").lower())


def same_param(a, b):
    ca, cb = compact(a), compact(b)
    return text_overlap(a, b) or (bool(ca) and bool(cb) and (ca == cb or ca in cb or cb in ca))


def same_value(a, b):
    """'≤ 0,80' == '≤ 0.8'; '= 0,0' == '= 0,00' (operator and number must both agree)."""
    pa, pb = (re.match(r"\s*([<>≤≥=]*)\s*([0-9]+(?:[.,][0-9]+)?)", x or "") for x in (a, b))
    if pa and pb:
        return pa.group(1) == pb.group(1) and float(pa.group(2).replace(",", ".")) == float(pb.group(2).replace(",", "."))
    return norm(a) == norm(b)


def same_rule(g, r):
    if g["predicate"] != r["type"] or not any(text_overlap(g["subject"], s) for s in (r.get("subjects") or [r["subject"]])):
        return False
    if g["predicate"] == "LIMIT":
        return same_value(g["object"], r["value"]) and same_param(g.get("qualifier", ""), r["parameter"])
    # a CONDITION may point to a Condition node: compare with its expression as well
    return any(text_overlap(g["object"], x) for x in (r["object"], r.get("condition", "")) if x)


def semantic(graph, gold):
    test_units = {g["unit_id"] for g in gold}
    gold_rules = [dict(r, unit_id=g["unit_id"]) for g in gold for r in g["relations"] if r["predicate"] in NORMATIVE]
    rules = [r for r in rule_views(graph) if r["unit_id"] in test_units]
    used, tp = set(), 0
    for g in gold_rules:
        for r in rules:
            if r["id"] not in used and r["unit_id"] == g["unit_id"] and same_rule(g, r):
                used.add(r["id"])
                tp += 1
                break
    p = tp / len(rules) if rules else 0.0
    rc = tp / len(gold_rules) if gold_rules else 0.0
    gold_concepts = {(e["label"], norm(e["text"])) for g in gold for e in g["entities"] if e["label"] != "VALUE"}
    concepts = [n for n in graph["nodes"] if n["kind"] in ("Concept", "LegalReference") and n["id"] not in TOP2TYPE]
    ok = sum(1 for n in concepts if any(t == TOP2TYPE.get(n.get("top_class")) and text_overlap(x, n["label"])
                                         for t, x in gold_concepts))
    return {"rules_test": len(rules), "gold_rules_test": len(gold_rules), "rule_TP": tp,
            "rule_precision": round(p, 3), "rule_recall": round(rc, 3),
            "rule_F1": round(2 * p * rc / (p + rc), 3) if p + rc else 0.0,
            "concept_precision": round(ok / max(1, len(concepts)), 3)}


def cmd_metrics(gold_path, out_dir, *specs):
    os.makedirs(out_dir, exist_ok=True)
    gold = load_jsonl(gold_path)
    rows = []
    for spec in specs:
        name, path = spec.split("=", 1)
        graph = json.load(open(path, encoding="utf-8"))
        rows.append({"graph": name, **structure(graph), **semantic(graph, gold)})
    keys = list(rows[0])
    with open(os.path.join(out_dir, "table2_kg_quality.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        w.writerows(rows)
    for k in keys:
        print(f"{k:<26}" + "".join(f"{str(r[k]):>22}" for r in rows))


# ------------------------------------------------------------------ coverage
def prov_key(s):
    """'Art. 8(1)' / 'Article 8(1)' -> ('8','1');  'Art. 10(a)' / 'Article 10, point (a)' -> ('10','a');
    anything with 'Annex I' -> ('ANNEX',)"""
    s = s or ""
    if re.search(r"Annex I\b", s) and not re.search(r"Art", s):
        return ("ANNEX",)
    m = re.search(r"Art(?:icle|\.)?\s*(\d+)\s*(?:\((\w+)\))?(?:,\s*point\s*\((\w)\))?", s)
    if not m:
        return ("?",)
    return tuple(x for x in (m.group(1), m.group(2) or m.group(3)) if x)


def requirements(checklist_path):
    ws = load_workbook(checklist_path, data_only=True)["Checklist"]
    data = list(ws.iter_rows(values_only=True))
    hdr = [str(h).strip() if h else "" for h in data[0]]
    reqs = []
    for row in data[1:]:
        r = dict(zip(hdr, ["" if v is None else str(v).strip() for v in row]))
        if not r.get("req_id") or r.get("expert_decision", "").upper() == "DELETE":
            continue
        r["final_requirement"] = r.get("corrected_requirement") or r.get("requirement")
        r["final_on_chain"] = r.get("corrected_on_chain") or r.get("on_chain")
        reqs.append(r)
    return reqs


def product_eq(a, b):
    """Same product category name, up to singular/plural (no partial overlap: 'virgin' != 'extra virgin')."""
    f = lambda x: " ".join(w[:-1] if w.endswith("s") and len(w) > 3 else w for w in norm(x).replace("-", " ").split())
    return f(a) == f(b)


def candidates(req, rules):
    key = prov_key(req["provision"])
    out = []
    for r in rules:
        rk = ("ANNEX",) if r["source"].startswith("Annex") else prov_key(r["source"])
        if key == ("ANNEX",) and rk == ("ANNEX",) and req["type"].upper() == "LIMIT":
            prod_ok = any(product_eq(req["applies_to"], s) for s in (r.get("subjects") or [r["subject"]]))
            val = re.search(r"(?:≤|≥|<|>|=)\s?\d+(?:,\d+)?", req["final_requirement"])
            val_ok = bool(val) and norm(val.group(0)).replace(" ", "") == norm(r["value"]).replace(" ", "")
            par_ok = text_overlap(req["final_requirement"].split("≤")[0].split(">")[0].split("=")[0], r["parameter"])
            if prod_ok and val_ok and par_ok:
                out.append((3, r))
            elif prod_ok and (val_ok or par_ok):
                out.append((1, r))
        elif key[0] != "ANNEX" and rk[:len(key)] == key:
            out.append((2 if len(key) > 1 or len(rk) == 1 else 1, r))
    out = sorted(out, key=lambda x: -x[0])
    return [r for _, r in out[:4]], (bool(out) and out[0][0] == 3)


def rule_text(r):
    obj = r["object"] or r["value"]
    q = f" [{r['parameter']}]" if r["parameter"] else ""
    return f"{r['id']}: {r['subject']} —{r['type']}→ {obj}{q} ({r['source']})"


def cmd_coverage_sheet(checklist_path, kg_path, out_path):
    graph = json.load(open(kg_path, encoding="utf-8"))
    rules = rule_views(graph)
    src = {n["id"]: n.get("source", "") for n in graph["nodes"] if n["kind"] == "Rule"}
    for r in rules:
        r["source"] = src.get(r["id"], "")
    wb = Workbook()
    ws = wb.active
    ws.title = "Instructions"
    for i, line in enumerate([
        "Reg2Chain - coverage of the regulation's requirements by the knowledge graph",
        "For each requirement of the validated checklist, the candidate rules of the knowledge graph are listed "
        "(same provision; for quality limits, same product, parameter and value).",
        "Choose in 'decision': COVERED (the graph contains the requirement), PARTIAL (only part of it), "
        "NOT_COVERED (absent). 'suggestion' is an automatic hint only (exact limit match).",
        "Record your time in the 'Log' sheet.",
    ], 1):
        c = ws.cell(row=i, column=1, value=line)
        c.alignment = Alignment(wrap_text=True)
        if i == 1:
            c.font = Font(bold=True, size=13, color="2F5496")
    ws.column_dimensions["A"].width = 140
    s = wb.create_sheet("Coverage")
    s.append(["req_id", "provision", "applies_to", "type", "requirement", "on_chain", "candidate_rules",
              "suggestion", "decision", "comment"])
    for req in requirements(checklist_path):
        cands, exact = candidates(req, rules)
        s.append([req["req_id"], req["provision"], req["applies_to"], req["type"], req["final_requirement"],
                  req["final_on_chain"], "\n".join(rule_text(r) for r in cands) or "(no candidate)",
                  "COVERED (exact limit match)" if exact else "", "", ""])
    for i, w in enumerate([8, 16, 26, 18, 60, 10, 70, 18, 14, 30], 1):
        s.column_dimensions[s.cell(row=1, column=i).column_letter].width = w
    for c in s[1]:
        c.font, c.fill = Font(bold=True, color="FFFFFF"), PatternFill("solid", fgColor="2F5496")
    for row in s.iter_rows(min_row=2):
        for c in row:
            c.alignment = Alignment(wrap_text=True, vertical="top")
    dv = DataValidation(type="list", formula1='"COVERED,PARTIAL,NOT_COVERED"', allow_blank=True)
    s.add_data_validation(dv)
    dv.add(f"I2:I{s.max_row + 50}")
    s.freeze_panes = "A2"
    log = wb.create_sheet("Log")
    log.append(["field", "value"])
    for k in ["expert name", "expertise", "date(s)", "total time (minutes)", "comments"]:
        log.append([k, ""])
    wb.save(out_path)
    print(f"coverage workbook: {s.max_row - 1} requirements -> {out_path}")


def cmd_coverage_score(review_path, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    ws = load_workbook(review_path, data_only=True)["Coverage"]
    data = list(ws.iter_rows(values_only=True))
    hdr = [str(h) for h in data[0]]
    rows = [dict(zip(hdr, ["" if v is None else str(v).strip() for v in r])) for r in data[1:] if r[0]]
    by = {}
    for r in rows:
        d = (r["decision"] or "UNDECIDED").upper()
        for group in ("ALL", f"on_chain={r['on_chain'] or '?'}", f"type={r['type'] or '?'}"):
            by.setdefault(group, Counter())[d] += 1
    lines = []
    for group, c in by.items():
        n = sum(c.values())
        cov = c["COVERED"] + 0.5 * c["PARTIAL"]
        lines.append(f"{group:<32} n={n:<3} covered={c['COVERED']:<3} partial={c['PARTIAL']:<3} "
                     f"not_covered={c['NOT_COVERED']:<3} undecided={c['UNDECIDED']:<3} "
                     f"coverage={100 * c['COVERED'] / n:.1f} % (with partial as 1/2: {100 * cov / n:.1f} %)")
    open(os.path.join(out_dir, "coverage_report.txt"), "w", encoding="utf-8").write("\n".join(lines) + "\n")
    print("\n".join(lines))


if __name__ == "__main__":
    {"metrics": cmd_metrics, "coverage-sheet": cmd_coverage_sheet,
     "coverage-score": cmd_coverage_score}[sys.argv[1]](*sys.argv[2:])


In [ ]:
!python scripts/kg_evaluation.py metrics {G}/gold_test.jsonl {K} baseline={K}/kg_baseline.json learned={K}/kg_learned.json refined={K}/kg_refined.json

### 5.3 Coverage of the regulation's requirements (legal expert, about 30 min)
The coverage workbook lists, for each validated checklist requirement, the candidate rules of the Reg2Chain graph (exact matches are pre-suggested for quality limits). The legal expert copies `step4_kg/coverage/Reg2Chain_kg_coverage.xlsx` to `step4_kg/coverage/Reg2Chain_kg_coverage_reviewed.xlsx` and marks each requirement COVERED / PARTIAL / NOT_COVERED (and her time in *Log*).

In [ ]:
!python scripts/kg_evaluation.py coverage-sheet step1_gold/Reg2Chain_requirements_checklist.xlsx {K}/kg_refined.json {K}/coverage/Reg2Chain_kg_coverage.xlsx

### 5.4 Coverage score
Output: `step4_kg/coverage/coverage_report.txt` (overall, by on-chain class, by requirement type).

In [ ]:
!python scripts/kg_evaluation.py coverage-score {K}/coverage/Reg2Chain_kg_coverage_reviewed.xlsx {K}/coverage

## 6. Regulatory Query Agent
For each of the 8 legal categories of olive oil, the agent queries the Reg2Chain knowledge graph (Cypher on Neo4j, Algorithm 1) and returns a compliance specification (JSON): quality limits, labelling obligations / permissions / prohibitions with their conditions, and the source provision of every rule. Rules stated for a broader concept (e.g. *olive oils*) are inherited through `SUBCLASS_OF`. The same query runs on `kg_refined.json` as a cross-check. Output: `step5_query/`.

In [ ]:
# session setup (needed after a reconnection or a change of runtime)
from google.colab import drive
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Reg2Chain_revision')

In [ ]:
%%writefile scripts/query_agent.py
"""
Section 6 - Regulatory Query Agent.

For each legal category of olive oil, queries the Reg2Chain knowledge graph and produces a
compliance specification (JSON) that is the input of contract generation (section 7):
  quality_limits  : LIMIT rules (parameter, operator, value, unit) with their source provision
  labelling       : obligations / permissions / prohibitions on label items, each with the
                    conditions attached to that label item and its source provision
  conditions      : conditions that apply to the category itself (e.g. classification conditions)
Rules stated for a broader concept (e.g. 'olive oils') are inherited through SUBCLASS_OF
(the top concept 'Product' is excluded); each inherited rule records the concept it comes from.

Two back-ends run the same query:
  neo4j : Cypher on the Neo4j database (credentials NEO4J_URI / NEO4J_USER / NEO4J_PASSWORD)
  json  : the same traversal on kg.json (used when Neo4j is not available, and as a cross-check)

  python query_agent.py kg.json OUT_DIR [--neo4j]
  python query_agent.py evaluate OUT_DIR gold_dev.jsonl gold_test.jsonl
"""
import json
import os
import re
import sys

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from extraction_common import load_jsonl, norm  # noqa: E402

REGULATION = "Commission Delegated Regulation (EU) 2022/2104"
CATEGORIES = ["ExtraVirginOliveOil", "VirginOliveOil", "LampanteOliveOil", "RefinedOliveOil",
              "OliveOilComposedOfRefinedOliveOilsAndVirginOliveOils", "CrudeOlivePomaceOil",
              "RefinedOlivePomaceOil", "OlivePomaceOil"]

# Algorithm 1 (paper): rules of a category, including rules inherited from its super-concepts
CYPHER_RULES = """
MATCH (c:Concept {id: $category})-[:SUBCLASS_OF*0..]->(a:Concept)
WHERE a.id <> 'Product'
MATCH (r:Rule)-[:APPLIES_TO]->(a)
OPTIONAL MATCH (r)-[:CONCERNS]->(o)
OPTIONAL MATCH (r)-[:DERIVED_FROM]->(p:Provision)
RETURN DISTINCT r {.*} AS rule, a.id AS via, o.id AS object_id,
       coalesce(o.label, o.id) AS object_label, o.expression AS object_expression,
       p.source AS source, p.text AS provision
"""
# conditions attached to a label item (CONDITION rules whose subject is that item)
CYPHER_CONDITIONS = """
MATCH (r:Rule {type: 'CONDITION'})-[:APPLIES_TO]->(x {id: $item})
OPTIONAL MATCH (r)-[:CONCERNS]->(o)
OPTIONAL MATCH (r)-[:DERIVED_FROM]->(p:Provision)
RETURN DISTINCT r {.*} AS rule, $item AS via, o.id AS object_id,
       coalesce(o.label, o.id) AS object_label, o.expression AS object_expression,
       p.source AS source, p.text AS provision
"""
CYPHER_LABEL = "MATCH (c {id: $id}) RETURN coalesce(c.label, c.id) AS label"


# ------------------------------------------------------------------ back-ends
class JsonBackend:
    name = "json"

    def __init__(self, graph):
        self.nodes = {n["id"]: n for n in graph["nodes"]}
        self.out = {}
        for e in graph["edges"]:
            self.out.setdefault(e["source"], {}).setdefault(e["type"], []).append(e["target"])
        self.applies = {}
        for e in graph["edges"]:
            if e["type"] == "APPLIES_TO":
                self.applies.setdefault(e["target"], []).append(e["source"])

    def ancestors(self, cid):
        seen, todo = [], [cid]
        while todo:
            x = todo.pop(0)
            if x in seen or x not in self.nodes:
                continue
            seen.append(x)
            todo += self.out.get(x, {}).get("SUBCLASS_OF", [])
        return [x for x in seen if x != "Product"]

    def _row(self, rid, via):
        r = self.nodes[rid]
        o = (self.out.get(rid, {}).get("CONCERNS") or [None])[0]
        p = (self.out.get(rid, {}).get("DERIVED_FROM") or [None])[0]
        on, pn = self.nodes.get(o, {}), self.nodes.get(p, {})
        return {"rule": {k: v for k, v in r.items() if k != "kind"}, "via": via, "object_id": o,
                "object_label": on.get("label", o), "object_expression": on.get("expression"),
                "source": pn.get("source"), "provision": pn.get("text")}

    def rules(self, category):
        rows = []
        for a in self.ancestors(category):
            rows += [self._row(rid, a) for rid in self.applies.get(a, []) if self.nodes[rid]["kind"] == "Rule"]
        return rows

    def conditions(self, item):
        return [self._row(rid, item) for rid in self.applies.get(item, [])
                if self.nodes[rid]["kind"] == "Rule" and self.nodes[rid]["type"] == "CONDITION"]

    def label(self, cid):
        return self.nodes.get(cid, {}).get("label", cid)

    def close(self):
        pass


class Neo4jBackend:
    name = "neo4j"

    def __init__(self):
        from neo4j import GraphDatabase
        uri, pwd = os.environ.get("NEO4J_URI"), os.environ.get("NEO4J_PASSWORD")
        user = os.environ.get("NEO4J_USER") or os.environ.get("NEO4J_USERNAME")
        if not (uri and user and pwd):
            raise RuntimeError("Neo4j credentials not set")
        self.driver = GraphDatabase.driver(uri, auth=(user, pwd))

    def _q(self, query, **kw):
        with self.driver.session() as s:
            return [dict(r) for r in s.run(query, **kw)]

    def rules(self, category):
        return self._q(CYPHER_RULES, category=category)

    def conditions(self, item):
        return self._q(CYPHER_CONDITIONS, item=item)

    def label(self, cid):
        r = self._q(CYPHER_LABEL, id=cid)
        return r[0]["label"] if r else cid

    def close(self):
        self.driver.close()


# ------------------------------------------------------------------ compliance JSON
def split_value(v):
    m = re.match(r"\s*([<>≤≥=]+)\s*([0-9]+(?:[.,][0-9]+)?)\s*(.*)$", v or "")
    return (m.group(1), m.group(2).replace(",", "."), m.group(3).strip()) if m else ("", "", (v or "").strip())


def entry(row):
    r = row["rule"]
    e = {"rule_id": r["id"], "type": r["type"], "source": row["source"] or r.get("source", "")}
    if row["via"]:
        e["applies_via"] = row["via"]
    return e


def specification(backend, category):
    # a rule stated both for the category and for a super-concept is kept once (the most specific link)
    best = {}
    for row in backend.rules(category):
        rid = row["rule"]["id"]
        key = (row["via"] != category, row["via"] or "")
        if rid not in best or key < best[rid][0]:
            best[rid] = (key, row)
    rows = [v[1] for v in best.values()]
    spec = {"category": category, "label": backend.label(category), "regulation": REGULATION,
            "quality_limits": [], "labelling": {"obligations": [], "permissions": [], "prohibitions": []},
            "conditions": []}
    for row in sorted(rows, key=lambda x: x["rule"]["id"]):
        r, e = row["rule"], entry(row)
        if row["via"] == category:
            e.pop("applies_via")
        if r["type"] == "LIMIT":
            op, num, unit = split_value(r.get("value", ""))
            e.update({"parameter": r.get("parameter", ""), "operator": r.get("operator") or op,
                      "value": float(num) if num else None, "value_text": r.get("value", ""),
                      "unit": r.get("unit") or unit})
            spec["quality_limits"].append(e)
        elif r["type"] == "CONDITION":
            e.update({"condition": row["object_expression"] or r.get("condition") or r.get("value", ""),
                      "condition_id": row["object_id"]})
            spec["conditions"].append(e)
        else:
            item = row["object_id"]
            conds = []
            if item:
                for c in sorted(backend.conditions(item), key=lambda x: x["rule"]["id"]):
                    conds.append({"rule_id": c["rule"]["id"], "condition_id": c["object_id"],
                                  "condition": c["object_expression"] or c["rule"].get("condition") or
                                  c["rule"].get("value", ""), "source": c["source"] or c["rule"].get("source", "")})
            e.update({"label_item": row["object_label"] or r.get("value", ""), "label_item_id": item})
            if r.get("value"):
                e["value"] = r["value"]
            if conds:
                e["conditions"] = conds
            key = {"OBLIGATION": "obligations", "PERMISSION": "permissions", "PROHIBITION": "prohibitions"}[r["type"]]
            spec["labelling"][key].append(e)
    spec["traceability"] = {"rules": len(rows), "with_source_provision": sum(1 for x in rows if x["source"])}
    return spec


def run(kg_path, out_dir, *flags):
    os.makedirs(out_dir, exist_ok=True)
    backends = [JsonBackend(json.load(open(kg_path, encoding="utf-8")))]
    if "--neo4j" in flags:
        try:
            backends.insert(0, Neo4jBackend())
        except Exception as ex:  # noqa: BLE001
            print(f"Neo4j not available ({ex}): JSON back-end only")
    results = {b.name: {c: specification(b, c) for c in CATEGORIES} for b in backends}
    main = results[backends[0].name]
    for c, spec in main.items():
        json.dump(spec, open(os.path.join(out_dir, f"{c}.json"), "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    json.dump(list(main.values()), open(os.path.join(out_dir, "compliance_all.json"), "w", encoding="utf-8"),
              ensure_ascii=False, indent=1)
    open(os.path.join(out_dir, "algorithm1_cypher.txt"), "w").write(CYPHER_RULES.strip() + "\n\n" +
                                                                    CYPHER_CONDITIONS.strip() + "\n")
    print(f"Regulatory Query Agent ({backends[0].name}): compliance JSON for {len(main)} categories -> {out_dir}")
    print(f"{'category':55s} {'limits':>6s} {'oblig.':>6s} {'perm.':>6s} {'prohib.':>7s} {'cond.':>5s} {'traceable':>9s}")
    for c, s in main.items():
        lab = s["labelling"]
        print(f"{c:55s} {len(s['quality_limits']):6d} {len(lab['obligations']):6d} {len(lab['permissions']):6d} "
              f"{len(lab['prohibitions']):7d} {len(s['conditions']):5d} "
              f"{s['traceability']['with_source_provision']:>4d}/{s['traceability']['rules']:<4d}")
    if len(backends) == 2:
        same = all(json.dumps(results["neo4j"][c], sort_keys=True) == json.dumps(results["json"][c], sort_keys=True)
                   for c in CATEGORIES)
        diff = [c for c in CATEGORIES if json.dumps(results["neo4j"][c], sort_keys=True) !=
                json.dumps(results["json"][c], sort_keys=True)]
        print("Cypher (Neo4j) and JSON back-ends return identical specifications" if same else
              f"back-ends differ for: {diff}")
    for b in backends:
        b.close()


# ------------------------------------------------------------------ evaluation of the quality limits
def singular(s):
    return " ".join(w[:-1] if w.endswith("s") and len(w) > 3 else w for w in norm(s).split())


def cmd_evaluate(out_dir, *gold_paths):
    """Quality limits returned per category vs the gold LIMIT relations of Annex I (all gold units)."""
    specs = json.load(open(os.path.join(out_dir, "compliance_all.json"), encoding="utf-8"))
    by_label = {singular(s["label"]): s for s in specs}
    gold = [g for p in gold_paths for g in load_jsonl(p)]
    lines, tp_all, n_gold, n_pred = [], 0, 0, 0
    per_cat = {s["category"]: [] for s in specs}
    for g in gold:
        for r in g["relations"]:
            if r["predicate"] == "LIMIT" and singular(r["subject"]) in by_label:
                per_cat[by_label[singular(r["subject"])]["category"]].append(r)
    for s in specs:
        gl = per_cat[s["category"]]
        preds = [x for x in s["quality_limits"] if x.get("value") is not None]
        used, tp = set(), 0
        for r in gl:
            op, num, _ = split_value(r["object"])
            pc = re.sub(r"[^0-9a-zδ]", "", r.get("qualifier", "").lower())
            for i, x in enumerate(preds):
                xc = re.sub(r"[^0-9a-zδ]", "", x["parameter"].lower())
                if i not in used and num and x["operator"] == op and abs(x["value"] - float(num)) < 1e-9 and \
                        pc and xc and (pc in xc or xc in pc):
                    used.add(i)
                    tp += 1
                    break
        tp_all, n_gold, n_pred = tp_all + tp, n_gold + len(gl), n_pred + len(preds)
        lines.append(f"{s['category']:55s} gold={len(gl):2d} returned={len(preds):2d} correct={tp:2d}")
    p, r = tp_all / max(1, n_pred), tp_all / max(1, n_gold)
    lines.append(f"ALL quality limits: gold={n_gold} returned={n_pred} correct={tp_all} precision={p:.3f} "
                 f"recall={r:.3f} F1={2 * p * r / max(1e-9, p + r):.3f}")
    open(os.path.join(out_dir, "query_evaluation.txt"), "w", encoding="utf-8").write("\n".join(lines) + "\n")
    print("\n".join(lines))


if __name__ == "__main__":
    if sys.argv[1] == "evaluate":
        cmd_evaluate(*sys.argv[2:])
    else:
        run(*sys.argv[1:])


In [ ]:
from google.colab import drive, userdata
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Reg2Chain_revision')
!pip install -q neo4j
for k, names in (('NEO4J_URI', ['NEO4J_URI']), ('NEO4J_USER', ['NEO4J_USER', 'NEO4J_USERNAME']),
                 ('NEO4J_PASSWORD', ['NEO4J_PASSWORD'])):
    for name in names:
        try:
            os.environ[k] = userdata.get(name)
            break
        except Exception:
            pass
    else:
        print(f'Secret {k} not set: JSON back-end only')
G = 'step1_gold/gold'; K = 'step4_kg'; Q = 'step5_query'
!python scripts/query_agent.py {K}/kg_refined.json {Q} --neo4j
!python scripts/query_agent.py evaluate {Q} {G}/gold_dev.jsonl {G}/gold_test.jsonl

## 7. Prompt Engineering Agent and contract generation
Four inputs are compared: A, the regulation text only; B, the compliance specification from Section 6 (JSON); C, the specification plus an excerpt of the domain ontology; D, the specification plus the smart-contract ontology (Solidity ontology of Cano-Benito et al., 2021), which describes how the rules can be represented as Solidity constructs. B is the reference: C and D isolate the contribution of each ontology. Every prompt asks for the same interface (`checkLimit`, `checkLabelling`, `isCompliant`) so that all contracts can be tested with the same oracle in Section 8.

Models (open weights, 4-bit, one GPU): Qwen2.5-Coder-7B-Instruct, Qwen2.5-Coder-14B-Instruct and deepseek-coder-6.7b-instruct. Six categories, three runs per setting (temperature 0.2, one seed per run). Every prompt and raw answer is logged in `step6_contracts/logs/generation_log.jsonl`. The script resumes where it stopped, and a generation whose prompt differs from the current one is moved to `logs/stale_generations.jsonl` and redone.

In [ ]:
# session setup (needed after a reconnection or a change of runtime)
from google.colab import drive
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Reg2Chain_revision')

Mounted at /content/drive


In [ ]:
%%writefile scripts/contract_generator.py
"""
Section 7 - Prompt Engineering Agent + smart contract generation (open LLM, Colab T4).

Ablation of the input given to the LLM (what does the knowledge layer contribute?):
  A_text      : the regulation text only (all provisions, with their article numbers)
  B_json      : the compliance specification of the category (Regulatory Query Agent, section 6)
  C_json_onto : the compliance specification + an ontology excerpt (class hierarchy of the category
                and of its label items, properties, condition definitions)
  D_json_solidity : the compliance specification + the smart-contract ontology (Solidity ontology,
                Cano-Benito et al. 2021: classes and properties of the language), used to represent the
                rules as Solidity constructs; B vs D isolates the contribution of this ontology

The same fixed interface is required in every condition, so that all contracts can be tested by
the same oracle in section 8:
  checkLimit(string parameter, int256 value)      value = measured value x 1000 (0.80 -> 800)
  (all public pure; identifiers are ASCII, e.g. 'DeltaK' for ΔK; one '// source:' comment per check)
  checkLabelling(string[] items)                  identifiers of the label items present
  isCompliant(string[] parameters, int256[] values, string[] items)
The identifiers of the parameters and label items are the same list for every category and
condition (the union over all categories): the model must decide from its input which ones apply.

Several runs per (condition, category) with low-temperature sampling and a fixed seed per run;
every prompt and raw answer is logged (step6_contracts/logs/generation_log.jsonl), and the run
resumes where it stopped.

Model comparison: the main model (qwen7b) runs the full ablation; the other models run the
same prompts (--model=deepseek6.7b / --model=qwen14b), by default on the best condition.

  python contract_generator.py QUERY_DIR ONTOLOGY_JSON UNITS_CSV OUT_DIR [--model=qwen7b] [--runs=3]
                               [--conditions=A_text,B_json,C_json_onto,D_json_solidity]
                               [--solidity-ontology=data/solidity_ontology.ttl]
                               [--categories=ExtraVirginOliveOil,...] [--dry-run]
"""
import csv
import json
import os
import re
import sys
import time



def source_of(unit_id):
    """Same provision labels as the Ontology Agent (e.g. 'Article 8(1)', 'Annex I, Table A, row 1')."""
    m = re.match(r"ART(\d+)(?:\.(\d+))?(?:\.([a-h]|lead))?", unit_id)
    if m:
        s = f"Article {m.group(1)}" + (f"({m.group(2)})" if m.group(2) else "")
        return s + (f", point ({m.group(3)})" if m.group(3) and m.group(3) != "lead" else "")
    m = re.match(r"ANX1\.A\.(\d)", unit_id)
    return f"Annex I, Table A, row {m.group(1)}" if m else unit_id

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct"   # main model (full ablation)
# models compared (open weights, ungated, 4-bit on one GPU, T4 or L4); short name = output folder
MODELS = {"qwen7b": "Qwen/Qwen2.5-Coder-7B-Instruct",
          "deepseek6.7b": "deepseek-ai/deepseek-coder-6.7b-instruct",
          "qwen14b": "Qwen/Qwen2.5-Coder-14B-Instruct"}
MAX_NEW_TOKENS = 3000
TEMPERATURE, TOP_P = 0.2, 0.95
SEEDS = [11, 22, 33, 44, 55]
CONDITIONS = ["A_text", "B_json", "C_json_onto", "D_json_solidity"]
# Solidity ontology (Cano-Benito, Cimmino and Garcia-Castro, IEEE Access 2021), CC BY 4.0:
# https://github.com/oeg-upm/Solidity-ontology (ontology.ttl)
SOLIDITY_ONTOLOGY = "data/solidity_ontology.ttl"
# categories with at least one quality limit in the knowledge graph (lampante: classification
# condition only; crude olive-pomace oil: no rule) - see section 6
CATEGORIES = ["ExtraVirginOliveOil", "VirginOliveOil", "RefinedOliveOil",
              "OliveOilComposedOfRefinedOliveOilsAndVirginOliveOils", "RefinedOlivePomaceOil", "OlivePomaceOil"]

SYSTEM = ("You are an expert Solidity developer who writes regulatory-compliance smart contracts. "
          "You implement exactly the rules given in the input, nothing invented, and you answer with one "
          "Solidity code block only.")

INTERFACE = """Write ONE Solidity contract named `{name}Compliance` (pragma solidity ^0.8.20; SPDX MIT; no imports)
that checks the compliance of a batch of **{label}** with Commission Delegated Regulation (EU) 2022/2104.

The contract MUST expose exactly these functions (all `public pure`, so that `isCompliant` can call the other two):
```solidity
// Quality: `value` is the measured value multiplied by 1000 (e.g. 0.80 -> 800, 20.0 -> 20000).
// Returns true if the value satisfies the limit of this category for that parameter,
// and true if this category has no limit for that parameter.
function checkLimit(string memory parameter, int256 value) public pure returns (bool);
// Labelling: `items` are the identifiers of the label items present on the label.
// Returns false if a mandatory item is missing or a prohibited item is present.
function checkLabelling(string[] memory items) public pure returns (bool);
// Both checks together (parameters[i] measured as values[i]).
function isCompliant(string[] memory parameters, int256[] memory values, string[] memory items)
    public pure returns (bool);
```
Use only these identifiers (exact ASCII strings):
- quality parameters: {params}
- label items: {items}

Requirements:
1. Implement every limit, obligation and prohibition that applies to {label} according to the input, and only those.
2. Traceability: write a comment `// source: <provision>` on the line immediately above EACH individual check
   (one comment per limit, per obligation, per prohibition), with the provision the rule comes from, e.g.:
```solidity
        // source: <provision of this rule>
        if (keccak256(bytes(parameter)) == keccak256(bytes("<ParameterId>"))) return value <= <limit x 1000>;
```
3. Permissions and conditions that cannot be checked from these inputs must be documented in comments (with their source), not invented as checks.
4. The file must start with `// SPDX-License-Identifier: MIT` and `pragma solidity ^0.8.20;` and compile with solc 0.8.20 without errors.

INPUT:
{input}
"""


# ------------------------------------------------------------------ inputs of the four settings (A, B, C, D)
def regulation_text(units_csv):
    units = list(csv.DictReader(open(units_csv, encoding="utf-8")))
    return "\n".join(f"[{source_of(u['unit_id'])}] {u['text']}" for u in units)


def ontology_excerpt(onto, spec):
    concepts = {c["id"]: c for c in onto["concepts"]}

    def chain(cid):
        out, seen = [], set()
        while cid and cid not in seen:
            seen.add(cid)
            out.append(cid)
            cid = concepts.get(cid, {}).get("parent")
        return " ⊑ ".join(out)

    items = sorted({e["label_item_id"] for k in spec["labelling"].values() for e in k if e.get("label_item_id")})
    lines = ["Class hierarchy (⊑ = subClassOf):", f"- {chain(spec['category'])}"]
    lines += [f"- {chain(i)}" for i in items if i in concepts]
    lines.append("Object properties (domain -> range):")
    lines += [f"- {p['property']} ({p['relation_type']}): {p['domain']} -> {p['range']}" for p in onto["properties"]]
    used = {c["condition_id"] for k in spec["labelling"].values() for e in k for c in e.get("conditions", [])}
    used |= {c.get("condition_id") for c in spec.get("conditions", [])}
    conds = [c for c in onto.get("conditions", []) if c["condition_id"] in used]
    if conds:
        lines.append("Conditions (hasCondition):")
        lines += [f"- {c['condition_id']}: {c.get('condition_expression', '')}" for c in conds]
    return "\n".join(lines)


def ascii_id(x):
    """Solidity string literals must be ASCII (unless unicode"..."): ΔK -> DeltaK."""
    return (x or "").replace("Δ", "Delta").replace("δ", "delta")


def identifiers(query_dir):
    params, items = set(), set()
    for f in os.listdir(query_dir):
        if f.endswith(".json") and f != "compliance_all.json":
            s = json.load(open(os.path.join(query_dir, f), encoding="utf-8"))
            params |= {ascii_id(x["parameter"]) for x in s["quality_limits"] if x.get("parameter")}
            items |= {e["label_item_id"] for k in s["labelling"].values() for e in k if e.get("label_item_id")}
    return sorted(params), sorted(items)


def solidity_excerpt(ttl_path):
    """Compact text rendering of the Solidity ontology: classes (with superclasses), object properties
    and datatype properties (domain -> range)."""
    from rdflib import OWL, RDF, RDFS, Graph
    g = Graph()
    g.parse(ttl_path, format="turtle")

    def name(x):
        return str(x).rsplit("#", 1)[-1].rsplit("/", 1)[-1]

    lines = ["Classes (Class < superclasses):"]
    for c in sorted(set(g.subjects(RDF.type, OWL.Class)), key=name):
        if "#" not in str(c) and "/" not in str(c):
            continue
        sup = [name(x) for x in g.objects(c, RDFS.subClassOf) if ("#" in str(x) or "/" in str(x))]
        lines.append(f"- {name(c)}" + (f" < {', '.join(sorted(sup))}" if sup else ""))
    for title, kind in (("Object properties (domain -> range):", OWL.ObjectProperty),
                        ("Datatype properties (domain -> range):", OWL.DatatypeProperty)):
        lines.append(title)
        for p in sorted(set(g.subjects(RDF.type, kind)), key=name):
            dom = ",".join(sorted(name(x) for x in g.objects(p, RDFS.domain))) or "-"
            rng = ",".join(sorted(name(x) for x in g.objects(p, RDFS.range))) or "-"
            lines.append(f"- {name(p)}: {dom} -> {rng}")
    return "\n".join(lines)


def build_messages(condition, spec, reg_text, onto, params, items, solidity=None):
    compact = {k: v for k, v in spec.items() if k != "traceability"}
    compact["quality_limits"] = [dict(x, parameter=ascii_id(x.get("parameter", ""))) for x in compact["quality_limits"]]
    if condition == "A_text":
        inp = "Regulation text (provisions with their article numbers):\n" + reg_text
    elif condition == "B_json":
        inp = "Compliance specification (JSON):\n" + json.dumps(compact, ensure_ascii=False, indent=1)
    elif condition == "D_json_solidity":
        inp = ("Compliance specification (JSON):\n" + json.dumps(compact, ensure_ascii=False, indent=1) +
               "\n\nSmart-contract ontology (Solidity ontology): use it to determine how the compliance rules "
               "are represented as Solidity constructs.\n" + solidity)
    else:
        inp = ("Compliance specification (JSON):\n" + json.dumps(compact, ensure_ascii=False, indent=1) +
               "\n\nOntology excerpt:\n" + ontology_excerpt(onto, spec))
    user = INTERFACE.format(name=spec["category"], label=spec["label"], params=", ".join(params),
                            items=", ".join(items), input=inp)
    return [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user}]


def extract_code(raw):
    blocks = re.findall(r"```(?:solidity|sol)?\s*\n(.*?)```", raw, flags=re.S)
    if blocks:
        return max(blocks, key=len).strip() + "\n"
    i = raw.find("// SPDX")
    return (raw[i:] if i >= 0 else raw).strip() + "\n"


# ------------------------------------------------------------------ model
def load_model(model_id):
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, device_map="auto",
        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                                               bnb_4bit_quant_type="nf4"))
    model.eval()
    return tok, model


def generate(tok, model, messages, seed):
    import torch
    torch.manual_seed(seed)
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=True, temperature=TEMPERATURE,
                             top_p=TOP_P, pad_token_id=tok.eos_token_id)
    gen = out[0][inputs["input_ids"].shape[1]:]
    return tok.decode(gen, skip_special_tokens=True), int(inputs["input_ids"].shape[1]), int(gen.shape[0])


# ------------------------------------------------------------------ main
def main(query_dir, onto_path, units_csv, out_dir, *flags):
    opts = dict(f[2:].split("=", 1) for f in flags if f.startswith("--") and "=" in f)
    runs = int(opts.get("runs", 3))
    conditions = opts.get("conditions", ",".join(CONDITIONS)).split(",")
    categories = opts.get("categories", ",".join(CATEGORIES)).split(",")
    dry = "--dry-run" in flags
    short = opts.get("model", "qwen7b")
    model_id = MODELS[short]
    onto = json.load(open(onto_path, encoding="utf-8"))
    reg = regulation_text(units_csv)
    params, items = identifiers(query_dir)
    solidity = solidity_excerpt(opts.get("solidity-ontology", SOLIDITY_ONTOLOGY)) \
        if "D_json_solidity" in conditions else None
    os.makedirs(os.path.join(out_dir, "logs"), exist_ok=True)
    log_path = os.path.join(out_dir, "logs", "generation_log.jsonl")
    done = set()
    if os.path.exists(log_path):
        # a logged generation counts as done only if its prompt is the current one; generations made
        # with an earlier version of the prompt are moved to logs/stale_generations.jsonl and redone
        keep, stale = [], []
        for line in open(log_path, encoding="utf-8"):
            x = json.loads(line)
            spec_path = os.path.join(query_dir, x["category"] + ".json")
            same = x["condition"] != "D_json_solidity" or solidity is not None
            if same and os.path.exists(spec_path):
                cur = build_messages(x["condition"], json.load(open(spec_path, encoding="utf-8")), reg, onto,
                                     params, items, solidity)
                same = cur == x["messages"]
            (keep if same else stale).append(line)
        if stale and not dry:
            with open(os.path.join(out_dir, "logs", "stale_generations.jsonl"), "a", encoding="utf-8") as f:
                f.writelines(stale)
            with open(log_path, "w", encoding="utf-8") as f:
                f.writelines(keep)
            print(f"{len(stale)} generation(s) made with an earlier prompt moved to logs/stale_generations.jsonl")
        elif stale:
            print(f"{len(stale)} generation(s) made with an earlier prompt will be redone")
        for line in keep:
            x = json.loads(line)
            done.add((x["model"], x["condition"], x["category"], x["run"]))
    todo = [(c, k, r) for c in conditions for k in categories for r in range(1, runs + 1)
            if (model_id, c, k, r) not in done]
    print(f"{len(todo)} generations to do for {model_id} ({len(done)} generations already in the log)")
    if dry:
        spec = json.load(open(os.path.join(query_dir, categories[0] + ".json"), encoding="utf-8"))
        for c in conditions:
            m = build_messages(c, spec, reg, onto, params, items, solidity)
            open(os.path.join(out_dir, "logs", f"prompt_example_{c}.txt"), "w", encoding="utf-8").write(
                m[0]["content"] + "\n\n" + m[1]["content"])
            print(f"  {c}: prompt of {len(m[1]['content'])} characters -> logs/prompt_example_{c}.txt")
        return
    if not todo:
        return
    tok, model = load_model(model_id)
    for c, k, r in todo:
        spec = json.load(open(os.path.join(query_dir, k + ".json"), encoding="utf-8"))
        messages = build_messages(c, spec, reg, onto, params, items, solidity)
        t0 = time.time()
        raw, n_in, n_out = generate(tok, model, messages, SEEDS[r - 1])
        code = extract_code(raw)
        path = os.path.join(out_dir, short, c, k, f"run{r}.sol")
        os.makedirs(os.path.dirname(path), exist_ok=True)
        open(path, "w", encoding="utf-8").write(code)
        rec = {"condition": c, "category": k, "run": r, "seed": SEEDS[r - 1], "model": model_id, "model_short": short,
               "temperature": TEMPERATURE, "top_p": TOP_P, "prompt_tokens": n_in, "output_tokens": n_out,
               "truncated": n_out >= MAX_NEW_TOKENS, "seconds": round(time.time() - t0, 1),
               "messages": messages, "raw": raw, "contract": path}
        with open(log_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
        print(f"{short} {c:12s} {k:55s} run{r}: {n_in} prompt tokens, {n_out} output tokens, {rec['seconds']} s"
              + (" (TRUNCATED)" if rec["truncated"] else ""))


if __name__ == "__main__":
    main(*sys.argv[1:])


Overwriting scripts/contract_generator.py


### 7.1 Prompts (dry run)
Writes one example prompt per input to `step6_contracts/logs/`. The Solidity ontology is read from `data/solidity_ontology.ttl` (downloaded from the authors' repository if missing).

In [ ]:
from google.colab import drive
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Reg2Chain_revision')
!pip install -q rdflib
if not os.path.exists('data/solidity_ontology.ttl'):
    !wget -q -O data/solidity_ontology.ttl https://raw.githubusercontent.com/oeg-upm/Solidity-ontology/master/ontology.ttl
Q = 'step5_query'; O = 'step3_ontology'; C = 'step6_contracts'
ARGS = f'{Q} {O}/refined/ontology_refined.json step1_gold/units.csv {C}'
!python scripts/contract_generator.py {ARGS} --dry-run

18 generations to do for Qwen/Qwen2.5-Coder-7B-Instruct (126 generations already in the log)
  A_text: prompt of 18491 characters -> logs/prompt_example_A_text.txt
  B_json: prompt of 10003 characters -> logs/prompt_example_B_json.txt
  C_json_onto: prompt of 12308 characters -> logs/prompt_example_C_json_onto.txt
  D_json_solidity: prompt of 16712 characters -> logs/prompt_example_D_json_solidity.txt


### 7.2 Generation
Qwen2.5-Coder-7B and -14B on the four inputs (72 contracts each), deepseek-coder-6.7b on input C (18 contracts): 162 contracts, about 4 h on an L4. If the session disconnects, run the cell again.

In [ ]:
from google.colab import drive
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Reg2Chain_revision')
!pip install -q transformers accelerate bitsandbytes rdflib
if not os.path.exists('data/solidity_ontology.ttl'):
    !wget -q -O data/solidity_ontology.ttl https://raw.githubusercontent.com/oeg-upm/Solidity-ontology/master/ontology.ttl
Q = 'step5_query'; O = 'step3_ontology'; C = 'step6_contracts'
ARGS = f'{Q} {O}/refined/ontology_refined.json step1_gold/units.csv {C}'
!python scripts/contract_generator.py {ARGS} --runs=3 --model=qwen7b
!python scripts/contract_generator.py {ARGS} --runs=3 --model=deepseek6.7b --conditions=C_json_onto
!python scripts/contract_generator.py {ARGS} --runs=3 --model=qwen14b

18 generations to do for Qwen/Qwen2.5-Coder-7B-Instruct (126 generations already in the log)
config.json: 100% 663/663 [00:00<00:00, 3.39MB/s]
tokenizer_config.json: 100% 7.30k/7.30k [00:00<00:00, 22.3MB/s]
vocab.json: 100% 2.78M/2.78M [00:00<00:00, 70.9MB/s]
merges.txt: 100% 1.67M/1.67M [00:00<00:00, 118MB/s]
tokenizer.json: 100% 7.03M/7.03M [00:00<00:00, 149MB/s]
model.safetensors.index.json: 100% 27.8k/27.8k [00:00<00:00, 68.1MB/s]
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0% 0/4 [00:00<?, ?it/s]
Reconstructing (incomplete total...):   0% 0.00/6.02G [00:00<?, ?B/s]         
Reconstructing (incomplete total...):   0% 0.00/15.2G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   0% 0.00/15.2G [00:00<?, ?B/s]
Reconstructing (incomplete total...):   3% 420M/15.2G [00:01<00:30, 479MB/s,  385MB/s  ]
Reconstructing (incomplete total...):  13% 2.01G/15.2G [00:05<00:48, 273MB/s,  168MB/s  ]
Reconstructing (incomplete total...):  

## 8. Evaluation 3: generated contracts (Table 3)
For each contract: compilation as generated (solc 0.8.20), then after a mechanical normalisation that is logged (N1 SPDX line, N2 pragma, N3 `external` to `public`, N4 compilation with viaIR); functional tests on a local EVM, where the quality limits come from the gold standard (boundary tests) and the labelling rules from the expert-validated specification; traceability of the `// source:` comments; gas; Slither. Inputs and models are compared with paired Wilcoxon tests (B vs A, C vs B, D vs B, and between models). Outputs in `step7_evaluation/`.

In [ ]:
# session setup (needed after a reconnection or a change of runtime)
from google.colab import drive
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Reg2Chain_revision')

Mounted at /content/drive


### 8.1 Automatic evaluation of the 162 contracts

In [ ]:
%%writefile scripts/evaluate_contracts.py
"""
Section 8 - Evaluation of the generated smart contracts (compilation, correctness, traceability, gas).

For every contract of step6_contracts/<model>/<condition>/<category>/run<k>.sol:
  1. raw compilation (solc 0.8.20, solc-js), no change to the code;
  2. documented, mechanical normalisation, then recompilation; every rule applied is logged:
       N1 missing SPDX line added             N2 missing pragma added
       N3 'external' -> 'public' when a function is called internally
       N4 'Stack too deep' -> compile with the IR pipeline (viaIR), code unchanged
     (no change to any rule, value, operator or identifier);
  3. functional tests on a local EVM (eth-tester / py-evm), with an oracle that does NOT come from
     the knowledge graph for the quality limits:
       - quality limits: the GOLD LIMIT relations of Annex I (manual annotation), two boundary tests
         per limit (value at the limit -> compliant, first value beyond -> non-compliant);
       - no invented limit: parameters without a gold limit for the category accept any value;
       - labelling: the expert-validated specification (refined KG, section 6): all mandatory items
         -> compliant, each mandatory item missing -> non-compliant, each prohibited item present ->
         non-compliant, optional (permitted) items present -> still compliant;
       - isCompliant: compliant batch -> true, one violated limit -> false;
  4. traceability: for each gold limit, the '// source:' (or @custom:source) comment closest above
     the line that tests the parameter must cite the right provision (Annex I, Table A, row k);
  5. gas: deployment gas and gas of isCompliant on the compliant batch (eth_estimateGas);
  6. Slither (static analysis) when the 'slither' command is available: findings by impact.

  python evaluate_contracts.py CONTRACTS_DIR QUERY_DIR GOLD_DEV GOLD_TEST OUT_DIR
Outputs: contract_results.csv (one line per contract), table3_contracts.csv/.tex (by model x
condition), normalisation_log.csv, slither_findings.csv.
"""
import csv
import json
import os
import re
import shutil
import statistics
import subprocess
import sys
import tempfile

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from extraction_common import load_jsonl, norm  # noqa: E402

SOLC_VERSION = "0.8.20"
SCALE = 1000
COMPILE_JS = r"""
const solc = require('solc'); const fs = require('fs');
const src = fs.readFileSync(process.argv[2], 'utf8'); const viaIR = process.argv[3] === '1';
const input = {language: 'Solidity', sources: {'c.sol': {content: src}},
  settings: {viaIR: viaIR, optimizer: {enabled: viaIR, runs: 200},
             outputSelection: {'*': {'*': ['abi', 'evm.bytecode.object']}}}};
console.log(JSON.stringify(JSON.parse(solc.compile(JSON.stringify(input)))));
"""
CONTRACT_PATH = re.compile(r"(?P<model>[^/]+)/(?P<condition>[^/]+)/(?P<category>[^/]+)/run(?P<run>\d+)\.sol$")


def ascii_id(x):
    return (x or "").replace("Δ", "Delta").replace("δ", "delta")


def compact(s):
    return re.sub(r"[^0-9a-z]", "", ascii_id(s or "").lower())


# ------------------------------------------------------------------ compilation
class Compiler:
    def __init__(self, workdir):
        self.dir = workdir
        if not os.path.exists(os.path.join(workdir, "node_modules", "solc")):
            subprocess.run(["npm", "install", "--silent", f"solc@{SOLC_VERSION}"], cwd=workdir, check=True,
                           stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        self.js = os.path.join(workdir, "compile.js")
        open(self.js, "w").write(COMPILE_JS)

    def compile(self, src, via_ir=False):
        with tempfile.NamedTemporaryFile("w", suffix=".sol", delete=False, dir=self.dir) as f:
            f.write(src)
        out = json.loads(subprocess.run(["node", self.js, f.name, "1" if via_ir else "0"], cwd=self.dir,
                                        capture_output=True, text=True, check=True).stdout)
        os.unlink(f.name)
        errors = [e for e in out.get("errors", []) if e["severity"] == "error"]
        warnings = [e for e in out.get("errors", []) if e["severity"] == "warning"]
        contracts = out.get("contracts", {}).get("c.sol", {})
        return {"ok": not errors and bool(contracts), "errors": [e["formattedMessage"] for e in errors],
                "warnings": len(warnings), "contracts": contracts}


def main_contract(contracts):
    """The contract exposing checkLimit (the generated code may contain helpers or libraries)."""
    for name, c in contracts.items():
        if any(x.get("name") == "checkLimit" for x in c["abi"]):
            return name, c
    return (next(iter(contracts.items())) if contracts else (None, None))


def normalise(src, compiler):
    """Mechanical fixes only; returns (src, via_ir, result, rules applied)."""
    rules = []
    if "SPDX-License-Identifier" not in src:
        src = "// SPDX-License-Identifier: MIT\n" + src
        rules.append("N1")
    if not re.search(r"pragma\s+solidity", src):
        src = re.sub(r"^(// SPDX[^\n]*\n)", r"\1pragma solidity ^0.8.20;\n", src, count=1)
        rules.append("N2")
    res = compiler.compile(src)
    if not res["ok"] and any("Undeclared identifier" in e or "external" in e for e in res["errors"]):
        src2 = re.sub(r"\)\s*external\b", ") public", src)
        src2 = src2.replace("calldata", "memory")
        res2 = compiler.compile(src2)
        if res2["ok"] or len(res2["errors"]) < len(res["errors"]):
            src, res = src2, res2
            rules.append("N3")
    via_ir = False
    if not res["ok"] and any("Stack too deep" in e for e in res["errors"]):
        res2 = compiler.compile(src, via_ir=True)
        if res2["ok"]:
            res, via_ir = res2, True
            rules.append("N4")
    return src, via_ir, res, rules


# ------------------------------------------------------------------ oracle
def split_value(v):
    m = re.match(r"\s*([<>≤≥=]+)\s*([0-9]+(?:[.,][0-9]+)?)", v or "")
    return (m.group(1), float(m.group(2).replace(",", "."))) if m else (None, None)


def singular(s):
    return " ".join(w[:-1] if w.endswith("s") and len(w) > 3 else w for w in norm(s).split())


def build_oracle(query_dir, gold_paths):
    specs = {s["category"]: s for s in json.load(open(os.path.join(query_dir, "compliance_all.json"),
                                                       encoding="utf-8"))}
    by_label = {singular(s["label"]): c for c, s in specs.items()}
    params = sorted({ascii_id(x["parameter"]) for s in specs.values() for x in s["quality_limits"]})
    oracle = {c: {"limits": [], "obligations": [], "prohibitions": [], "permissions": []} for c in specs}
    for g in [x for p in gold_paths for x in load_jsonl(p)]:
        if not g["unit_id"].startswith("ANX1.A."):
            continue
        row = g["unit_id"].split(".")[-1]
        for r in g["relations"]:
            cat = by_label.get(singular(r["subject"]))
            if r["predicate"] != "LIMIT" or not cat:
                continue
            op, val = split_value(r["object"])
            pid = [p for p in params if compact(p) == compact(r.get("qualifier", "")) or
                   (compact(r.get("qualifier", "")) and compact(r.get("qualifier", "")) in compact(p))]
            if op and pid:
                oracle[cat]["limits"].append({"parameter": pid[0], "op": op, "value": val,
                                              "source": f"Annex I, Table A, row {row}", "gold": r})
    for c, s in specs.items():
        lab = s["labelling"]
        oracle[c]["obligations"] = sorted({e["label_item_id"] for e in lab["obligations"] if e.get("label_item_id")})
        oracle[c]["prohibitions"] = sorted({e["label_item_id"] for e in lab["prohibitions"] if e.get("label_item_id")})
        oracle[c]["permissions"] = sorted({e["label_item_id"] for e in lab["permissions"] if e.get("label_item_id")}
                                          - set(oracle[c]["obligations"]) - set(oracle[c]["prohibitions"]))
    return oracle, params


def boundary_cases(lim):
    v = round(lim["value"] * SCALE)
    return {"≤": [(v, True), (v + 1, False)], "<": [(v - 1, True), (v, False)], "≥": [(v, True), (v - 1, False)],
            ">": [(v + 1, True), (v, False)], "=": [(v, True), (v + 1, False)]}[lim["op"]]


# ------------------------------------------------------------------ EVM tests
class EVM:
    def __init__(self):
        from web3 import EthereumTesterProvider, Web3
        self.w3 = Web3(EthereumTesterProvider())
        self.acct = self.w3.eth.accounts[0]

    def deploy(self, compiled):
        c = self.w3.eth.contract(abi=compiled["abi"], bytecode=compiled["evm"]["bytecode"]["object"])
        tx = c.constructor().transact({"from": self.acct, "gas": 29_000_000})
        rec = self.w3.eth.wait_for_transaction_receipt(tx)
        return self.w3.eth.contract(address=rec.contractAddress, abi=compiled["abi"]), rec.gasUsed


def call(fn):
    try:
        return bool(fn.call())
    except Exception:  # noqa: BLE001 - a revert is a failed test
        return None


def run_tests(contract, oracle, params):
    tests = []
    f = contract.functions
    names = {x["name"] for x in contract.abi if x.get("type") == "function"}
    if not {"checkLimit", "checkLabelling", "isCompliant"} <= names:
        return tests, None, sorted({"checkLimit", "checkLabelling", "isCompliant"} - names)
    for lim in oracle["limits"]:
        for value, expected in boundary_cases(lim):
            tests.append(("limit", f"{lim['parameter']} {lim['op']} {lim['value']} @ {value}", expected,
                          call(f.checkLimit(lim["parameter"], value))))
    limited = {lim["parameter"] for lim in oracle["limits"]}
    for p in params:
        if p not in limited and p != "PackagingCapacity":
            tests.append(("no_invented_limit", f"{p} @ 10^9", True, call(f.checkLimit(p, 10 ** 9))))
    full = list(oracle["obligations"])
    tests.append(("labelling", "all mandatory items", True, call(f.checkLabelling(full))))
    for item in oracle["obligations"]:
        tests.append(("labelling", f"missing {item}", False, call(f.checkLabelling([x for x in full if x != item]))))
    for item in oracle["prohibitions"]:
        tests.append(("labelling", f"prohibited {item}", False, call(f.checkLabelling(full + [item]))))
    if oracle["permissions"]:
        tests.append(("labelling", "optional items added", True, call(f.checkLabelling(full + oracle["permissions"]))))
    ok_p = [lim["parameter"] for lim in oracle["limits"]]
    ok_v = [boundary_cases(lim)[0][0] for lim in oracle["limits"]]
    tests.append(("isCompliant", "compliant batch", True, call(f.isCompliant(ok_p, ok_v, full))))
    if oracle["limits"]:
        bad_v = list(ok_v)
        bad_v[0] = boundary_cases(oracle["limits"][0])[1][0]
        tests.append(("isCompliant", "one limit violated", False, call(f.isCompliant(ok_p, bad_v, full))))
    gas = None
    try:
        gas = f.isCompliant(ok_p, ok_v, full).estimate_gas()
    except Exception:  # noqa: BLE001
        pass
    return tests, gas, []


# ------------------------------------------------------------------ traceability
SOURCE_RE = re.compile(r"(?://|\*|///)\s*(?:@custom:)?source\s*:?\s*(.+)", re.I)


def traceability(src, oracle):
    lines = src.splitlines()
    any_src, right = 0, 0
    for lim in oracle["limits"]:
        idx = [i for i, l in enumerate(lines) if f'"{lim["parameter"]}"' in l]
        cited = None
        for i in idx:
            for j in range(i, max(-1, i - 6), -1):
                m = SOURCE_RE.search(lines[j])
                if m:
                    cited = m.group(1)
                    break
            if cited:
                break
        if cited:
            any_src += 1
            row = lim["source"].rsplit(" ", 1)[-1]
            if re.search(r"Annex\s*I\b", cited) and re.search(rf"row\s*{row}\b", cited):
                right += 1
    n = len(oracle["limits"])
    return (any_src / n if n else None), (right / n if n else None)


# ------------------------------------------------------------------ slither
def slither(src, via_ir):
    if not shutil.which("slither"):
        return None
    d = tempfile.mkdtemp()
    p = os.path.join(d, "c.sol")
    with open(p, "w") as fh:
        fh.write(src)
    args = ["slither", p, "--json", "-"] + (["--solc-args", "--via-ir"] if via_ir else [])
    r = subprocess.run(args, capture_output=True, text=True, env=dict(os.environ, SOLC_VERSION=SOLC_VERSION))
    try:
        dets = json.loads(r.stdout)["results"].get("detectors", [])
    except Exception:  # noqa: BLE001
        return None
    shutil.rmtree(d, ignore_errors=True)
    return [(x["check"], x["impact"], x["confidence"]) for x in dets]


# ------------------------------------------------------------------ main
def paired_tests(rows, out_dir):
    """Wilcoxon signed-rank tests on the share of passed tests, paired by (category, run)."""
    try:
        from scipy.stats import wilcoxon
    except ImportError:
        return
    score = {(r["model"], r["condition"], r["category"], r["run"]):
             (r["tests_passed"] / r["tests_total"] if r["tests_total"] else 0.0) for r in rows}
    pairs = [(("qwen7b", "C_json_onto"), ("qwen7b", "A_text")), (("qwen7b", "B_json"), ("qwen7b", "A_text")),
             (("qwen7b", "C_json_onto"), ("qwen7b", "B_json")),
             (("qwen14b", "C_json_onto"), ("qwen14b", "A_text")), (("qwen14b", "B_json"), ("qwen14b", "A_text")),
             (("qwen14b", "C_json_onto"), ("qwen14b", "B_json")),
             (("qwen7b", "D_json_solidity"), ("qwen7b", "B_json")), (("qwen7b", "D_json_solidity"), ("qwen7b", "A_text")),
             (("qwen14b", "D_json_solidity"), ("qwen14b", "B_json")),
             (("qwen14b", "D_json_solidity"), ("qwen14b", "A_text")),
             (("qwen14b", "C_json_onto"), ("qwen7b", "C_json_onto")),
             (("qwen14b", "C_json_onto"), ("deepseek6.7b", "C_json_onto")),
             (("deepseek6.7b", "C_json_onto"), ("qwen7b", "C_json_onto"))]
    lines = []
    for a, b in pairs:
        keys = sorted({(k[2], k[3]) for k in score if k[:2] == a} & {(k[2], k[3]) for k in score if k[:2] == b})
        if not keys:
            continue
        xa = [score[a + k] for k in keys]
        xb = [score[b + k] for k in keys]
        diff = statistics.mean(xa) - statistics.mean(xb)
        try:
            p = wilcoxon(xa, xb).pvalue if any(x != y for x, y in zip(xa, xb)) else 1.0
        except ValueError:
            p = 1.0
        lines.append(f"{a[0]}/{a[1]} vs {b[0]}/{b[1]}: n={len(keys)} mean test pass {statistics.mean(xa):.3f} vs "
                     f"{statistics.mean(xb):.3f} (diff {diff:+.3f}), Wilcoxon p={p:.4f}")
    with open(os.path.join(out_dir, "contract_significance.txt"), "w") as fh:
        fh.write("\n".join(lines) + "\n")
    print("\n".join(lines))


def mean_sd(xs):
    xs = [x for x in xs if x is not None]
    if not xs:
        return "–"
    return f"{statistics.mean(xs):.3f}" + (f" ± {statistics.stdev(xs):.3f}" if len(xs) > 1 else "")


def main(contracts_dir, query_dir, gold_dev, gold_test, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    work = os.path.join(tempfile.gettempdir(), "reg2chain_solc")   # node_modules outside Drive (faster)
    os.makedirs(work, exist_ok=True)
    compiler = Compiler(work)
    oracle, params = build_oracle(query_dir, [gold_dev, gold_test])
    print("Oracle: " + ", ".join(f"{c}: {len(o['limits'])} gold limits / {len(o['obligations'])} mandatory / "
                                 f"{len(o['prohibitions'])} prohibited" for c, o in oracle.items() if o["limits"]))
    evm = EVM()
    rows, norm_log, sl_rows, failures = [], [], [], []
    files = sorted(os.path.join(dp, f) for dp, _, fs in os.walk(contracts_dir) for f in fs if f.endswith(".sol"))
    for path in files:
        m = CONTRACT_PATH.search(path.replace(os.sep, "/"))
        if not m:
            continue
        meta = m.groupdict()
        with open(path, encoding="utf-8") as fh:
            src = fh.read()
        raw = compiler.compile(src)
        nsrc, via_ir, res, rules = (src, False, raw, []) if raw["ok"] else normalise(src, compiler)
        if not raw["ok"]:
            norm_log.append({**meta, "rules": "+".join(rules) or "none", "compiles_after": res["ok"],
                             "first_error": (raw["errors"] or [""])[0].splitlines()[0][:200]})
        row = {**meta, "compile_raw": raw["ok"], "compile_normalised": res["ok"], "normalisation": "+".join(rules),
               "warnings": res["warnings"], "lines": len(src.splitlines())}
        tests, gas_call, missing = [], None, []
        if res["ok"]:
            name, comp = main_contract(res["contracts"])
            try:
                contract, gas_deploy = evm.deploy(comp)
                row["deploy"] = True
                row["gas_deploy"] = gas_deploy
                tests, gas_call, missing = run_tests(contract, oracle[meta["category"]], params)
            except Exception as ex:  # noqa: BLE001
                row["deploy"] = False
                row["deploy_error"] = str(ex)[:200]
        row["missing_functions"] = ",".join(missing)
        row["gas_isCompliant"] = gas_call
        for kind in ("limit", "no_invented_limit", "labelling", "isCompliant"):
            t = [x for x in tests if x[0] == kind]
            row[f"{kind}_pass"] = (sum(1 for x in t if x[3] == x[2]) / len(t)) if t else None
        row["tests_total"] = len(tests)
        row["tests_passed"] = sum(1 for x in tests if x[3] == x[2])
        row["all_tests_pass"] = bool(tests) and row["tests_passed"] == len(tests)
        row["trace_any"], row["trace_correct"] = traceability(src, oracle[meta["category"]])
        failures += [{**meta, "kind": k, "test": d, "expected": e, "got": g} for k, d, e, g in tests if g != e]
        sl = slither(nsrc, via_ir) if res["ok"] else None
        if sl is not None:
            row["slither_high_medium"] = sum(1 for _, imp, _ in sl if imp in ("High", "Medium"))
            row["slither_low_info"] = sum(1 for _, imp, _ in sl if imp not in ("High", "Medium"))
            sl_rows += [{**meta, "check": c, "impact": i, "confidence": cf} for c, i, cf in sl]
        rows.append(row)
        print(f"{meta['model']:13s} {meta['condition']:12s} {meta['category'][:30]:30s} run{meta['run']}: "
              f"raw={'OK' if raw['ok'] else 'ERR'} norm={'OK' if res['ok'] else 'ERR'} "
              f"tests={row['tests_passed']}/{row['tests_total']} trace={row['trace_correct']}")

    def dump(name, data):
        if not data:
            return
        keys = list(dict.fromkeys(k for d in data for k in d))
        with open(os.path.join(out_dir, name), "w", newline="", encoding="utf-8") as f:
            w = csv.DictWriter(f, fieldnames=keys)
            w.writeheader()
            w.writerows(data)

    dump("contract_results.csv", rows)
    dump("normalisation_log.csv", norm_log)
    dump("test_failures.csv", failures)
    dump("slither_findings.csv", sl_rows)

    # Table 3: model x condition
    groups = list(dict.fromkeys((r["model"], r["condition"]) for r in rows))
    table = []
    for g in groups:
        rs = [r for r in rows if (r["model"], r["condition"]) == g]
        n = len(rs)
        table.append({
            "model": g[0], "condition": g[1], "contracts": n,
            "compile_raw": f"{sum(r['compile_raw'] for r in rs) / n:.3f}",
            "compile_normalised": f"{sum(r['compile_normalised'] for r in rs) / n:.3f}",
            "limit_tests": mean_sd([r["limit_pass"] for r in rs]),
            "no_invented_limit": mean_sd([r["no_invented_limit_pass"] for r in rs]),
            "labelling_tests": mean_sd([r["labelling_pass"] for r in rs]),
            "isCompliant_tests": mean_sd([r["isCompliant_pass"] for r in rs]),
            "all_tests_pass": f"{sum(r['all_tests_pass'] for r in rs) / n:.3f}",
            "traceability_correct": mean_sd([r["trace_correct"] for r in rs]),
            "gas_deploy": mean_sd([r.get("gas_deploy") for r in rs]).split(".")[0],
            "gas_isCompliant": mean_sd([r.get("gas_isCompliant") for r in rs]).split(".")[0],
            "slither_high_medium": mean_sd([r.get("slither_high_medium") for r in rs]),
        })
    dump("table3_contracts.csv", table)
    cols = ["model", "condition", "compile_raw", "compile_normalised", "limit_tests", "labelling_tests",
            "all_tests_pass", "traceability_correct", "gas_isCompliant"]
    tex = ["\\begin{tabular}{llrrrrrrr}", "\\toprule",
           "Model & Input & Comp. raw & Comp. norm. & Limit tests & Label tests & All pass & Trace. & Gas \\\\",
           "\\midrule"]
    tex += [" & ".join(str(t[c]).replace("_", "\\_").replace("±", "$\\pm$") for c in cols) + " \\\\" for t in table]
    tex += ["\\bottomrule", "\\end{tabular}"]
    with open(os.path.join(out_dir, "table3_contracts.tex"), "w") as fh:
        fh.write("\n".join(tex) + "\n")
    paired_tests(rows, out_dir)
    print()
    for t in table:
        print(" | ".join(f"{k}={v}" for k, v in t.items()))


if __name__ == "__main__":
    main(*sys.argv[1:])


Overwriting scripts/evaluate_contracts.py


In [ ]:
from google.colab import drive
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Reg2Chain_revision')
!pip install -q "web3[tester]" slither-analyzer solc-select
!solc-select install 0.8.20 > /dev/null && solc-select use 0.8.20
G = 'step1_gold/gold'
!python -W ignore scripts/evaluate_contracts.py step6_contracts step5_query {G}/gold_dev.jsonl {G}/gold_test.jsonl step7_evaluation

Switched global version to 0.8.20
Oracle: ExtraVirginOliveOil: 8 gold limits / 7 mandatory / 0 prohibited, VirginOliveOil: 7 gold limits / 7 mandatory / 0 prohibited, LampanteOliveOil: 2 gold limits / 0 mandatory / 0 prohibited, RefinedOliveOil: 4 gold limits / 0 mandatory / 0 prohibited, OliveOilComposedOfRefinedOliveOilsAndVirginOliveOils: 4 gold limits / 6 mandatory / 1 prohibited, RefinedOlivePomaceOil: 4 gold limits / 0 mandatory / 0 prohibited, OlivePomaceOil: 5 gold limits / 6 mandatory / 1 prohibited
deepseek6.7b  C_json_onto  ExtraVirginOliveOil            run1: raw=OK norm=OK tests=24/27 trace=1.0
deepseek6.7b  C_json_onto  ExtraVirginOliveOil            run2: raw=OK norm=OK tests=27/27 trace=1.0
deepseek6.7b  C_json_onto  ExtraVirginOliveOil            run3: raw=OK norm=OK tests=27/27 trace=1.0
deepseek6.7b  C_json_onto  OliveOilComposedOfRefinedOlive run1: raw=OK norm=OK tests=20/23 trace=1.0
deepseek6.7b  C_json_onto  OliveOilComposedOfRefinedOlive run2: raw=OK norm=OK tes

### 8.2 Blind expert review of a sample of contracts (blockchain developer)
Batch 1: 14 contracts, i.e. 2 categories (extra virgin olive oil, olive-pomace oil) × run 1 × the 7 (model, input) groups of inputs A–C, shuffled and renamed K01–K14 (seed 2104). Batch 2 (same protocol): the 4 contracts of input D for the two Qwen models, and the Qwen-7B / input B / extra virgin olive oil contract (K04), regenerated with the final prompt, which replaces K04 in the scores. The expert (Mahdi Ghorbel) does not know the model or the input; the keys `review_key*.csv` are not given to him. He saves each workbook with the suffix `_reviewed.xlsx`.

In [ ]:
%%writefile scripts/expert_review.py
"""
Section 8.2 - Blind expert review of a sample of generated contracts (blockchain developer).

  python expert_review.py build CONTRACTS_DIR UNITS_CSV OUT_DIR
      Stratified sample: 2 categories (extra virgin olive oil: richest set of limits and labelling
      obligations; olive-pomace oil: includes a prohibition) x run 1 x every (model, input) group.
      Contracts are shuffled (seed 2104) and renamed K01..Knn: the expert does not know the model
      or the input. The key is written to review_key.csv (not to be given to the expert).
      -> Reg2Chain_contract_review.xlsx (Instructions, Reference, Review, one sheet per contract)

  python expert_review.py build CONTRACTS_DIR UNITS_CSV OUT_DIR --batch=2 --start=15
                           --groups=qwen7b/D_json_solidity,qwen14b/D_json_solidity
                           [--add=qwen7b/B_json/ExtraVirginOliveOil] [--replaces=K04]
      Additional batch (same protocol, new ids). --add puts single contracts in the batch (e.g. a
      contract regenerated after the first review); --replaces marks the first-batch reviews that
      the new ones supersede (they are not scored).

  python expert_review.py score OUT_DIR [contract_results.csv]
      Reads every review_key*.csv and the matching *_reviewed.xlsx in OUT_DIR
      -> expert_review_summary.csv / .txt by (model, input), and the agreement between the expert's
         correctness scores and the automatic tests (Spearman correlation).
"""
import csv
import os
import random
import re
import statistics
import sys

from openpyxl import Workbook, load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation

SEED = 2104
CATEGORIES = ["ExtraVirginOliveOil", "OlivePomaceOil"]
RUN = "1"
HEAD, FILL = Font(bold=True, color="FFFFFF"), PatternFill("solid", fgColor="2F5496")
WRAP = Alignment(wrap_text=True, vertical="top")
MONO = Font(name="Consolas", size=10)

CRITERIA = [
    ("Q1_limits_correct", "1-5", "Quality limits (Annex I, Table A): values, operators and scaling (x1000) are correct for "
                                 "this category. 5 = all correct, 1 = mostly wrong."),
    ("Q2_labelling_correct", "1-5", "Labelling rules (Articles 4-12): mandatory items required, prohibited items rejected, "
                                    "permissions not turned into obligations. 5 = all correct."),
    ("Q3_missing_rules", "text", "Rules of the regulation for this category that are missing (list them, with article)."),
    ("Q4_invented_rules", "text", "Checks that are NOT in the regulation for this category (invented or wrong category)."),
    ("Q5_traceability", "1-5", "The '// source:' comments cite the right provision for each check. 5 = all right."),
    ("Q6_code_quality", "1-5", "Readability, structure, maintainability, gas-conscious code. 5 = production quality."),
    ("Q7_security_issue", "NONE/LOW/MEDIUM/HIGH", "Most severe security or design issue you see."),
    ("Q8_security_comment", "text", "Describe the issue(s), if any."),
    ("Q9_deployable", "YES/MINOR_CHANGES/NO", "Could this contract be deployed as it is for the stated purpose?"),
    ("Q10_minutes", "number", "Time spent on this contract (minutes)."),
]


def style_header(ws, widths):
    for i, w in enumerate(widths, start=1):
        c = ws.cell(row=1, column=i)
        c.font, c.fill, c.alignment = HEAD, FILL, WRAP
        ws.column_dimensions[c.column_letter].width = w
    ws.freeze_panes = "B2"


def build(contracts_dir, units_csv, out_dir, *flags):
    opts = dict(f[2:].split("=", 1) for f in flags if f.startswith("--") and "=" in f)
    batch = opts.get("batch", "1")
    suffix = "" if batch == "1" else f"_batch{batch}"
    os.makedirs(out_dir, exist_ok=True)
    if "groups" in opts:
        groups = [tuple(g.split("/")) for g in opts["groups"].split(",") if g]
    else:
        groups = sorted({(m, c) for m in os.listdir(contracts_dir) if os.path.isdir(os.path.join(contracts_dir, m))
                         and m != "logs" for c in os.listdir(os.path.join(contracts_dir, m))})
    sample = [(m, c, k) for m, c in groups for k in CATEGORIES
              if os.path.exists(os.path.join(contracts_dir, m, c, k, f"run{RUN}.sol"))]
    sample += [tuple(a.split("/")) for a in opts.get("add", "").split(",") if a]
    random.Random(SEED + int(batch) - 1).shuffle(sample)
    start = int(opts.get("start", 1))
    key = [{"review_id": f"K{i:02d}", "model": m, "condition": c, "category": k, "run": RUN,
            "replaces": opts.get("replaces", "") if (m, c, k) in [tuple(a.split("/")) for a in
                                                                   opts.get("add", "").split(",") if a] else ""}
           for i, (m, c, k) in enumerate(sample, start=start)]
    with open(os.path.join(out_dir, f"review_key{suffix}.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(key[0]))
        w.writeheader()
        w.writerows(key)

    label = {"ExtraVirginOliveOil": "extra virgin olive oil", "OlivePomaceOil": "olive-pomace oil"}
    wb = Workbook()
    ws = wb.active
    ws.title = "Instructions"
    ws.column_dimensions["A"].width = 150
    lines = [
        "Reg2Chain - expert review of generated smart contracts (blind)",
        f"You review {len(key)} Solidity contracts (sheets {key[0]['review_id']}...{key[-1]['review_id']}). Each one checks the compliance of a batch "
        "of olive oil of ONE category with Commission Delegated Regulation (EU) 2022/2104.",
        "The contracts were generated automatically by different systems. You are not told which system produced which "
        "contract: please judge each one on its own, against the regulation (sheet 'Reference').",
        "Common interface: checkLimit(parameter, value) with value = measured value x 1000 (0.80 -> 800); "
        "checkLabelling(items) with the identifiers of the label items present; isCompliant(...) = both.",
        "Fill one row per contract in sheet 'Review' (columns Q1 to Q10). Scores 1-5: 1 = very poor, 3 = acceptable, "
        "5 = excellent. Use the drop-down lists where present.",
        "Work without AI tools. Please do not discuss the contracts with the other experts before you finish.",
        "Expected time: about 5 minutes per contract. Record your total time in the 'Log' sheet.",
        "",
        "Criteria:",
    ] + [f"  {n} ({t}): {d}" for n, t, d in CRITERIA]
    for r, line in enumerate(lines, start=1):
        c = ws.cell(row=r, column=1, value=line)
        c.alignment = WRAP
        if r == 1:
            c.font = Font(bold=True, size=14, color="2F5496")

    ref = wb.create_sheet("Reference")
    ref.append(["provision", "text"])
    style_header(ref, [26, 150])
    for u in csv.DictReader(open(units_csv, encoding="utf-8")):
        if re.match(r"ART(4|5|6|7|8|9|10|11|12)\b|ANX1\.A\.(1|8)$", u["unit_id"]):
            ref.append([u["unit_id"], u["text"]])
    for row in ref.iter_rows(min_row=2):
        for c in row:
            c.alignment = WRAP

    rv = wb.create_sheet("Review")
    rv.append(["review_id", "category"] + [n for n, _, _ in CRITERIA])
    style_header(rv, [10, 24] + [16, 16, 40, 40, 14, 14, 18, 40, 18, 10])
    for k in key:
        rv.append([k["review_id"], label[k["category"]]] + [""] * len(CRITERIA))
    n = len(key) + 1
    for col, vals in (("C", "1,2,3,4,5"), ("D", "1,2,3,4,5"), ("G", "1,2,3,4,5"), ("H", "1,2,3,4,5"),
                      ("I", "NONE,LOW,MEDIUM,HIGH"), ("K", "YES,MINOR_CHANGES,NO")):
        dv = DataValidation(type="list", formula1=f'"{vals}"', allow_blank=True)
        rv.add_data_validation(dv)
        dv.add(f"{col}2:{col}{n}")
    for row in rv.iter_rows(min_row=2):
        for c in row:
            c.alignment = WRAP

    for k in key:
        src = open(os.path.join(contracts_dir, k["model"], k["condition"], k["category"], f"run{RUN}.sol"),
                   encoding="utf-8").read()
        s = wb.create_sheet(k["review_id"])
        s.append(["line", f"{k['review_id']} - contract for {label[k['category']]}"])
        style_header(s, [6, 160])
        for i, line in enumerate(src.splitlines(), start=1):
            s.append([i, line])
            c = s.cell(row=i + 1, column=2)
            c.font = MONO
            if line.startswith("="):
                c.data_type = "s"

    log = wb.create_sheet("Log")
    log.append(["field", "value"])
    for f_ in ["expert name", "expertise", "date", "total time (minutes)", "general comments"]:
        log.append([f_, ""])
    style_header(log, [26, 100])
    path = os.path.join(out_dir, f"Reg2Chain_contract_review{suffix}.xlsx")
    wb.save(path)
    print(f"{len(key)} contracts in the blind review workbook -> {path} (key: review_key{suffix}.csv, keep it private)")


def _num(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return None


def spearman(x, y):
    def ranks(v):
        order = sorted(range(len(v)), key=lambda i: v[i])
        r = [0.0] * len(v)
        i = 0
        while i < len(v):
            j = i
            while j + 1 < len(v) and v[order[j + 1]] == v[order[i]]:
                j += 1
            for k in range(i, j + 1):
                r[order[k]] = (i + j) / 2 + 1
            i = j + 1
        return r
    rx, ry = ranks(x), ranks(y)
    mx, my = statistics.mean(rx), statistics.mean(ry)
    num = sum((a - mx) * (b - my) for a, b in zip(rx, ry))
    den = (sum((a - mx) ** 2 for a in rx) * sum((b - my) ** 2 for b in ry)) ** 0.5
    return num / den if den else float("nan")


def score(out_dir, results_csv=None, *rest):
    if out_dir.endswith(".xlsx"):                      # old call: score REVIEWED_XLSX OUT_DIR [results]
        out_dir, results_csv = results_csv, (rest[0] if rest else None)
    key, reviews = {}, []
    for kf in sorted(f for f in os.listdir(out_dir) if f.startswith("review_key") and f.endswith(".csv")):
        suffix = kf[len("review_key"):-len(".csv")]
        bkey = {k["review_id"]: k for k in csv.DictReader(open(os.path.join(out_dir, kf), encoding="utf-8"))}
        xlsx = os.path.join(out_dir, f"Reg2Chain_contract_review{suffix}_reviewed.xlsx")
        if not os.path.exists(xlsx):
            print(f"not reviewed yet: {xlsx}")
            continue
        key.update(bkey)
        rows = list(load_workbook(xlsx, data_only=True)["Review"].iter_rows(values_only=True))
        head = [str(h) for h in rows[0]]
        reviews += [dict(zip(head, r)) for r in rows[1:] if r and r[0] in bkey]
    replaced = {k.get("replaces") for k in key.values() if k.get("replaces")}
    reviews = [r for r in reviews if r["review_id"] not in replaced]
    auto = {}
    if results_csv and os.path.exists(results_csv):
        for r in csv.DictReader(open(results_csv, encoding="utf-8")):
            t = int(r["tests_total"] or 0)
            auto[(r["model"], r["condition"], r["category"], r["run"])] = int(r["tests_passed"] or 0) / t if t else 0.0
    groups = {}
    for r in reviews:
        k = key[r["review_id"]]
        groups.setdefault((k["model"], k["condition"]), []).append((r, k))
    out, lines = [], []
    for g, items in sorted(groups.items()):
        rr = [r for r, _ in items]
        row = {"model": g[0], "condition": g[1], "contracts": len(rr)}
        for n in ("Q1_limits_correct", "Q2_labelling_correct", "Q5_traceability", "Q6_code_quality"):
            v = [_num(r[n]) for r in rr if _num(r[n]) is not None]
            row[n] = f"{statistics.mean(v):.2f}" if v else "–"
        row["security_medium_high"] = sum(1 for r in rr if str(r["Q7_security_issue"]).upper() in ("MEDIUM", "HIGH"))
        row["deployable_yes_or_minor"] = sum(1 for r in rr if str(r["Q9_deployable"]).upper() in ("YES", "MINOR_CHANGES"))
        row["minutes"] = sum(_num(r["Q10_minutes"]) or 0 for r in rr)
        out.append(row)
        lines.append(" | ".join(f"{a}={b}" for a, b in row.items()))
    with open(os.path.join(out_dir, "expert_review_summary.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(out[0]))
        w.writeheader()
        w.writerows(out)
    if auto:
        xs, ys = [], []
        for r in reviews:
            k = key[r["review_id"]]
            a = auto.get((k["model"], k["condition"], k["category"], k["run"]))
            q1, q2 = _num(r["Q1_limits_correct"]), _num(r["Q2_labelling_correct"])
            if a is not None and q1 is not None and q2 is not None:
                xs.append((q1 + q2) / 2)
                ys.append(a)
        if len(xs) > 2:
            lines.append(f"Agreement expert correctness (mean Q1,Q2) vs automatic test pass rate: "
                         f"Spearman rho = {spearman(xs, ys):.3f} (n={len(xs)})")
    lines.append(f"Total expert time: {sum(_num(r['Q10_minutes']) or 0 for r in reviews):.0f} min")
    with open(os.path.join(out_dir, "expert_review_summary.txt"), "w", encoding="utf-8") as f:
        f.write("\n".join(lines) + "\n")
    print("\n".join(lines))


if __name__ == "__main__":
    {"build": build, "score": score}[sys.argv[1]](*sys.argv[2:])


Overwriting scripts/expert_review.py


In [ ]:
from google.colab import drive
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Reg2Chain_revision')
R = 'step7_evaluation/expert_review'
# batch 1 (already built; running it again rewrites the blank workbook only)
# !python scripts/expert_review.py build step6_contracts step1_gold/units.csv {R}
# batch 2
!python scripts/expert_review.py build step6_contracts step1_gold/units.csv {R} --batch=2 --start=15 --groups=qwen7b/D_json_solidity,qwen14b/D_json_solidity --add=qwen7b/B_json/ExtraVirginOliveOil --replaces=K04

5 contracts in the blind review workbook -> step7_evaluation/expert_review/Reg2Chain_contract_review_batch2.xlsx (key: review_key_batch2.csv, keep it private)


After the review (`Reg2Chain_contract_review_reviewed.xlsx` and `Reg2Chain_contract_review_batch2_reviewed.xlsx` in the same folder):

In [4]:
from google.colab import drive
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Reg2Chain_revision')
R = 'step7_evaluation/expert_review'
!python scripts/expert_review.py score {R} step7_evaluation/contract_results.csv

model=deepseek6.7b | condition=C_json_onto | contracts=2 | Q1_limits_correct=4.50 | Q2_labelling_correct=1.00 | Q5_traceability=2.00 | Q6_code_quality=2.00 | security_medium_high=2 | deployable_yes_or_minor=0 | minutes=17.0
model=qwen14b | condition=A_text | contracts=2 | Q1_limits_correct=3.50 | Q2_labelling_correct=1.50 | Q5_traceability=1.50 | Q6_code_quality=2.00 | security_medium_high=2 | deployable_yes_or_minor=0 | minutes=18.0
model=qwen14b | condition=B_json | contracts=2 | Q1_limits_correct=5.00 | Q2_labelling_correct=3.00 | Q5_traceability=5.00 | Q6_code_quality=3.00 | security_medium_high=2 | deployable_yes_or_minor=0 | minutes=17.0
model=qwen14b | condition=C_json_onto | contracts=2 | Q1_limits_correct=5.00 | Q2_labelling_correct=3.00 | Q5_traceability=5.00 | Q6_code_quality=3.00 | security_medium_high=2 | deployable_yes_or_minor=0 | minutes=10.0
model=qwen14b | condition=D_json_solidity | contracts=2 | Q1_limits_correct=5.00 | Q2_labelling_correct=3.00 | Q5_traceability=5.

## 9. Export for the repository
Copies the scripts (taken from the `%%writefile` cells of this notebook), the gold standard, the results and the artefacts into `repository/` and `repository.zip`. Annotator workbooks are not exported.

In [ ]:
# session setup (needed after a reconnection or a change of runtime)
from google.colab import drive
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Reg2Chain_revision')

In [3]:
%%writefile scripts/export_repository.py
"""
Collect the artefacts of the study into one folder (and a zip) for the public GitHub repository.

  python export_repository.py NOTEBOOK.ipynb OUT_DIR

- scripts/      : every %%writefile cell of the notebook, so the code in the repository is
                  exactly the code that was run
- data/         : regulation text, provision units, gold standard (dev/test), annotation guidelines,
                  agreement and calibration reports, requirements checklist, Solidity ontology
- results/      : evaluation outputs (Tables 3 to 6 of the paper), extraction predictions and LLM logs
- artefacts/    : ontologies (OWL + JSON), knowledge graphs, compliance specifications,
                  generated contracts and the generation log (every prompt and raw answer)
- README.md, LICENSE, requirements.txt, SUMMARY.md (main numbers)
Annotator and review workbooks and credentials are not exported.
"""
import glob
import importlib
import json
import os
import re
import shutil
import sys

COPY = {
    "data": ["data/regulation_text.txt", "data/solidity_ontology.ttl",
             "step1_gold/units.csv", "step1_gold/gold/gold_dev.jsonl", "step1_gold/gold/gold_test.jsonl",
             "step1_gold/gold/agreement.txt", "step1_gold/gold/calibration_report.txt",
             "step1_gold/gold/correction_report.txt", "step1_gold/Reg2Chain_requirements_checklist.xlsx"],
    "results/extraction": ["step2_extraction/results/*", "step2_extraction/hybrid_strategy.txt"],
    "results/extraction/predictions": ["step2_extraction/preds/*"],
    "results/extraction/llm_logs": ["step2_extraction/logs/*"],
    "results/ontology": ["step3_ontology/refined/refinement_report.*", "step3_ontology/refined/schema_gaps.csv"],
    "results/kg": ["step4_kg/table2_kg_quality.csv", "step4_kg/coverage/coverage_report.txt"],
    "results/query": ["step5_query/query_evaluation.txt", "step5_query/algorithm1_cypher.txt"],
    "results/contracts": ["step7_evaluation/*.csv", "step7_evaluation/*.tex", "step7_evaluation/*.txt",
                          "step7_evaluation/expert_review/expert_review_summary.*",
                          "step7_evaluation/expert_review/review_key*.csv"],
    "artefacts/ontology": ["step3_ontology/learned/ontology_learned.*", "step3_ontology/refined/ontology_refined.*"],
    "artefacts/kg": ["step4_kg/kg_*.json"],
    "artefacts/compliance_specifications": ["step5_query/*.json"],
    "artefacts/contracts": ["step6_contracts"],
}
EXCLUDE = {"stale_generations.jsonl"}  # contract generated with an earlier prompt, replaced before evaluation

README = """# Reg2Chain: Bridging Regulations to Blockchain through Automated Smart Contract Generation

Code, data and results of the paper *Reg2Chain: Bridging Regulations to Blockchain through Automated
Smart Contract Generation* (Kohail, De Luzi, Grati, Boukadi and Mecella).

Reg2Chain turns a regulation into a knowledge graph and uses it to guide the generation of smart
contracts by open LLMs. The case study is Commission Delegated Regulation (EU) 2022/2104 on olive
oil marketing standards.

Pipeline: provision units -> entity and relation extraction (rules + Qwen2.5-7B-Instruct) ->
ontology learning (Text2Onto approach) and expert refinement -> knowledge graph (Neo4j) ->
regulatory query agent (Cypher) -> smart contract generation (Qwen2.5-Coder-7B/14B-Instruct,
deepseek-coder-6.7b-instruct) -> evaluation.

## Contents

| Folder | Content |
|---|---|
| `Reg2Chain_pipeline.ipynb` | the notebook that runs the whole pipeline and all evaluations, with its outputs |
| `scripts/` | every script of the notebook (written by its `%%writefile` cells) |
| `data/` | regulation text, the 63 provision units, the gold standard (16 development / 47 test units) and its annotation guidelines, the agreement and calibration reports, the requirements checklist (55 requirements), the Solidity ontology |
| `results/extraction/` | Table 3 (`table1_extraction.*`), relation extraction with gold entities, significance tests, predictions of every system and LLM logs |
| `results/ontology/` | expert refinement of the ontology (decisions, effort, schema gaps) |
| `results/kg/` | Table 4 (`table2_kg_quality.csv`) and coverage of the requirements checklist |
| `results/query/` | evaluation of the Regulatory Query Agent and its Cypher queries |
| `results/contracts/` | Table 5 (`table3_contracts.*`), per-contract results, normalisation log, Slither findings, significance tests, blind expert review (Table 6) and its key |
| `artefacts/` | learned and refined ontologies (OWL), knowledge graphs, compliance specifications, the 162 generated contracts (`contracts/<model>/<input>/<category>/run<k>.sol`) and `contracts/logs/generation_log.jsonl` (every prompt and raw model answer) |

File names follow the numbering of the notebook (Table 1 = extraction, Table 2 = knowledge graph,
Table 3 = contracts); in the paper these are Tables 3, 4 and 5.

## Running the pipeline

The notebook was run on Google Colab with one GPU (NVIDIA T4 or L4). Open `Reg2Chain_pipeline.ipynb`
in Colab, copy the repository to a Google Drive folder named `Reg2Chain_revision` and run the
sections in order; each section starts with a setup cell and can be resumed after a disconnection.
All models are open-weight models downloaded from Hugging Face and run locally in 4-bit.
Section 5 needs a Neo4j instance: its URI, user and password are read from the Colab Secrets
(`NEO4J_URI`, `NEO4J_USER`, `NEO4J_PASSWORD`) and are not part of the repository.

## Licence

Code: MIT licence (see `LICENSE`). Data, annotations and results: CC BY 4.0.
The regulation text is reused under the EUR-Lex reuse policy; the Solidity ontology is from
Cano-Benito et al. (2021), *Toward the ontological modeling of smart contracts: a Solidity use case*.
"""

LICENSE = """MIT License

Copyright (c) 2026 The authors of Reg2Chain

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.
"""

REQUIREMENTS = """spacy==3.8.*
en_core_web_sm==3.8.0
transformers
accelerate
bitsandbytes
torch
rdflib
networkx
neo4j
openpyxl
pymupdf
scipy
web3[tester]
slither-analyzer
solc-select
# solc-js 0.8.20 (npm install solc@0.8.20) for compilation
"""


def scripts_from_notebook(nb_path, out):
    nb = json.load(open(nb_path, encoding="utf-8"))
    os.makedirs(out, exist_ok=True)
    n = 0
    for c in nb["cells"]:
        src = "".join(c["source"]) if isinstance(c["source"], list) else c["source"]
        m = re.match(r"%%writefile\s+(\S+)\n", src)
        if c["cell_type"] == "code" and m:
            with open(os.path.join(out, os.path.basename(m.group(1))), "w", encoding="utf-8") as f:
                f.write(src[m.end():])
            n += 1
    return n


def write_guidelines(path):
    """The annotation guidelines are defined in build_annotation_workbooks.py (sheet 'Guidelines')."""
    sys.path.insert(0, "scripts")
    try:
        rows = importlib.import_module("build_annotation_workbooks").GUIDELINES
    except Exception as e:  # keep the export going, but say so
        print("annotation guidelines not exported:", e)
        return
    lines = []
    for title, text in rows:
        lines.append(f"## {title}" if text is None else f"**{title}.** {text}")
        lines.append("")
    with open(path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines).replace("## Reg2Chain", "# Reg2Chain", 1))


def first_lines(path, n=12):
    if not os.path.exists(path):
        return f"(missing: {path})"
    with open(path, encoding="utf-8") as f:
        return "".join(f.readlines()[:n]).rstrip()


def main(nb_path, out_dir):
    if os.path.exists(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir)
    n = scripts_from_notebook(nb_path, os.path.join(out_dir, "scripts"))
    shutil.copy(nb_path, os.path.join(out_dir, os.path.basename(nb_path)))
    copied, missing = 0, []
    ignore = shutil.ignore_patterns(*EXCLUDE, "__pycache__")
    for dest, patterns in COPY.items():
        os.makedirs(os.path.join(out_dir, dest), exist_ok=True)
        for p in patterns:
            hits = [h for h in glob.glob(p) if os.path.basename(h) not in EXCLUDE]
            if not hits:
                missing.append(p)
            for h in hits:
                if os.path.isdir(h):
                    shutil.copytree(h, os.path.join(out_dir, dest), dirs_exist_ok=True, ignore=ignore)
                else:
                    shutil.copy(h, os.path.join(out_dir, dest))
                copied += 1
    write_guidelines(os.path.join(out_dir, "data", "annotation_guidelines.md"))
    for name, text in (("README.md", README), ("LICENSE", LICENSE), ("requirements.txt", REQUIREMENTS)):
        with open(os.path.join(out_dir, name), "w", encoding="utf-8") as f:
            f.write(text)
    summary = ["# Main results", "",
               "## Inter-annotator agreement", "```", first_lines("step1_gold/gold/agreement.txt"), "```",
               "## Table 3 of the paper - extraction (test units)", "```", first_lines("step2_extraction/results/table1_extraction.csv"), "```",
               "```", first_lines("step2_extraction/results/re_gold_entities.csv", 20), "```",
               "## Table 4 of the paper - knowledge graph", "```", first_lines("step4_kg/table2_kg_quality.csv", 20), "```",
               "## Coverage of the requirements", "```", first_lines("step4_kg/coverage/coverage_report.txt", 3), "```",
               "## Regulatory query agent", "```", first_lines("step5_query/query_evaluation.txt", 10), "```",
               "## Table 5 of the paper - contracts", "```", first_lines("step7_evaluation/table3_contracts.csv", 10), "```",
               "```", first_lines("step7_evaluation/contract_significance.txt", 12), "```",
               "## Table 6 of the paper - blind expert review", "```",
               first_lines("step7_evaluation/expert_review/expert_review_summary.txt", 12), "```"]
    with open(os.path.join(out_dir, "SUMMARY.md"), "w", encoding="utf-8") as f:
        f.write("\n".join(summary) + "\n")
    n_sol = len(glob.glob(os.path.join(out_dir, "artefacts", "contracts", "**", "*.sol"), recursive=True))
    zip_path = shutil.make_archive(out_dir.rstrip("/"), "zip", out_dir)
    print(f"{n} scripts, {copied} files/folders copied, {n_sol} contracts -> {out_dir} ({zip_path})")
    if missing:
        print("not found (skipped):", ", ".join(missing))


if __name__ == "__main__":
    main(*sys.argv[1:3])


Writing scripts/export_repository.py


FileNotFoundError: [Errno 2] No such file or directory: 'scripts/export_repository.py'

In [ ]:
from google.colab import drive
import os
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Reg2Chain_revision')
!python scripts/export_repository.py Reg2Chain_pipeline.ipynb repository

24 scripts, 43 files/folders copied -> repository (/content/drive/MyDrive/Reg2Chain_revision/repository.zip)
not found (skipped): step1_gold/gold_precalibration/agreement.txt, step1_gold/gold_precalibration/correction_report.txt


In [ ]:
# Size of the generation inputs: rules per compliance specification and prompt length per setting
import json, collections, os
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/Reg2Chain_revision')

tok = collections.defaultdict(list)
for line in open('step6_contracts/logs/generation_log.jsonl'):
    r = json.loads(line)
    tok[r['condition']].append(r['prompt_tokens'])
print('Prompt length (tokens) per setting: min - max')
for c, v in sorted(tok.items()):
    print(f'  {c}: {min(v)} - {max(v)}')

print('\nRules per compliance specification')
for f in sorted(os.listdir('step5_query')):
    if f.endswith('.json'):
        s = json.load(open('step5_query/' + f))
        if isinstance(s, dict) and 'quality_limits' in s:
            n = len(s['quality_limits']) + sum(len(x) for x in s['labelling'].values()) + len(s['conditions'])
            print(f'  {f}: {n} rules')

Mounted at /content/drive
Prompt length (tokens) per setting: min - max
  A_text: 4979 - 5009
  B_json: 1222 - 3111
  C_json_onto: 1346 - 4409
  D_json_solidity: 2566 - 4455

Rules per compliance specification
  CrudeOlivePomaceOil.json: 0 rules
  ExtraVirginOliveOil.json: 24 rules
  LampanteOliveOil.json: 1 rules
  OliveOilComposedOfRefinedOliveOilsAndVirginOliveOils.json: 16 rules
  OlivePomaceOil.json: 16 rules
  RefinedOliveOil.json: 4 rules
  RefinedOlivePomaceOil.json: 4 rules
  VirginOliveOil.json: 23 rules


In [1]:
# Inventory of the project folder and check of the notebook outputs (read only)
import os, json, re
from google.colab import drive
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
root = '/content/drive/MyDrive/Reg2Chain_revision'
for d, dirs, files in sorted(os.walk(root)):
    dirs[:] = [x for x in sorted(dirs) if not x.startswith(('.', '_', 'repository'))]
    rel = os.path.relpath(d, root)
    if rel.count(os.sep) > 2:
        continue
    names = [f for f in sorted(files) if not f.startswith('.')]
    print(f'{rel}/  ({len(names)} files)', ', '.join(names[:12]) + (' ...' if len(names) > 12 else ''))

nb = json.load(open(root + '/Reg2Chain_pipeline.ipynb', encoding='utf-8'))
out = json.dumps([c.get('outputs', []) for c in nb['cells']])
print('\nPossible secrets in the notebook outputs:',
      sorted(set(re.findall(r'neo4j\+s://[^\s"\\]+|ghp_[A-Za-z0-9]+|hf_[A-Za-z0-9]{10,}|password[^,"\\]{0,30}', out, re.I))) or 'none')

Mounted at /content/drive
./  (2 files) Reg2Chain_pipeline.ipynb, repository.zip
data/  (3 files) CELEX_32022R2104_EN_TXT.pdf, regulation_text.txt, solidity_ontology.ttl
repository/  (3 files) README.md, SUMMARY.md, requirements.txt
repository/artefacts/  (0 files) 
repository/artefacts/compliance_specifications/  (9 files) CrudeOlivePomaceOil.json, ExtraVirginOliveOil.json, LampanteOliveOil.json, OliveOilComposedOfRefinedOliveOilsAndVirginOliveOils.json, OlivePomaceOil.json, RefinedOliveOil.json, RefinedOlivePomaceOil.json, VirginOliveOil.json, compliance_all.json
repository/artefacts/contracts/  (0 files) 
repository/artefacts/kg/  (3 files) kg_baseline.json, kg_learned.json, kg_refined.json
repository/artefacts/ontology/  (4 files) ontology_learned.json, ontology_learned.ttl, ontology_refined.json, ontology_refined.ttl
repository/data/  (4 files) calibration_report.txt, gold_dev.jsonl, gold_test.jsonl, units.csv
repository/results/  (0 files) 
repository/results/contracts/  (11 file

In [2]:
# Publish the exported repository folder to GitHub (the token is read from the Colab Secrets, never printed)
GITHUB_USER = 'optim762-prog'
REPO = 'Reg2Chain'

import os, shutil, subprocess, base64
from google.colab import drive, userdata
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')
token = userdata.get('GITHUB_TOKEN')

# 1. copy the exported folder to the local disk of Colab
src = '/content/drive/MyDrive/Reg2Chain_revision/repository'
work = '/content/repo'
shutil.rmtree(work, ignore_errors=True)
shutil.copytree(src, work)

def git(*args):
    auth = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    r = subprocess.run(['git', '-c', f'http.extraheader=AUTHORIZATION: basic {auth}', *args],
                       cwd=work, capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(token, '***').replace(auth, '***').strip())
    return r.returncode

# 2. create the first version (commit) with all the files
git('init', '-b', 'main')
git('config', 'user.name', GITHUB_USER)
git('config', 'user.email', f'{GITHUB_USER}@users.noreply.github.com')
git('add', '.')
git('commit', '-q', '-m', 'Reg2Chain: code, data and results of the revised paper')

# 3. send it to GitHub
git('remote', 'add', 'origin', f'https://github.com/{GITHUB_USER}/{REPO}.git')
ok = git('push', '-u', 'origin', 'main') == 0
print('\nPublished:' if ok else '\nPush failed, see the message above.', f'https://github.com/{GITHUB_USER}/{REPO}')

Initialized empty Git repository in /content/repo/.git/





branch 'main' set up to track 'origin/main'.
To https://github.com/optim762-prog/Reg2Chain.git
 * [new branch]      main -> main

Published: https://github.com/optim762-prog/Reg2Chain
